<a class="anchor" id="0"></a> 

# **Trabajo Final de Máster**

***Predicción de la Composición de Carbohidratos ingeridos en un Alimento a partir de la Respuesta Glucémica Postprandial mediante Modelos de Aprendizaje Automático***

**Nombre:** Alejandro Magdiel Muñiz Corona

**Tutor:** Manuel de Luna Amat

Máster en Business Analytics, Inteligencia Artificial y Machine Learning

Universidad Francisco de Vitoria

## English summary

> This notebook is the main artifact of my Master's Thesis (TFM) at Universidad Francisco de Vitoria (2025). The body of the notebook is written in Spanish, matching the thesis. The summary below is for international readers.

**Question.** Can we infer the carbohydrate content of a meal just by looking at the postprandial glucose curve it produces?

**Data.** [CGMacros](https://physionet.org/content/cgmacros/1.0.0/) (PhysioNet, 2024): 45 participants, ~10 days each, with two parallel CGM streams (Abbott Libre + Dexcom G6), photographed and labeled meal events, anthropometrics, clinical labs, gut-microbiome relative abundances and gut-health scores.

**Pipeline.**
1. Clean and harmonize the CGMacros tables; PCA on microbiome abundances.
2. For each meal, extract postprandial-glucose features: incremental AUC, peak, time-to-peak, slope, post-meal variability — on both Libre and Dexcom signals.
3. Merge with meal labels, demographics, clinical labs and microbiome PCs.
4. Train tree-based regressors (`XGBoost`, `RandomForest`) on three feature sets: Libre-only, Dexcom-only, and both sensors combined.
5. KMeans clustering of glucose curves to derive an interpretable response-profile feature; refit with this added signal. SHAP for interpretability.

**Headline result.** Best model: XGBoost with both CGM streams + clinical + microbiome features → **RMSE ≈ 23.2 g carbs, R² ≈ 0.29**, in line with prior literature on this inverse problem. Adding the cluster-derived response-profile feature yielded a small but consistent improvement.

**Limitations.** Small cohort (45 subjects, ~1,700 cleaned meal events), self-reported meal logs, and high inter-subject variability cap the achievable R². Per-subject or sequence models (LSTM/Transformer on raw CGM traces) are natural next steps.

_See `README.md` for repository structure and `docs/TFM_thesis.pdf` for the full thesis._



> **Note on outputs.** Embedded plot images were stripped from this notebook to keep the file under GitHub's inline-render size limit. Code, markdown, dataframes and statistical outputs are preserved. All polished figures are available in the thesis PDF at `docs/TFM_thesis.pdf`. To regenerate the plots locally, re-run the notebook after downloading the dataset (see `data/README.md`).

# **Tabla de Contenidos**


1. [Objetivo e Hipótesis](#1-objetivo-e-hipótesis)
2.	[Introducción al Conjunto de Datos](#2-introducción-al-conjunto-de-datos)
3.	[Importación de Librerías](#3-importación-de-librerías)
4.	[Carga y Entendimiento del Conjunto de Datos](#4-carga-y-entendimiento-del-conjunto-de-datos)
5. [Tratamiento de los conjuntos de datos](#5-tratamiento-de-los-conjuntos-de-datos)
6. [Exploratory Data Analysis (EDA)](#6-exploratory-data-analysis-eda)
7. [Reemplazo de los datos en el conjunto de datos original](#7-reemplazo-de-los-datos-en-el-conjunto-de-datos-original)
8. [Respuesta Glucémica Postprandial](#8-respuesta-glucémica-postprandial)
9. [Filtrar los eventos de comida](#9-filtrar-los-eventos-de-comida)
10. [Visualización de curvas válidas](#10-visualización-de-curvas-válidas)
11. [Área incremental bajo la curva (iAUC)](#11-área-incremental-bajo-la-curva-iauc)
12. [Merge de los conjuntos de datos para modelado](#12-merge-de-los-conjuntos-de-datos-para-modelos)
13. [Feature Engineering](#13-feature-engineering)
14. [EDA de los Conjuntos de Datos Final](#14-eda-de-los-conjunto-de-datos-final)
15. [Selección de variables](#15-selección-de-variables)
16. [Preprocesamiento de variables](#16-preprocesamiento-de-variables)
17. [División del conjunto de datos en train y test](#17-división-del-conjunto-de-datos-en-train-y-test)
18. [Entrenamiento de modelos de predicción](#18-entrenamiento-de-modelos-de-predicción)
19. [Clustering](#19-clustering)
20. [Conclusión](#20-conclusión)

# **1. Objetivo e Hipótesis** <a class="anchor" id="1"></a>


[Tabla de Contenidos](#0.1)

**Objetivo:**
Inferir (predecir) la composición de macronutrientes (carbohidratos, proteínas, grasas, fibra) de una comida utilizando solo la respuesta glucémica postprandial (curva de glucosa continua) medida por sensores CGM (Continuous Glucose Monitor).

**Hipótesis**: La forma y características de la curva de glucosa postprandial (por ejemplo, su área bajo la curva, pendiente, pico máximo) contienen suficiente información para estimar el contenido de macronutrientes ingeridos en una comida.


# **2. Introducción al Conjunto de Datos** <a class="anchor" id="2"></a>

[Tabla de Contenidos](#0.1)

**Conjunto de datos de CGMacros**:

En este trabajo se utilizará el conjunto de datos CGMacros, una base de datos multimodal púbica que recoge información detallada sobre la respuesta glucémica postprandial de 45 participantes durante un período de 10 días. El dataset incluye lecturas minuto a minuto de dos sensores de glucosa continua (CGM), datos de actividad física registrados con un reloj inteligente, composición nutricional de las comidas (macronutrientes y calorías), fotografías de alimentos, así como variables demográficas, antropométricas, biomarcadores clínicos y perfiles del microbioma intestinal. Esta base de datos, publicada con fines de investigación, permite explorar enfoques innovadores en nutrición personalizada, como la predicción inversa de macronutrientes a partir de patrones de glucosa, que constituye el objetivo principal de este análisis.

# **3. Importación de librerías** <a class="anchor" id="3"></a>

[Tabla de Contenidos](#0.1)

In [340]:
# Manipulación de datos
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform

# Visualización
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

# División del conjunto de datos
from sklearn.model_selection import train_test_split

# Preprocesamiento de datos
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

# Análisis de componentes principales
from sklearn.decomposition import PCA

# Modelos de imputación
from sklearn.impute import KNNImputer

# Modelos de Regresión
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR

#Pipelines
from sklearn.pipeline import make_pipeline

# Fine tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Evaluación de los modelos
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, make_scorer, mean_absolute_percentage_error

# Explicabilidad de los modelos
import shap

# Time series y manejo de fechas
from datetime import datetime, timedelta
from scipy.signal import find_peaks

# Utilidades del sistema y archivos
import os
import glob
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import re

# Configuraciones generales del entorno
sns.set_theme()
pd.set_option('display.max_columns', None)

# **4. Carga y entendimiento del conjunto de datos** <a class="anchor" id="4"></a>

[Tabla de Contenidos](#0.1)

El conjunto de datos CGMacros está organizado en una estructura de carpetas, donde cada participante tiene su propio archivo `.csv` con lecturas minuto a minuto de glucosa, datos del reloj inteligente, e información nutricional de las comidas registradas durante aproximadamente 10 días. Para este análisis, se recorrerán todas las carpetas individuales `(CGMacros-0XX/)`, se leerán los archivos correspondientes `(CGMacros-0XX.csv)`, y se combinarán en un único DataFrame consolidado.

In [341]:
import os
import pandas as pd

def load_data(path):
    """
    Carga los datos de múltiples archivos CSV ubicados en subcarpetas dentro de un directorio dado.
    Estándariza los nombres de columnas y asegura que todos los DataFrames tengan las mismas columnas.
    
    Parámetros:
        path (str): Ruta al directorio raíz que contiene las carpetas de pacientes.
    
    Retorna:
        pd.DataFrame: DataFrame consolidado con los datos de todos los pacientes.
    """
    
    data = []              # Lista para almacenar los DataFrames de cada archivo
    all_columns = []       # Lista para registrar todas las columnas únicas

    # Identificar todas las columnas únicas presentes en los archivos
    for patient in os.listdir(path):
        if patient.startswith('C'):  # Solo considera carpetas de pacientes
            patient_path = os.path.join(path, patient)

            for file in os.listdir(patient_path):
                if file.endswith('.csv'):
                    file_path = os.path.join(patient_path, file)
                    df = pd.read_csv(file_path, nrows=1)  # Leer solo encabezado

                    # Estandarizar nombres de columnas
                    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

                    # Agregar columnas nuevas a la lista total si aún no están
                    for col in df.columns:
                        if col not in all_columns:
                            all_columns.append(col)

    # Cargar los DataFrames completos y alinear las columnas
    for patient in os.listdir(path):
        if patient.startswith('C'):
            patient_code = patient[-2:]  # Obtener código del paciente (últimos 2 caracteres)
            patient_path = os.path.join(path, patient)

            for file in os.listdir(patient_path):
                if file.endswith('.csv'):
                    file_path = os.path.join(patient_path, file)
                    df = pd.read_csv(file_path)

                    # Estandarizar nombres de columnas
                    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

                    # Añadir identificador del paciente
                    df['patient'] = patient_code

                    # Asegurar que todas las columnas estén presentes
                    for col in all_columns:
                        if col not in df.columns:
                            df[col] = None

                    # Agregar el DataFrame a la lista principal
                    data.append(df)

    # Unificar todos los DataFrames y resetear índice
    final_df = pd.concat(data, ignore_index=True)

    return final_df

In [342]:
df = load_data('CGMACROS') # Llamar a la función para leer los datos desde la carpeta con esa ruta

In [343]:
df

,unnamed:_0,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient,steps,recordindex,intensity,sugar
0,0,2020-05-01 10:30:00,84.000000,NaN,56.0,1.0484,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01,None,NaN,NaN,NaN
1,1,2020-05-01 10:31:00,84.133333,NaN,56.0,1.0484,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01,None,NaN,NaN,NaN
2,2,2020-05-01 10:32:00,84.266667,NaN,57.0,1.0484,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01,None,NaN,NaN,NaN
3,3,2020-05-01 10:33:00,84.400000,NaN,54.0,1.0484,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01,None,NaN,NaN,NaN
4,4,2020-05-01 10:34:00,84.533333,NaN,55.0,1.0484,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687575,15310,2025-05-22 00:03:00,262.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49,None,NaN,NaN,NaN
687576,15311,2025-05-22 00:04:00,261.600000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49,None,NaN,NaN,NaN
687577,15312,2025-05-22 00:05:00,261.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49,None,NaN,NaN,NaN
687578,15313,2025-05-22 00:06:00,260.800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49,None,NaN,NaN,NaN


In [344]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687580 entries, 0 to 687579
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   unnamed:_0           137785 non-null  object 
 1   timestamp            687580 non-null  object 
 2   libre_gl             687360 non-null  float64
 3   dexcom_gl            629825 non-null  float64
 4   hr                   610256 non-null  float64
 5   calories_(activity)  652134 non-null  float64
 6   mets                 501078 non-null  float64
 7   meal_type            1706 non-null    object 
 8   calories             1706 non-null    float64
 9   carbs                1706 non-null    float64
 10  protein              1706 non-null    float64
 11  fat                  1706 non-null    float64
 12  fiber                1705 non-null    float64
 13  amount_consumed      1642 non-null    float64
 14  image_path           3197 non-null    object 
 15  patient          

In [345]:
# Borrar la primer columna del DataFrame porque es el índice que viene con el dataset
df.drop(columns = ['unnamed:_0'], inplace = True)

In [346]:
# Observar los nombres de las columnas
df.columns

Index(['timestamp', 'libre_gl', 'dexcom_gl', 'hr', 'calories_(activity)',
       'mets', 'meal_type', 'calories', 'carbs', 'protein', 'fat', 'fiber',
       'amount_consumed', 'image_path', 'patient', 'steps', 'recordindex',
       'intensity', 'sugar'],
      dtype='object')

In [347]:
# Verificar que se tenga información de todos los pacientes
print(len(df['patient'].value_counts()))

45


In [348]:
# Verificar cuánta información se tiene de cada paciente
df['patient'].value_counts()

patient
28    18735
29    17730
22    17625
48    17340
14    17250
39    17205
36    17115
02    17025
15    16875
26    16320
16    16290
20    16260
17    16185
23    16185
10    16155
21    16125
13    15840
33    15840
47    15690
45    15570
30    15465
49    15315
11    14805
08    14760
01    14730
34    14700
12    14655
46    14655
03    14565
27    14535
44    14535
42    14520
43    14520
38    14505
05    14460
06    14460
19    14430
35    14400
09    14370
41    14310
04    14275
18    14085
32    13785
31    13725
07     5655
Name: count, dtype: int64

Con este value_counts() se puede observar que se tiene la información correspondiente a los 45 pacientes del estudio, pero no todos los pacientes cuentan con la misma cantidad de información medida:
- El paciente que más información tiene es el número 28, con 18.735 registros.
- El paciente con menos información es el 07 que tiene 5.655 registros.
- La mayoría de los pacientes tienen entre 17.000 y 13.000 registros.

## Conjuntos de datos suplementarios

Además del archivo principal con lecturas de glucosa y comidas por participante, se utilizan tres archivos suplementarios que contienen variables clínicas y de salud intestinal asociadas a cada sujeto:

`bio.csv`: Incluye datos demográficos (edad, género), antropométricos (peso, altura, IMC) y analíticas de sangre obtenidas el primer día del estudio, tales como niveles de glucosa en ayuno, insulina, HbA1c, colesterol total, HDL, LDL, VLDL y triglicéridos.

`microbes.csv`: Contiene un vector binario para cada participante que indica la presencia (1) o ausencia (0) de 1,979 especies bacterianas identificadas en muestras de microbioma intestinal analizadas mediante un kit Viome.

`gut_health_test.csv`: Proporciona 22 scores ordinales relacionados con el estado funcional del sistema digestivo de cada sujeto (por ejemplo, salud intestinal general, eficiencia digestiva, actividad inflamatoria), codificados como Good (3), Average (2) o Not Optimal (1).

Estas variables suplementarias permiten enriquecer los modelos predictivos con información relevante sobre la salud metabólica y digestiva de cada participante, aportando un contexto personalizado clave para el análisis de respuestas glucémicas postprandiales.



In [349]:
# Se cargan los datasets con datos suplementarios para el análisis
bio = pd.read_csv('CGMacros/bio.csv')
microbes = pd.read_csv('CGMacros/microbes.csv')
gut_scores = pd.read_csv('CGMacros/gut_health_test.csv')

In [350]:
# Función para normalizar los nombres de las columnas
def normalize_columns(df):
    """
    Normaliza los nombres de las columnas de un DataFrame:
    - Elimina espacios y comas innecesarias
    - Convierte todo a minúsculas
    - Sustituye símbolos por guiones bajos
    """
    df.columns = (
        df.columns
        .str.strip()                              # Quitar espacios al inicio y al final
        .str.lower()                              # Convertir a minúsculas
        .str.replace('.', '_')                    # Reemplazar puntos por guiones bajos
        .str.replace('-', '_')                    # Reemplazar guiones por guiones bajos
        .str.replace(',', '')                     # Eliminar comas
        .str.replace(r'\s+', '_', regex=True)     # Reemplazar espacios múltiples por un guion bajo
        .str.replace(r'_+', '_', regex=True)      # Reemplazar guiones bajos múltiples por uno solo
    )
    return df


In [351]:
# Normalizamos las columnas de cada dataset
bio = normalize_columns(bio)
microbes = normalize_columns(microbes)
gut_scores = normalize_columns(gut_scores)

### Bio

In [352]:
bio.head()

,subject,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2
0,1,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67,216,74,142,130,13,2.9,11:06:00 AM,89,9:40,73,12:11,81,13:18
1,2,49,F,30.946742,169.2,62.0,Hispanic/Latino,5.5,93,14.8,61,181,91,90,78,12,2.0,7:38:00 AM,91,7:52,123,9:21,80,10:22
2,3,59,F,26.948690,157.0,64.0,Hispanic/Latino,6.5,118,17.4,154,190,74,116,90,31,2.6,7:25:00 AM,119,7:38,166,9:23,98,10:23
3,4,33,F,42.384279,262.6,66.0,Hispanic/Latino,5.5,105,19.4,300,267,46,221,164,60,5.8,7:20:00 AM,109,7:37,110,9:04,90,10:01
4,5,51,F,30.957534,172.0,62.5,Hispanic/Latino,6.6,144,12.9,392,269,38,231,157,78,7.1,7:45:00 AM,139,8:59,215,10:52,130,11:54


In [353]:
bio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   subject                     45 non-null     int64  
 1   age                         45 non-null     int64  
 2   gender                      45 non-null     object 
 3   bmi                         45 non-null     float64
 4   body_weight                 45 non-null     float64
 5   height                      45 non-null     float64
 6   self_identify               45 non-null     object 
 7   a1c_pdl_(lab)               45 non-null     float64
 8   fasting_glu_pdl_(lab)       45 non-null     int64  
 9   insulin                     45 non-null     float64
 10  triglycerides               45 non-null     int64  
 11  cholesterol                 45 non-null     int64  
 12  hdl                         45 non-null     int64  
 13  non_hdl                     45 non-nu

### Microbes

In [354]:
microbes

subject  abiotrophia_defectiva  abiotrophia_sp_hmsc24b09  \
0         1                    0.0                       0.0   
1         2                    0.0                       0.0   
2         3                    0.0                       0.0   
3         4                    0.0                       1.0   
4         5                    0.0                       0.0   
5         6                    0.0                       0.0   
6         7                    0.0                       0.0   
7         8                    0.0                       0.0   
8         9                    0.0                       0.0   
9        10                    0.0                       0.0   
10       11                    0.0                       0.0   
11       12                    0.0                       0.0   
12       13                    0.0                       0.0   
13       14                    0.0                       0.0   
14       15                    0.0                       0.0   
15       16                    0.0                       0.0   
16       17                    0.0                       0.0   
17       18                    0.0                       0.0   
18       19                    0.0                       0.0   
19       20                    0.0                       0.0   
20       21                    1.0                       0.0   
21       22                    0.0                       0.0   
22       23                    0.0                       0.0   
23       26                    0.0                       0.0   
24       27                    1.0                       0.0   
25       28                    1.0                       0.0   
26       29                    0.0                       0.0   
27       30                    0.0                       0.0   
28       31                    0.0                       0.0   
29       32                    0.0                       0.0   
30       33                    0.0                       0.0   
31       34                    0.0                       0.0   
32       35                    0.0                       0.0   
33       36                    0.0                       0.0   
34       38                    0.0                       0.0   
35       39                    0.0                       0.0   
36       41                    1.0                       0.0   
37       42                    1.0                       0.0   
38       43                    0.0                       0.0   
39       44                    0.0                       0.0   
40       45                    0.0                       0.0   
41       46                    0.0                       0.0   
42       47                    0.0                       0.0   
43       48                    NaN                       NaN   
44       49                    0.0                       0.0   

    acetivibrio_ethanolgignens  acetivibrio_ethanolgignens_strain_acet_33324  \
0                          1.0                                           0.0   
1                          0.0                                           1.0   
2                          0.0                                           1.0   
3                          0.0                                           0.0   
4                          1.0                                           0.0   
5                          0.0                                           0.0   
6                          0.0                                           0.0   
7                          0.0                                           0.0   
8                          0.0                                           0.0   
9                          0.0                                           0.0   
10                         1.0                                           0.0   
11                         0.0                                           0.0   
12                 

In [355]:
microbes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Columns: 1980 entries, subject to bacterium_lf_3
dtypes: float64(1979), int64(1)
memory usage: 696.2 KB


### Gut_scores

In [356]:
gut_scores.head()

,subject,gut_lining_health,lps_biosynthesis_pathways,biofilm_chemotaxis_and_virulence_pathways,tma_production_pathways,ammonia_production_pathways,metabolic_fitness,active_microbial_diversity,butyrate_production_pathways,flagellar_assembly_pathways,putrescine_production_pathways,uric_acid_production_pathways,bile_acid_metabolism_pathways,inflammatory_activity,gut_microbiome_health,digestive_efficiency,protein_fermentation,gas_production,methane_gas_production_pathways,sulfide_gas_production_pathways,oxalate_metabolism_pathways,salt_stress_pathways,microbiome_induced_stress
0,1,2.0,2.0,1.0,2.0,1.0,3.0,2.0,3.0,2.0,2.0,2.0,3.0,2.0,3.0,1.0,1.0,2.0,3.0,1.0,1.0,3.0,2.0
1,2,1.0,1.0,1.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,3.0,2.0,1.0
2,3,2.0,1.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,3.0,2.0,1.0,3.0,2.0
3,4,1.0,1.0,1.0,1.0,2.0,1.0,2.0,2.0,1.0,2.0,3.0,1.0,2.0,1.0,2.0,1.0,3.0,3.0,3.0,1.0,2.0,2.0
4,5,1.0,1.0,2.0,3.0,1.0,2.0,1.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0,2.0,3.0,3.0,2.0,2.0,2.0,2.0


In [357]:
gut_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 23 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   subject                                    47 non-null     int64  
 1   gut_lining_health                          42 non-null     float64
 2   lps_biosynthesis_pathways                  42 non-null     float64
 3   biofilm_chemotaxis_and_virulence_pathways  42 non-null     float64
 4   tma_production_pathways                    42 non-null     float64
 5   ammonia_production_pathways                42 non-null     float64
 6   metabolic_fitness                          42 non-null     float64
 7   active_microbial_diversity                 42 non-null     float64
 8   butyrate_production_pathways               42 non-null     float64
 9   flagellar_assembly_pathways                42 non-null     float64
 10  putrescine_production_pathwa

# **5. Tratamiento de los conjuntos de datos** <a class="anchor" id="5"></a>

[Tabla de Contenidos](#0.1)

Se identifican algunos problemas en los datasets suplementarios:
* El conjunto `microbes` tiene 1980 columnas, y un registro sin datos.
* El conjunto `gut_scores` tiene 5 registros sin datos, pero hay 47 sujetos registrados, lo que quiere decir que hay 2 sujetos que no se tienen en los demás conjuntos de datos, por lo que se eliminarán del estudio.

In [358]:
# Se cambia el tipo de dato de la columna Patient para que coincida con el tipo de dato de los otros datasets en la columna 'Subject'
df['patient'] = df['patient'].str.extract('(\d+)').astype('int64')

In [359]:
# Verificar pacientes en gut_scores con todos los valores nulos (excepto 'subject')
gut_scores_nulos = gut_scores[gut_scores.drop(columns='subject').isna().all(axis=1)]
print("Pacientes en gut_scores con valores vacíos:", gut_scores_nulos['subject'].tolist())

# Verificar pacientes en microbes con todos los valores nulos(excepto 'subject')
microbes_nulos = microbes[microbes.drop(columns='subject').isna().all(axis=1)]
print("Pacientes en microbes con valores vacíos:", microbes_nulos['subject'].tolist())

# Verificar pacientes que están en gut_scores que no están en el dataset principal
pacientes_principales = set(df['patient'].unique())
pacientes_gut_scores = set(gut_scores['subject'].unique())

pacientes_sobrantes = pacientes_gut_scores - pacientes_principales
print("Pacientes extra en gut_scores:", pacientes_sobrantes)


Pacientes en gut_scores con valores vacíos: [24, 25, 26, 28, 48]
Pacientes en microbes con valores vacíos: [48]
Pacientes extra en gut_scores: {24, 25}


Podemos observar que dos de los pacientes que no tienen datos en `gut_scores` de hecho son pacientes que no se han tomado en cuenta en el estudio, por lo que se eliminarán del dataset mencionado.

## Tratamiento del conjunto `microbes`

Se tiene un registro prácticamente solo con valores nulos, y se tienen 1980 columnas, por lo que el tratamiento a realizar es el siguiente:
- **Imputación del registro faltante**: La imputación se realizará con la moda en cada columna, manteniendo el fomrato binario en cada caso, mantiene la distribución original de cada bacteria, además de la rapidez y efectividad práctica.
- **Análisis de componentes principales (PCA)**: Este procedimiento permite agrupar la información de las 1980 columnas en una cantidad de columnas mucho más pequeño, lo que permite trasladar la información al dataset principal sin ocupar demasiada memoria, y sin perder la información.

### Tratamiento de valores nulos

In [360]:
# Verificar cuántas columnas tienen 0 como moda
modas = microbes.iloc[:, 1:].mode().iloc[0]
modas.value_counts()

0
0.0    1924
1.0      55
Name: count, dtype: int64

In [361]:
# Imputar con la moda por columna en microbes
microbes_imputado = microbes.copy()
microbes_imputado.iloc[:, 1:] = microbes.iloc[:, 1:].apply(lambda col: col.fillna(col.mode()[0]), axis=0)

### Análisis de componentes principales (PCA)

In [362]:
# Separar IDs y bacterias
microbes_features = microbes_imputado.drop(columns='subject')

# Reducción de los componentes
pca = PCA(n_components=40)
microbes_pca = pca.fit_transform(microbes_features)

# Obtener la variabilidad explicada por cada componente y el acumulado
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Crear el gráfico
plt.figure(figsize=(8, 5))

# Gráfico de barras para la varianza explicada
plt.bar(range(1, len(explained_variance) + 1), explained_variance * 100, color='lightblue', edgecolor='black', label='Varianza Explicada (%)')

# Gráfico de línea para la varianza acumulada
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance * 100, color='orange', marker='o', linestyle='--', label='Varianza Acumulada (%)')

# Detalles del gráfico
plt.xlabel('Componente Principal')
plt.ylabel('Porcentaje de varianza explicada (%)')
plt.title('Varianza explicada por cada componente principal')
plt.xticks(range(1, len(explained_variance) + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

In [363]:
# Convertir a DataFrame solo con los dos primeros componentes
microbes_pca_df = pd.DataFrame(microbes_pca[:, :3], columns=[f'microbe_PC{i+1}' for i in range(3)])

# Agregar la columna 'subject' al DataFrame
microbes_pca_df['subject'] = microbes_imputado['subject']


In [364]:
microbes_pca_df

,microbe_PC1,microbe_PC2,microbe_PC3,subject
0,12.073691,-3.625122,10.078317,1
1,10.296673,1.411377,0.198310,2
2,12.298997,1.759080,2.869390,3
3,8.923125,2.201819,1.253253,4
4,7.241053,-6.608969,2.575672,5
5,8.087649,-3.566292,-3.655229,6
6,9.947281,-1.438770,-2.454894,7
7,12.215265,2.523248,3.953508,8
8,8.465938,-3.214550,-0.595050,9
9,7.283385,-2.347704,-3.563742,10


## Tratamiento del conjunto de datos `Gut_scores`

Para el tratamiento de los valores nulos presentes en el dataset de `Gut_scores` se realizará el siguiente procedimiento:
* **Eliminar pacientes**: Se eliminan los pacientes extras que no son tomados en cuenta en los demás conjuntos de datos.
* **Merge con el dataset `microbes_imputado`**: Se juntarán los datasets para poder enriquecer la información sobre cada paciente.
* **Imputación por knn**: Se imputará la información de cada paciente de acuerdo con la información de su microbioma y relacionandolo con el vecino más cercano.
* **Separación del dataset `gut_scores`**: Se separa nuevamente para llevar a cabo el merge con el dataset principal.

In [365]:
gut_scores = gut_scores[~gut_scores['subject'].isin([24, 25])]

In [366]:
# Merge de gut_scores con microbes imputado
gut_micro_merge = gut_scores.merge(microbes_imputado, on='subject', how='left')

# Guardamos el orden de columnas original de gut_scores (excepto 'subject')
gut_columns = gut_scores.columns.drop('subject')

### Imputación de los valores nulos

In [367]:
# Aplicar KNNImputer
imputer = KNNImputer(n_neighbors=5)

# Aplicamos imputación solo a las columnas numéricas (exceptuando 'subject')
gut_imputed_values = imputer.fit_transform(gut_micro_merge.drop(columns=['subject']))

In [368]:
# Reconstruimos el DataFrame imputado
gut_scores_imputed = pd.DataFrame(gut_imputed_values[:, :len(gut_columns)], columns=gut_columns)
gut_scores_imputed['subject'] = gut_micro_merge['subject'].values

# Reordenamos columnas para dejar 'subject' al inicio
cols = ['subject'] + gut_columns.tolist()

# Redondear y convertir a entero solo las columnas imputadas
gut_scores_imputed[gut_columns] = gut_scores_imputed[gut_columns].round().astype(int)


In [369]:
gut_scores_imputed

,gut_lining_health,lps_biosynthesis_pathways,biofilm_chemotaxis_and_virulence_pathways,tma_production_pathways,ammonia_production_pathways,metabolic_fitness,active_microbial_diversity,butyrate_production_pathways,flagellar_assembly_pathways,putrescine_production_pathways,uric_acid_production_pathways,bile_acid_metabolism_pathways,inflammatory_activity,gut_microbiome_health,digestive_efficiency,protein_fermentation,gas_production,methane_gas_production_pathways,sulfide_gas_production_pathways,oxalate_metabolism_pathways,salt_stress_pathways,microbiome_induced_stress,subject
0,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,1
1,1,1,1,2,2,2,2,1,2,2,2,1,2,2,1,1,1,1,1,3,2,1,2
2,2,1,1,2,1,2,2,2,2,2,2,2,2,2,2,1,2,3,2,1,3,2,3
3,1,1,1,1,2,1,2,2,1,2,3,1,2,1,2,1,3,3,3,1,2,2,4
4,1,1,2,3,1,2,1,2,2,1,1,2,1,1,2,2,3,3,2,2,2,2,5
5,1,1,1,3,2,2,2,2,2,2,2,1,1,1,2,2,1,1,1,1,2,2,6
6,2,1,2,3,2,2,2,2,2,2,2,1,2,2,2,2,2,1,2,1,3,2,7
7,2,1,1,1,1,2,2,2,2,1,2,2,2,2,1,1,2,3,2,2,2,1,8
8,2,1,1,3,2,2,2,3,1,2,2,2,2,1,2,2,2,2,2,1,3,2,9
9,2,1,1,2,3,3,2,2,1,2,3,1,2,2,3,3,3,3,2,1,3,2,10


## **Conjunto de datos de eventos de comida**

Se crea un conjunto de datos que permite identificar cómo son los eventos de comida que se produjeron en las mediciones del estudio. Esto es posible porque los sujetos registraban el inicio de cada uno de estos eventos de comida con una fotografía y la información de su platillo, por lo que se toma en cuenta que cada registro del dataset `df`  en el que la columna `meal_type` no es un valor nulo, es el inicio de un evento de comida, y además este evento lleva toda la información necesaria sobre el platillo, como son los macronutrientes.

Teniendo este dataset separado del dataset original, podemos verificar que los datos de cada evento de comida estén completos, sean veraces y coherentes.

In [370]:
# Filtramos el DataFrame para solo los puntos donde existe meal_type, que es el inicio de cada comida
meals = df[df['meal_type'].notna()]
meals

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient,steps,recordindex,intensity,sugar
233,2020-05-01 14:23:00,69.800000,109.4,95.0,4.61296,44.0,Lunch,1170.0,85.0,88.0,54.2,12.0,100.0,photos/00000005-PHOTO-2020-5-1-14-23-0.jpg,1,None,NaN,NaN,NaN
618,2020-05-01 20:48:00,84.800000,114.8,81.0,1.36292,13.0,Dinner,80.0,18.0,0.0,0.0,0.0,100.0,photos/00000007-PHOTO-2020-5-1-20-48-0.jpg,1,None,NaN,NaN,NaN
825,2020-05-02 00:15:00,81.000000,97.4,78.0,4.40328,42.0,Snacks,110.0,24.0,0.0,2.0,0.0,100.0,NaN,1,None,NaN,NaN,NaN
1308,2020-05-02 08:18:00,88.400000,101.8,81.0,3.14520,30.0,Breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000010-PHOTO-2020-5-2-8-18-0.jpg,1,None,NaN,NaN,NaN
1530,2020-05-02 12:00:00,82.000000,80.0,84.0,2.93552,28.0,Lunch,840.0,89.0,17.0,42.0,3.0,100.0,photos/00000012-PHOTO-2020-5-2-12-0-0.jpg,1,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684069,2025-05-19 13:37:00,82.933333,94.8,93.0,1.02443,11.0,lunch,445.0,43.0,20.0,20.0,13.0,100.0,photos/00000082-PHOTO-2025-5-19-13-37-0.jpg,49,None,NaN,NaN,NaN
684473,2025-05-19 20:21:00,88.466667,106.0,78.0,0.93130,10.0,dinner,370.0,38.0,26.0,12.0,2.0,200.0,photos/00000084-PHOTO-2025-5-19-20-21-0.jpg,49,None,NaN,NaN,NaN
685118,2025-05-20 07:06:00,138.333333,152.6,81.0,2.23512,24.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000085-PHOTO-2025-5-20-7-6-0.jpg,49,None,NaN,NaN,NaN
685465,2025-05-20 12:53:00,84.000000,100.0,84.0,1.02443,11.0,lunch,725.0,94.0,44.0,20.0,4.0,100.0,photos/00000098-PHOTO-2025-5-20-12-53-0.jpg,49,None,NaN,NaN,NaN


Encontramos que existen 1.706 eventos de comida registrados creados por los 45 pacientes del estudio. Para cada uno de ellos se tiene una medición de los dos sensores de glucosa: `libre_gl` y `dexcom_gl`, el ritmo cardíaco `hr`, las calorías de la actividad en cuestión `calories_(activity)`, la energía (en calorías) que gasta la persona al realizar la actividad física actual `mets`, el tipo de comida que era `meal_type`, las calorias del alimento `calories`,  los macronutrientes `carbs`, `protein`, `fat`, y un micronutriente como la fibra `fiber`. También la cantidad consumida `amount_consumed`, la foto del alimento `image_path` y qué sujeto tomó el alimento `patient`.

También hay dos columnas que marcan el número de pasos dado en ese minuto `steps` y la columna `recordindex` que puede ser un indicador de cada registro en el tiempo. Pero al ser todos los registros de estas columnas, valores nulos, se eliminarán.

In [371]:
meals.drop(columns=['steps', 'recordindex'], inplace=True)

In [372]:
df.drop(columns=['steps', 'recordindex'], inplace=True)

Ahora se verifica que se tenga acceso a las imagenes

In [373]:
# Seleccionar el registro que se quiera visualizar
indice = 20

# Obtener información del registro seleccionado
img_path = meals.iloc[indice]['image_path']
patient = meals.iloc[indice]['patient']

# Construir la ruta completa hacia la imagen
patient_path = f'CGMacros/CGMacros-0{patient:02d}/'
path_final = patient_path + img_path

# Cargar y mostrar la imagen
img = Image.open(path_final)

plt.imshow(img)
plt.axis('off')  # Ocultar ejes
plt.title(f"Imagen del evento {indice} del paciente {int(patient):02d}", fontsize=12, pad=10)
plt.tight_layout()
plt.show()


<Figure size 640x480 with 1 Axes>

Sabemos que el dataset es funcional, pero hace falta hacer algunas exploraciones para verificar que todo esté en ordén

In [374]:
meals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1706 entries, 233 to 686614
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   timestamp            1706 non-null   object 
 1   libre_gl             1705 non-null   float64
 2   dexcom_gl            1684 non-null   float64
 3   hr                   1625 non-null   float64
 4   calories_(activity)  1684 non-null   float64
 5   mets                 1320 non-null   float64
 6   meal_type            1706 non-null   object 
 7   calories             1706 non-null   float64
 8   carbs                1706 non-null   float64
 9   protein              1706 non-null   float64
 10  fat                  1706 non-null   float64
 11  fiber                1705 non-null   float64
 12  amount_consumed      1642 non-null   float64
 13  image_path           1644 non-null   object 
 14  patient              1706 non-null   int64  
 15  intensity            381 non-null    fl

Lo primero que realizaremos es convertir las variables al tipo respectivo que deberían ser, siendo la única que no coincide la columna `timestamp`:

In [375]:
meals['timestamp'] = pd.to_datetime(meals['timestamp'])

In [376]:
meals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1706 entries, 233 to 686614
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp            1706 non-null   datetime64[ns]
 1   libre_gl             1705 non-null   float64       
 2   dexcom_gl            1684 non-null   float64       
 3   hr                   1625 non-null   float64       
 4   calories_(activity)  1684 non-null   float64       
 5   mets                 1320 non-null   float64       
 6   meal_type            1706 non-null   object        
 7   calories             1706 non-null   float64       
 8   carbs                1706 non-null   float64       
 9   protein              1706 non-null   float64       
 10  fat                  1706 non-null   float64       
 11  fiber                1705 non-null   float64       
 12  amount_consumed      1642 non-null   float64       
 13  image_path           1644 non-null

Existen columnas con muy pocos valores no nulos, como son: `intensity` y `sugar`, y su imputación sería muy complicada por tener tan pocos datos. 

Entre otros problemas:
* Los sensores `libre_gl` y `dexcom_gl`, ninguno tiene todos los registros completos, pero el sensor `libre_gl` tiene al menos 1705 de ellos.
* Algunas columnas como `hr`, `calories_(activity)` y `mets`, tienen más de 1300 datos, pero no están completas.
* Eso mismo sucede con otras columnas: `image_path`, `amount_consumed`, `fiber`.

Esto se resolverá de la siguiente manera:
* Los valores de los sensores en este punto no se analizarán porque son un valor en una medición mucho mayor que se realizará posteriormente para el análisis de la curva de respuesta glucémica postpandrial completa. Así que no se tratarán en este momento.
* Todas las demás variables que son mediciones continuas tampoco se tratarán en este mismo contexto.
* Las variables que son más específicas para el evento de comida, como `amount_consumed` y `fiber` serán analizadas para verificar su capacidad para ser imputadas, o en su defecto, el tratamiento adecuado. 
* `image_path` no puede ser imputada, porque no se pueden generar imagenes.
* `intensity` y `sugar` serán eliminadas de los datasets.

In [377]:
meals.drop(columns=['intensity', 'sugar'], inplace=True)
df.drop(columns=['intensity', 'sugar'], inplace=True)

### Imputación de valores nulos

Para la posible imputación de los valores de las columnas `amount_consumed` y `fiber` se exploran cada una de estas para un mejor entendimiento.

### `amount_consumed`

La variable `amount_consumed` es un valor porcentual que representa el porcentaje consumido de la comida, por lo que diversos valores no tienen sentido (cualquier menor a 0 y mayor a 100).
Es una variable importante a rescatar porque representa un factor de ajuste directo sobre los macronutrientes. Si una comida tiene 60g de carbohidratos pero solo se consumió el 50%, entonces el consumo real son 30g. Esto afecta directamente las variables de macronutrientes.

Entonces lo primero es identificar los valores anómalos:

In [378]:
meals['amount_consumed'].value_counts()

amount_consumed
100.00    1120
1.00       207
300.00      42
200.00      39
0.00        39
2.00        35
3.00        30
75.00       29
400.00      21
4.00        15
600.00      13
500.00      11
50.00        8
5.00         8
6.00         5
60.00        4
70.00        2
90.00        2
700.00       2
9.00         2
375.00       1
150.00       1
8.00         1
0.75         1
7.00         1
80.00        1
25.00        1
900.00       1
Name: count, dtype: int64

Se encuentran valores mayores a 100, los cuales no deberían ser posibles, pero la investigación indica que, en la app de registro de los eventos de comida "MyFitnessPal" solo se permite registrar un tipo de alimento con una cantidad de macronutrientes predeterminada, es decir, que si uno de los sujetos del estudio comió dos veces la cantidad predeterminada, puede haber registrado como `amount_consumed` el 200%. 

Para verificar esta información, se exploran los registros:

In [379]:
meals[meals['amount_consumed'] > 100]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
90495,2023-11-04 19:16:00,67.666667,109.0,79.0,NaN,34.0,dinner,442.0,49.0,40.0,4.0,0.0,200.0,photos/00000013-PHOTO-2023-11-4-19-16-0.jpg,7
91457,2023-11-05 19:18:00,50.666667,102.2,71.0,NaN,13.0,dinner,341.0,29.0,29.0,13.0,0.0,200.0,photos/00000019-PHOTO-2023-11-5-19-18-0.jpg,7
92288,2023-11-06 18:54:00,40.000000,96.4,69.0,NaN,32.0,dinner,590.0,46.0,44.0,52.0,7.0,300.0,photos/00000025-PHOTO-2023-11-6-18-54-0.jpg,7
94226,2023-11-09 18:30:00,40.000000,102.6,68.0,NaN,24.0,dinner,659.0,40.0,24.0,25.0,17.0,400.0,photos/00000044-PHOTO-2023-11-9-18-30-0.jpg,7
431255,2022-06-30 19:30:00,73.000000,105.8,94.0,5.36760,NaN,snack,262.0,19.0,14.0,3.0,1.0,300.0,NaN,31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
678702,2025-05-15 20:10:00,115.533333,140.2,79.0,1.02443,11.0,dinner,752.0,63.0,16.0,55.0,20.0,600.0,photos/00000048-PHOTO-2025-5-15-20-10-0.jpg,49
680102,2025-05-16 19:30:00,104.933333,120.6,83.0,0.93130,10.0,dinner,575.0,63.0,19.0,30.0,5.0,400.0,photos/00000054-PHOTO-2025-5-16-19-30-0.jpg,49
681529,2025-05-17 19:17:00,147.800000,181.8,85.0,2.42138,26.0,dinner,412.0,124.0,56.0,36.0,8.0,400.0,photos/00000062-PHOTO-2025-5-17-19-17-0.jpg,49
682994,2025-05-18 19:42:00,180.800000,210.4,73.0,0.93130,10.0,dinner,590.0,69.0,31.0,42.0,9.0,500.0,photos/00000072-PHOTO-2025-5-18-19-42-0.jpg,49


In [380]:
# Filtrar registros con amount_consumed > 100 y ruta de imagen no nula
registros = meals[(meals['amount_consumed'] > 100) & (meals['image_path'].notna())].iloc[:3]

# Verificar si hay registros válidos
if registros.empty:
    print("No se encontraron registros con 'amount_consumed' > 100 y 'image_path' no nulo.")
else:
    # Crear figura con un subplot por imagen
    fig, axs = plt.subplots(1, len(registros), figsize=(5 * len(registros), 5))

    # Asegurar que axs sea iterable
    if len(registros) == 1:
        axs = [axs]

    # Mostrar cada imagen
    for i, (idx, registro) in enumerate(registros.iterrows()):
        img_path = registro['image_path']
        patient = int(registro['patient'])

        patient_path = f'CGMacros/CGMacros-0{patient:02d}/'
        path_final = patient_path + img_path

        img = Image.open(path_final)
        axs[i].imshow(img)
        axs[i].axis('off')
        axs[i].set_title(f'Paciente {patient:02d}\nÍndice {idx}', fontsize=10, pad=6)

    plt.tight_layout()
    plt.show()



<Figure size 1500x500 with 3 Axes>

**Análisis por registro**:
La primer comida que tiene un valor mayor a 100, de acuerdo con la imagen, consta de:
* 2 tortillas, que suele contener de 12-15g de carbohidratos cada una = 24-30g de carbohidratos
* pollo, que suele contener 0g de carbohidratos.
* pico de gallo, que suele contener 1-2g de carbohidratos.
* frijoles, que suelen contener 5-7g por una cucharada.

Lo que quiere decir que este registro puede tener un valor de entre 42 - 47 g de carbohidratos, y su valor actual es 48. Lo que indica que un 200 podría ser un valor equivocado.

El segundo registro, de acuerdo a lo visto en la imagen, parece ser una rebanada de pizza delgada con pollo:
* Una rebanada de pizza delgada = 25-30g de carbohidratos.
Siendo su valor en registro actual de 29g, se considera que un valor de 200 tambiénn está equivocado. 

Finalmente, el tercer registro parece incluir:
* Nopales, 4-6 g de carbohidratos.
* Tortilla, 12-15g de carbohidratos.
* Pollo
Su valor en registro actual es de 46 gramos de carbohidratos, pero su valor cálculado es de 16-21g. Pero no es coherente que su valor de `amount_consumed` sea de 300, en todo caso los valores de macronurientes deberían ser los estimados en este análisis, y el valor de `amount_consumed`, ser el de 300. 

Teniendo esto en cuenta, todos los valores mayores a 100, se tomarán como un 100.



Ahora bien, seguimos lidiando con el problema de valores que están en números decimalos (0-0.99), y de números que están en un rango de 1-10. 
Analicemos estos registros:

In [381]:
meals[(meals['amount_consumed'].between(0,0.9))]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
163221,2023-03-03 21:24:00,116.133333,98.0,75.0,1.35355,11.0,Dinner,0.0,0.0,0.0,0.0,0.0,0.00,photos/00000044-PHOTO-2023-3-3-21-24-0.jpg,12
414874,2021-01-23 17:36:00,118.200000,149.0,87.0,2.45778,27.0,dinner,0.0,0.0,0.0,0.0,0.0,0.00,photos/00000006-PHOTO-2021-1-23-17-36-0.jpg,30
417402,2021-01-25 11:44:00,238.000000,163.0,81.0,1.04027,11.0,lunch,435.0,16.0,66.0,14.0,4.0,0.75,photos/00000012-PHOTO-2021-1-25-11-44-0.jpg,30
424974,2021-01-30 17:56:00,83.066667,112.4,80.0,1.03928,11.0,dinner,0.0,0.0,0.0,0.0,0.0,0.00,photos/00000040-PHOTO-2021-1-30-17-56-0.jpg,30
426478,2021-01-31 19:00:00,247.600000,296.6,NaN,0.94560,10.0,dinner,0.0,0.0,0.0,0.0,0.0,0.00,photos/00000046-PHOTO-2021-1-31-19-0-0.jpg,30
436770,2022-07-04 15:25:00,91.000000,118.8,79.0,1.05435,NaN,snack,2.4,0.0,0.3,0.1,0.0,0.00,photos/00000058-PHOTO-2022-7-4-15-25-0.jpg,31
436829,2022-07-04 16:24:00,93.933333,120.0,74.0,0.95850,NaN,dinner,0.0,0.0,0.0,0.0,0.0,0.00,photos/00000059-PHOTO-2022-7-4-16-24-0.jpg,31
437014,2022-07-04 19:29:00,81.600000,97.8,76.0,0.95850,NaN,dinner,0.0,0.0,0.0,0.0,0.0,0.00,photos/00000060-PHOTO-2022-7-4-19-29-0.jpg,31
437021,2022-07-04 19:36:00,80.600000,97.0,84.0,1.24605,NaN,dinner,0.0,0.0,0.0,0.0,0.0,0.00,photos/00000061-PHOTO-2022-7-4-19-36-0.jpg,31
437031,2022-07-04 19:46:00,86.600000,97.2,80.0,0.95850,NaN,dinner,0.0,0.0,0.0,0.0,0.0,0.00,photos/00000062-PHOTO-2022-7-4-19-46-0.jpg,31


In [382]:
# Filtrar registros con amount_consumed entre 0 y 0.9 y con imagen disponible
registros = meals[(meals['amount_consumed'].between(0, 0.9)) & (meals['image_path'].notna())].iloc[:3]

if registros.empty:
    print("No se encontraron registros con 'amount_consumed' entre 0 y 0.9 y 'image_path' no nulo.")
else:
    # Crear subplots: 1 fila por registro, 2 columnas (antes y después)
    fig, axs = plt.subplots(len(registros), 2, figsize=(10, 5 * len(registros)))

    # Asegurar que axs sea una lista de listas para iterar de forma uniforme
    if len(registros) == 1:
        axs = [axs]

    for i, (idx, registro) in enumerate(registros.iterrows()):
        patient = int(registro['patient'])
        img_path_before = registro['image_path']
        patient_path = f'CGMacros/CGMacros-0{patient:02d}/'

        # Imagen antes de comer
        path_before = patient_path + img_path_before
        img_before = Image.open(path_before)
        axs[i][0].imshow(img_before)
        axs[i][0].axis('off')
        axs[i][0].set_title("Antes de comer", fontsize=10, pad=6)

        # Buscar la imagen posterior del mismo paciente
        df_paciente = df[(df['patient'] == patient) & (df['image_path'].notna())]
        df_paciente_sorted = df_paciente.sort_values(by='image_path')

        posteriores = df_paciente_sorted[df_paciente_sorted['image_path'] > img_path_before]

        if not posteriores.empty:
            siguiente = posteriores.iloc[0]
            path_after = patient_path + siguiente['image_path']
            img_after = Image.open(path_after)
            axs[i][1].imshow(img_after)
            axs[i][1].axis('off')
            axs[i][1].set_title("Después de comer", fontsize=10, pad=6)
        else:
            axs[i][1].axis('off')
            axs[i][1].set_title("Sin imagen posterior", fontsize=10, pad=6)

    plt.tight_layout()
    plt.show()


<Figure size 1000x1500 with 6 Axes>

Los registros que tienen un 0 se tomarán como 100%, es posiblemente un error de registro, podemos observarlo en el primer registro, que terminó con su platillo y simplemente olvidaron registrar la cantidad total consumida y esta fue tomada como 0.
Por su parte, el registro que es un 0.75, es evidentemente un registro que representa un 75%.
La segunda imagen es un error en la que el paciente olvidó registrar la fotografía posterior.

Ahora solo queda el problema de los números que están entre 1-10.

In [383]:
meals[(meals['amount_consumed'].between(1,10))]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
220046,2023-07-14 11:58:00,66.066667,103.0,83.0,1.75357,13.0,snack,280.0,38.0,4.0,12.0,480.0,1.0,photos/00000005-PHOTO-2023-7-14-11-58-0.jpg,16
221265,2023-07-15 08:17:00,89.333333,116.6,96.0,2.02335,15.0,Breakfast,448.0,66.0,22.0,10.5,0.0,1.0,photos/00000009-PHOTO-2023-7-15-8-17-0.jpg,16
221501,2023-07-15 12:13:00,61.933333,93.0,85.0,1.75357,13.0,snack,460.0,45.0,12.0,23.0,230.0,3.0,photos/00000011-PHOTO-2023-7-15-12-13-0.jpg,16
222660,2023-07-16 07:32:00,85.000000,108.8,111.0,7.55384,56.0,Breakfast,608.0,66.0,66.0,10.5,0.0,1.0,photos/00000017-PHOTO-2023-7-16-7-32-0.jpg,16
222934,2023-07-16 12:06:00,72.200000,103.0,95.0,4.58626,34.0,snack,220.0,19.0,7.0,13.0,510.0,1.0,photos/00000019-PHOTO-2023-7-16-12-6-0.jpg,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427352,2021-02-01 09:34:00,123.000000,153.0,113.0,4.34792,47.0,breakfast,268.0,24.0,22.0,10.5,0.0,1.0,photos/00000047-PHOTO-2021-2-1-9-34-0.jpg,30
427540,2021-02-01 12:42:00,242.000000,182.4,76.0,0.94520,10.0,lunch,725.0,94.0,44.0,20.0,4.0,1.0,photos/00000049-PHOTO-2021-2-1-12-42-0.jpg,30
427983,2021-02-01 20:05:00,113.866667,140.6,76.0,1.03972,11.0,dinner,610.0,37.0,30.0,36.0,1.0,5.0,photos/00000051-PHOTO-2021-2-1-20-5-0.jpg,30
428754,2021-02-02 08:56:00,112.066667,NaN,NaN,NaN,NaN,breakfast,902.0,73.0,22.0,42.0,7.0,1.0,photos/00000052-PHOTO-2021-2-2-8-56-0.jpg,30


In [384]:
# Filtrar registros con amount_consumed entre 1 y 10, y con imagen disponible
registros = meals[(meals['amount_consumed'].between(1, 10)) & (meals['image_path'].notna())].iloc[:5]

if registros.empty:
    print("No se encontraron registros con 'amount_consumed' entre 1 y 10 y 'image_path' no nulo.")
else:
    # Crear subplots: 1 fila por registro, 2 columnas (antes y después)
    fig, axs = plt.subplots(len(registros), 2, figsize=(10, 5 * len(registros)))

    # Asegurar que axs sea una lista de listas para iteración uniforme
    if len(registros) == 1:
        axs = [axs]

    for i, (idx, registro) in enumerate(registros.iterrows()):
        patient = int(registro['patient'])
        img_path_before = registro['image_path']
        patient_path = f'CGMacros/CGMacros-0{patient:02d}/'

        # Imagen antes de comer
        path_before = patient_path + img_path_before
        img_before = Image.open(path_before)
        axs[i][0].imshow(img_before)
        axs[i][0].axis('off')
        axs[i][0].set_title("Antes de comer", fontsize=10, pad=6)

        # Buscar imagen posterior del mismo paciente
        df_paciente = df[(df['patient'] == patient) & (df['image_path'].notna())]
        df_paciente_sorted = df_paciente.sort_values(by='image_path')

        posteriores = df_paciente_sorted[df_paciente_sorted['image_path'] > img_path_before]

        if not posteriores.empty:
            siguiente = posteriores.iloc[0]
            path_after = patient_path + siguiente['image_path']
            img_after = Image.open(path_after)
            axs[i][1].imshow(img_after)
            axs[i][1].axis('off')
            axs[i][1].set_title("Después de comer", fontsize=10, pad=6)
        else:
            axs[i][1].axis('off')
            axs[i][1].set_title("Sin imagen posterior", fontsize=10, pad=6)

    plt.tight_layout()
    plt.show()


<Figure size 1000x2500 with 10 Axes>

Por lo que se puede observar en los registros fotográficos, el número que aparece en `amount_consumed` parece referirse a la cantidad sobrante en su platillo, es decir, el porcentaje de comida que no consumieron. 
Entonces la transformación se realizará de esa manera.

Entonces, aplicamos la transformación como se indicó:

In [385]:
# Función para corregir valores en la columna 'amount_consumed'
def corregir_amount_consumed(x):
    if x == 0:
        return 100                      # Se interpreta como consumo total
    elif 0 < x < 1:
        return int(x * 100)             # Ej: 0.75 → 75%
    elif 1 <= x <= 10:
        return 100 - int(x * 10)        # Ej: 3 → 70%
    elif x > 100:
        return 100                      # No puede exceder el 100%
    else:
        return x                        # Valor ya válido, no se modifica

# Aplicar la corrección a la columna
meals['amount_consumed'] = meals['amount_consumed'].apply(corregir_amount_consumed)



In [386]:
meals['amount_consumed'].value_counts()

amount_consumed
100.0    1290
90.0      209
80.0       36
70.0       32
75.0       30
60.0       19
50.0       16
40.0        5
10.0        2
25.0        1
20.0        1
30.0        1
Name: count, dtype: int64

Ahora tenemos una columna `meal_consumed` totalmente válida y coherente para el estudio. Por medio de la cual, podemos hacer una mejor imputación de nulos.

Hay que tomar en cuenta que la gran mayoría de los eventos de comida se terminaron por completo, es decir, un valor de 100 es la mejor opción para la imputación, siendo la moda por gran mayoría de este dato.

In [387]:
meals['amount_consumed'].isna().sum()

64

In [388]:
meals['amount_consumed'] = meals['amount_consumed'].fillna(100)
meals['amount_consumed'].isna().sum()

0

### `fiber`

In [389]:
meals['fiber'].value_counts()

fiber
0.0      586
4.0      182
5.0      135
1.0      106
7.0      101
        ... 
583.0      1
460.0      1
180.0      1
916.0      1
26.0       1
Name: count, Length: 63, dtype: int64

In [390]:
print(meals['fiber'].describe())
print("\nNulos en fiber:", meals['fiber'].isna().sum())


count    1705.000000
mean       17.897243
std       143.671402
min         0.000000
25%         0.000000
50%         4.000000
75%         7.000000
max      2830.000000
Name: fiber, dtype: float64

Nulos en fiber: 1


In [391]:
# Visualización uniforme de la variable 'fiber'
plt.figure(figsize=(12, 4))

# Histograma
plt.subplot(1, 2, 1)
sns.histplot(meals['fiber'].dropna(), bins=20, kde=True, color='#4C72B0')
plt.title('Distribución de fibra', fontsize=12)
plt.xlabel('Gramos de fibra', fontsize=10)
plt.ylabel('Frecuencia', fontsize=10)

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=meals['fiber'], color='#4C72B0')
plt.title('Boxplot de fibra', fontsize=12)
plt.xlabel('Gramos de fibra', fontsize=10)

plt.tight_layout()
plt.show()


<Figure size 1200x400 with 2 Axes>

In [392]:
# Mapa de calor de correlaciones entre variables nutricionales
num_cols = ['calories', 'carbs', 'protein', 'fat', 'fiber', 'amount_consumed']
corr_matrix = meals[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlación entre variables nutricionales', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


<Figure size 800x600 with 2 Axes>

La variable `fiber` presenta una distribución muy sesgada a la derecha, con muchos valores cercanos a 0 y varios outliers extremos, probablemente errores o comidas atípicas. El boxplot confirma esta asimetría y muestra un IQR reducido (0 a 7). Tiene baja correlación con otras variables (máxima de 0.32 con fat), por lo que no es útil para modelos predictivos basados en otras columnas. Su media (17.9) y desviación estándar (143.7) están infladas por los outliers, mientras que la mediana es 4. Solo hay un valor nulo, lo que facilita su tratamiento

In [393]:
Q1 = meals['fiber'].quantile(0.25)
Q3 = meals['fiber'].quantile(0.75)
IQR = Q3 - Q1

# Límite superior sugerido
upper_limit = Q3 + 1.5 * IQR
print("Límite superior para fiber:", upper_limit)


outliers_fiber = meals[meals['fiber'] > upper_limit]
print(f"Número de outliers encontrados: {len(outliers_fiber)}")
outliers_fiber


Límite superior para fiber: 17.5
Número de outliers encontrados: 89


,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
31941,2020-03-11 12:09:00,77.800000,111.0,80.0,0.97317,11.0,Lunch,1180.0,81.0,88.0,54.5,18.0,50.0,photos/00000005-PHOTO-2020-3-11-12-9-0.jpg,3
46416,2023-09-11 11:30:00,77.866667,104.6,97.0,5.59020,42.0,Lunch,1180.0,81.0,88.0,54.5,18.0,100.0,photos/00000005-PHOTO-2023-9-11-11-30-0.jpg,4
60709,2020-08-17 13:08:00,87.800000,116.0,89.0,2.80980,30.0,Lunch,1180.0,81.0,88.0,54.5,18.0,100.0,photos/00000005-PHOTO-2020-8-17-13-8-0.jpg,5
75164,2023-04-07 13:08:00,68.800000,107.0,71.0,1.19317,11.0,lunch,1180.0,81.0,88.0,54.5,18.0,100.0,photos/00000005-PHOTO-2023-4-7-13-8-0.jpg,6
78378,2023-04-09 18:42:00,74.000000,106.0,74.0,2.82022,26.0,dinner,588.0,47.0,16.0,42.0,21.0,100.0,photos/00000021-PHOTO-2023-4-9-18-42-0.jpg,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639430,2025-10-30 12:04:00,126.000000,161.2,74.0,2.57832,24.0,lunch,1180.0,81.0,88.0,54.5,18.0,75.0,photos/00000008-PHOTO-2025-10-30-12-4-0.jpg,47
655153,2022-11-14 13:02:00,55.800000,84.0,86.0,1.00628,11.0,lunch,1180.0,81.0,88.0,54.5,18.0,75.0,photos/00000005-PHOTO-2022-11-14-13-2-0.jpg,48
672444,2025-05-11 11:52:00,120.000000,152.2,NaN,1.07470,12.0,lunch,1180.0,81.0,88.0,54.5,18.0,75.0,photos/00000006-PHOTO-2025-5-11-11-52-0.jpg,49
674417,2025-05-12 20:45:00,115.533333,145.0,81.0,0.97750,11.0,dinner,393.0,42.0,13.0,22.0,21.0,100.0,photos/00000021-PHOTO-2025-5-12-20-45-0.jpg,49


In [394]:
meals[meals['fiber'] > 50]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
220046,2023-07-14 11:58:00,66.066667,103.0,83.0,1.75357,13.0,snack,280.0,38.0,4.0,12.0,480.0,90.0,photos/00000005-PHOTO-2023-7-14-11-58-0.jpg,16
221501,2023-07-15 12:13:00,61.933333,93.0,85.0,1.75357,13.0,snack,460.0,45.0,12.0,23.0,230.0,70.0,photos/00000011-PHOTO-2023-7-15-12-13-0.jpg,16
222934,2023-07-16 12:06:00,72.200000,103.0,95.0,4.58626,34.0,snack,220.0,19.0,7.0,13.0,510.0,90.0,photos/00000019-PHOTO-2023-7-16-12-6-0.jpg,16
226048,2023-07-18 16:00:00,77.600000,100.4,85.0,1.75357,13.0,snack,580.0,73.0,7.0,31.0,230.0,80.0,NaN,16
226288,2023-07-18 20:00:00,102.800000,118.2,78.0,1.34890,10.0,dinner,30.0,2.0,2.0,2.0,55.0,90.0,NaN,16
227728,2023-07-19 20:00:00,84.000000,110.4,114.0,7.55384,56.0,dinner,440.0,38.0,14.0,26.0,1020.0,90.0,NaN,16
228928,2023-07-20 16:00:00,73.200000,108.6,93.0,1.75357,13.0,snack,410.0,63.0,6.0,15.0,480.0,90.0,NaN,16
229168,2023-07-20 20:00:00,110.000000,155.2,88.0,1.34890,10.0,dinner,1590.0,107.0,54.0,508.0,2830.0,80.0,NaN,16
230608,2023-07-21 20:00:00,79.000000,103.0,90.0,1.75357,13.0,dinner,1250.0,17.0,44.0,27.0,904.0,70.0,NaN,16
232048,2023-07-22 20:00:00,99.400000,135.0,86.0,1.75357,13.0,dinner,870.0,110.0,69.0,28.0,2670.0,60.0,NaN,16


In [395]:
# Filtrar los primeros 5 registros con fibra > 50 y con imagen disponible
registros = meals[(meals['fiber'] > 50) & (meals['image_path'].notna())].head(5)

# Crear subplots: 1 por imagen
fig, axs = plt.subplots(1, len(registros), figsize=(5 * len(registros), 5))

# Asegurar que axs sea iterable
if len(registros) == 1:
    axs = [axs]

# Mostrar imágenes
for i, (idx, row) in enumerate(registros.iterrows()):
    patient = int(row['patient'])
    img_path = row['image_path']
    full_path = f'CGMacros/CGMacros-0{patient:02d}/{img_path}'

    try:
        img = Image.open(full_path)
        axs[i].imshow(img)
        axs[i].axis('off')
        axs[i].set_title(f'Paciente {patient:02d}\nFibra: {row["fiber"]:.1f} g', fontsize=10, pad=6)
    except FileNotFoundError:
        axs[i].axis('off')
        axs[i].set_title("Imagen no encontrada", fontsize=10, pad=6)

plt.tight_layout()
plt.show()

<Figure size 2500x500 with 5 Axes>

In [396]:
# Filtrar los primeros 5 registros con fibra > 50 y con imagen disponible
registros = meals[(meals['fiber'] > 1000) & (meals['image_path'].notna())].head(5)

# Crear subplots: 1 por imagen
fig, axs = plt.subplots(1, len(registros), figsize=(5 * len(registros), 5))

# Asegurar que axs sea iterable
if len(registros) == 1:
    axs = [axs]

# Mostrar imágenes
for i, (idx, row) in enumerate(registros.iterrows()):
    patient = int(row['patient'])
    img_path = row['image_path']
    full_path = f'CGMacros/CGMacros-0{patient:02d}/{img_path}'

    try:
        img = Image.open(full_path)
        axs[i].imshow(img)
        axs[i].axis('off')
        axs[i].set_title(f'Paciente {patient:02d}\nFibra: {row["fiber"]:.1f} g', fontsize=10, pad=6)
    except FileNotFoundError:
        axs[i].axis('off')
        axs[i].set_title("Imagen no encontrada", fontsize=10, pad=6)

plt.tight_layout()
plt.show()

<Figure size 2500x500 with 5 Axes>

En este punto podemos encontrar errores de registro graves, teniendo en cuenta un límite fisiológico de 50g de fibra:
* La primer comida que se muestra, tiene 480 gramos, cuando un alimento como ese, suele tener entre 15-18 gramos de fibra.
* Las demás imágenes también tienen datos que no son realistas, más de 50g de fibra cada uno.

También hay valores con más de 1000g de fibra, que son muy incoherentes, es posible que sea un error en la entrada de datos, porque 1086 es un valor erróneo, pero 10.86 puede ser un valor realista. Igual que pasa con 2300 y 23g.

El tratamiento que se realizará es el siguiente:
1. Cuando el valor de fibra sea mayor a 1000, se dividirá entre 100.
2. Si los valores son mayores a 100, se dividen entre 10.
3. También hacemos capping a todos los valores que son outliers en el rango intercuartil.



In [397]:
# Función de tratamiento
def tratar_fiber(x):
    if pd.isna(x):
        return x 
    
    # Corrección de escala
    if x > 1000:
        x = x / 100
    elif x > 100:
        x = x / 10
        
    # Capping si aún es outlier
    if x > upper_limit:
        x = upper_limit
    return x

Aplicamos la función, y el valor nulo que existía, será rellando con la mediana.

In [398]:
# Aplicamos el tratamiento
meals['fiber'] = meals['fiber'].apply(tratar_fiber)

# Imputamos el nulo con la mediana final
meals['fiber'] = meals['fiber'].fillna(meals['fiber'].median())

In [399]:
# Distribución de la variable 'fiber' ya tratada
plt.figure(figsize=(7, 5))
sns.histplot(meals['fiber'], bins=30, kde=True, color='#4C72B0')
plt.title("Distribución de fibra (valores corregidos)", fontsize=12)
plt.xlabel("Fibra (g)", fontsize=10)
plt.ylabel("Frecuencia", fontsize=10)
plt.tight_layout()
plt.show()


<Figure size 700x500 with 1 Axes>

In [400]:
meals['fiber'].describe()

count    1706.000000
mean        4.673206
std         5.181006
min         0.000000
25%         0.000000
50%         4.000000
75%         7.000000
max        17.500000
Name: fiber, dtype: float64

### Análisis final de meals

El dataset de meals, que incluye los registros de los eventos de comida, tiene todos los datos de los eventos de comida completos, sin tomar en cuenta datos que son sensores de medición.

In [401]:
meals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1706 entries, 233 to 686614
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp            1706 non-null   datetime64[ns]
 1   libre_gl             1705 non-null   float64       
 2   dexcom_gl            1684 non-null   float64       
 3   hr                   1625 non-null   float64       
 4   calories_(activity)  1684 non-null   float64       
 5   mets                 1320 non-null   float64       
 6   meal_type            1706 non-null   object        
 7   calories             1706 non-null   float64       
 8   carbs                1706 non-null   float64       
 9   protein              1706 non-null   float64       
 10  fat                  1706 non-null   float64       
 11  fiber                1706 non-null   float64       
 12  amount_consumed      1706 non-null   float64       
 13  image_path           1644 non-null

# **6. Exploratory Data Analysis (EDA)** <a class="anchor" id="6"></a>

[Tabla de Contenidos](#0.1)

Ahora que se tienen los datasets "correctamente ordenados", es decir, sin valores nulos, sin registros de "pacientes" que no son tomados en los demás conjuntos. Se procede a realizar un análisis exploratorio de los datos.

En la sección anterior se analizaron dos variables que tenían datos nulos, y para su corrección, se hizo una limpieza de datos en estas dos columnas en cuanto a outliers y el problema de valores nulos: `amount_consumed` y `fiber`.

En esta sección se analizan las demás variables del conjunto de meals, además de las variables de los conjuntos suplementarios.

## Dataset: `meals`

Es importante tomar en cuenta que solamente se realiza análisis exploratorio de los datos específicos para los eventos de comida, los datos de mediciones de sensores continuos, se analizan posteriormente.

Primero se analizan las variables de macronutrientes que no se han analizado para poder hacer análisis de las demás variables con estas variables ya corregidas en caso de necesitarlo.

### `carbs`

In [402]:
# Función para analizar un macronutriente (histograma, boxplot, estadísticos y outliers)
def analizar_macronutriente(df, columna):
    print(f"\nAnálisis de '{columna}'")
    print(df[columna].describe())

    # Gráficos: Histograma + KDE y Boxplot
    plt.figure(figsize=(12, 4))

    # Histograma
    plt.subplot(1, 2, 1)
    sns.histplot(df[columna].dropna(), bins=30, kde=True, color='#4C72B0')
    plt.title(f"Distribución de {columna}", fontsize=12)
    plt.xlabel(f"{columna} (g)", fontsize=10)
    plt.ylabel("Frecuencia", fontsize=10)

    # Boxplot
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[columna], color='#4C72B0')
    plt.title(f"Boxplot de {columna}", fontsize=12)
    plt.xlabel(f"{columna} (g)", fontsize=10)

    plt.tight_layout()
    plt.show()

    # Cálculo de outliers usando el método IQR
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[columna] < lower) | (df[columna] > upper)]
    print(f"Outliers detectados: {len(outliers)}")

    return outliers



In [403]:
outliers_carbs = analizar_macronutriente(meals, 'carbs')


Análisis de 'carbs'
count    1706.000000
mean       52.080715
std        40.066376
min         0.000000
25%        24.000000
50%        50.000000
75%        73.000000
max       761.000000
Name: carbs, dtype: float64


<Figure size 1200x400 with 2 Axes>

Outliers detectados: 20


`carbs`presenta una distribución sesgada a la derecha, con una cola larga y outliers extremos superiores a 200–700g.

In [404]:
meals[meals['carbs'] > meals['carbs'].quantile(0.99)]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
51240,2023-09-14 19:54:00,73.666667,100.0,97.0,1.73030,13.0,Dinner,1860.0,204.0,102.0,72.0,9.0,100.0,photos/00000040-PHOTO-2023-9-14-19-54-0.jpg,4
56914,2023-09-18 18:28:00,89.666667,100.6,89.0,1.73030,13.0,Dinner,1592.0,229.0,27.0,67.0,16.0,100.0,photos/00000091-PHOTO-2023-9-18-18-28-0.jpg,4
97198,2023-01-31 20:40:00,84.800000,94.4,81.0,1.17821,11.0,dinner,1680.0,168.0,50.0,110.0,0.0,100.0,photos/00000013-PHOTO-2023-1-31-20-40-0.jpg,8
148164,2020-10-08 17:50:00,137.666667,143.8,82.0,1.57443,NaN,dinner,1365.0,163.0,68.0,31.0,17.5,100.0,photos/dinner-PHOTO-2020-10-8-17-50-0.jpg,11
149597,2020-10-09 17:43:00,122.400000,135.2,80.0,1.57443,NaN,dinner,1460.0,168.0,56.0,56.0,8.0,100.0,photos/00000052-PHOTO-2020-10-9-17-43-0.jpg,11
253887,2024-02-07 18:39:00,87.266667,124.6,97.0,5.48910,38.0,Dinner,1736.0,276.0,108.0,66.0,17.5,60.0,photos/00000013-PHOTO-2024-2-7-18-39-0.jpg,18
285722,2024-03-22 16:28:00,98.733333,118.0,103.0,2.88372,28.0,dinner,1530.0,190.0,46.0,69.0,15.0,50.0,photos/00000044-PHOTO-2024-3-22-16-28-0.jpg,20
294500,2024-03-28 18:46:00,122.666667,159.0,100.0,1.02990,10.0,dinner,1119.0,159.0,51.0,35.0,8.0,70.0,photos/00000117-PHOTO-2024-3-28-18-46-0.jpg,20
299301,2020-06-09 20:23:00,97.200000,107.0,65.0,2.08806,26.0,dinner,470.0,382.0,28.0,22.0,6.0,70.0,photos/00000016-PHOTO-2020-6-9-20-23-0.jpg,21
300785,2020-06-10 21:07:00,124.666667,125.4,66.0,0.88341,11.0,snack,1050.0,255.0,15.0,0.0,0.0,90.0,photos/00000030-PHOTO-2020-6-10-21-7-0.jpg,21


In [405]:
# Limpiar registros con imagen disponible y valores altos de carbs
outliers_carbs = meals[(meals['carbs'] > 146.5) & (meals['image_path'].notna())].head(5)

# Gráfica
fig, axs = plt.subplots(1, len(outliers_carbs), figsize=(5 * len(outliers_carbs), 5))

if len(outliers_carbs) == 1:
    axs = [axs]

for i, (idx, row) in enumerate(outliers_carbs.iterrows()):
    img_path = row['image_path']
    patient = int(row['patient'])
    full_path = f'CGMacros/CGMacros-0{patient:02d}/{img_path}'

    try:
        img = Image.open(full_path)
        axs[i].imshow(img)
        axs[i].axis('off')
        axs[i].set_title(f'Paciente {patient}\nCarbs: {row["carbs"]:.1f}g')
    except FileNotFoundError:
        axs[i].axis('off')
        axs[i].set_title("Imagen no encontrada")

plt.tight_layout()
plt.show()

<Figure size 2500x500 with 5 Axes>

Se puede observar que existen errores de escala en los registros, pues las comidas que es muestran en las imágenes no pueden tener la cantidad tan grande de carbohidratos que se menciona en los registros.

Por ejemplo, una rebanada de pizza suele tener entre 30-40g de carbohidratos, y en la imagen se aprecian 2 rebanadas, con registro de 204g.
Además hay un máximo de hasta 761 gramos, lo cual es muy poco probable para una comida. Es por esto que se procede a la limpieza de la columna de la siguiente manera:

1. Los valores extremos los vamos a dividir entre 100, se asume un error de escala de la aplicación o al momento del registro.
2. Los valores que son extremos pero no tanto, se dividen entre 10.
3. Cuando son valores mayores a 180, se divide entre 2, porque se observó alguna inconsistencia que puede corregirse de esta forma.
4. Después se hara capping para quitar outliers según el rango intercuartil.

In [406]:
def corregir_carbs(x):
    if pd.isna(x):
        return x
    elif x > 500:
        return x / 100
    elif x > 250:
        return x / 10
    elif x > 180:
        return x / 2  # Casos como la pizza
    else:
        return x


meals['carbs'] = meals['carbs'].apply(corregir_carbs)


In [407]:
Q1 = meals['carbs'].quantile(0.25)
Q3 = meals['carbs'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

meals['carbs'] = meals['carbs'].apply(lambda x: min(x, upper_limit) if pd.notna(x) else x)

In [408]:
outliers_carbs = analizar_macronutriente(meals, 'carbs')


Análisis de 'carbs'
count    1706.000000
mean       50.410615
std        31.428291
min         0.000000
25%        24.000000
50%        49.000000
75%        73.000000
max       146.500000
Name: carbs, dtype: float64


<Figure size 1200x400 with 2 Axes>

Outliers detectados: 0


In [409]:
meals['carbs'].mode()[0] 


66.0

De esta forma, la variable `carbs` ha sido efectivamente depurada, combinando corrección de errores de escala con capping según IQR, con criterio visual y fisiológico.

En cuanto a análisis, se puede observar que la mayoría de las comidas se encuentran en un rango de 20 a 80g de carbohidratos. Ahora se tiene una distribución aún con un poco de sesgo positivo, pero mucho menor al anterior, con media en 50.4g y moda de 66g.

### `fiber`

In [410]:
analizar_macronutriente(meals, 'fiber')


Análisis de 'fiber'
count    1706.000000
mean        4.673206
std         5.181006
min         0.000000
25%         0.000000
50%         4.000000
75%         7.000000
max        17.500000
Name: fiber, dtype: float64


<Figure size 1200x400 with 2 Axes>

Outliers detectados: 0


,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient


### `protein`

In [411]:
outliers_protein = analizar_macronutriente(meals, 'protein')


Análisis de 'protein'
count    1706.000000
mean       29.247421
std        25.662044
min         0.000000
25%        10.000000
50%        22.000000
75%        44.000000
max       148.000000
Name: protein, dtype: float64


<Figure size 1200x400 with 2 Axes>

Outliers detectados: 7


La variable `protein` tiene distribución sesgada a la derecha (comidas con mucha proteína). También tiene algunos outliers presentes pero pocos (7/1706), con un máximo registrado de casi 150g, lo cual es muy poco común salvo en comidas proteicas muy muy grandes.

En general:
* Valores hasta 90–100g de proteína son posibles, aunque no comunes.
* 148g probablemente sea un error o reflejo de una comida exageradamente alta en carne.

A diferencia de `carbs`, los valores altos no parecen errores de escala, sino más bien outliers naturales.

In [412]:
# Calculamos el límite superior
Q3 = meals['protein'].quantile(0.75)
IQR = Q3 - meals['protein'].quantile(0.25)
upper_limit = Q3 + 1.5 * IQR

# Filtramos outliers que tengan imagen
outliers_protein = meals[(meals['protein'] > upper_limit) & (meals['image_path'].notna())]
outliers_protein

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
51240,2023-09-14 19:54:00,73.666667,100.0,97.0,1.73030,13.0,Dinner,1860.0,102.0,102.0,72.0,9.0,100.0,photos/00000040-PHOTO-2023-9-14-19-54-0.jpg,4
58345,2023-09-19 18:19:00,86.666667,101.0,95.0,1.73030,13.0,Dinner,2250.0,124.0,116.0,141.0,16.0,100.0,photos/00000101-PHOTO-2023-9-19-18-19-0.jpg,4
107111,2023-02-07 17:53:00,76.200000,116.0,86.0,1.28532,12.0,dinner,1734.0,78.0,148.0,91.0,15.0,100.0,photos/00000056-PHOTO-2023-2-7-17-53-0.jpg,8
145359,2020-10-06 19:05:00,89.066667,111.6,56.0,1.21110,NaN,dinner,689.0,22.0,119.0,13.0,10.0,100.0,photos/00000029-PHOTO-2020-10-6-19-5-0.jpg,11
253887,2024-02-07 18:39:00,87.266667,124.6,97.0,5.48910,38.0,Dinner,1736.0,27.6,108.0,66.0,17.5,60.0,photos/00000013-PHOTO-2024-2-7-18-39-0.jpg,18
258189,2024-02-10 18:21:00,94.000000,122.0,88.0,2.16675,15.0,Dinner,963.0,20.0,97.0,52.0,3.0,90.0,photos/00000037-PHOTO-2024-2-10-18-21-0.jpg,18
534426,2025-02-07 17:54:00,103.400000,143.0,NaN,2.82400,32.0,dinner,1331.0,87.0,96.0,66.0,12.0,100.0,photos/00000012-PHOTO-2025-2-7-17-54-0.jpg,39


In [413]:
# Visualización de imágenes correspondientes a los outliers en proteína
fig, axs = plt.subplots(1, len(outliers_protein), figsize=(5 * len(outliers_protein), 5))

# Asegurar que axs sea iterable
if len(outliers_protein) == 1:
    axs = [axs]

# Mostrar cada imagen con su valor de proteína
for i, (idx, row) in enumerate(outliers_protein.iterrows()):
    patient = int(row['patient'])
    img_path = row['image_path']
    full_path = f'CGMacros/CGMacros-0{patient:02d}/{img_path}'

    try:
        img = Image.open(full_path)
        axs[i].imshow(img)
        axs[i].axis('off')
        axs[i].set_title(f'Paciente {patient:02d}\nProteína: {row["protein"]:.1f} g', fontsize=10, pad=6)
    except FileNotFoundError:
        axs[i].axis('off')
        axs[i].set_title("Imagen no encontrada", fontsize=10, pad=6)

plt.tight_layout()
plt.show()

<Figure size 3500x500 with 7 Axes>

Es interesante que algunos de los registros son los mismos que aparecían en los errores de escala en el análisis de `carbs`.

Se pueden observar platillos relativamente "normales", ninguno que tenga una cantidad de proteína, en todo caso algunos parecen errores de escala leve (multiplicado x2) y otros son claramente irreales (148g, 119g, etc.).

Por lo tanto la limpieza se realizará así, siguiendo umbrales fisiológicamente improbables:
1. Corregir valores extremos según su rango:
    1. ">200g": casi imposible → divide entre 100
    1. 100g–200g: sospechoso → divide entre 10
    1. 70g–100g: posible pero poco probable → divide entre 2
2. Capping de valores hasta el rango intercuartil.



In [414]:
def limpiar_protein(x):
    if pd.isna(x):
        return x
    elif x > 200:
        return x / 100   # Error muy grave: 210 → 2.1
    elif x > 100:
        return x / 10    # Error moderado: 148 → 14.8
    elif x > 70:
        return x / 2     # Posible error leve: 80 → 40
    else:
        return x

meals['protein'] = meals['protein'].apply(limpiar_protein)


In [415]:
Q1 = meals['protein'].quantile(0.25)
Q3 = meals['protein'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

# Aplicamos capping final
meals['protein'] = meals['protein'].apply(lambda x: min(x, upper_limit) if pd.notna(x) else x)

In [416]:
outliers_protein = analizar_macronutriente(meals, 'protein')


Análisis de 'protein'
count    1706.000000
mean       25.520457
std        19.899525
min         0.000000
25%        10.000000
50%        22.000000
75%        38.000000
max        69.000000
Name: protein, dtype: float64


<Figure size 1200x400 with 2 Axes>

Outliers detectados: 0


In [417]:
meals['protein'].mode()[0]

22.0

La variable `protein` ya está lista para análisis, tiene sus picos representativos de comidas con más o menos proteínas, sin puntos extremos ni ruido. 
Se puede observar una distribución con un poco de sesgo positivo, con media de 25g de proteína, y moda de 22g, coincidente con la mediana.

### `fat`

In [418]:
outliers_fat = analizar_macronutriente(meals, 'fat')


Análisis de 'fat'
count    1706.000000
mean       19.783763
std        20.752374
min         0.000000
25%        10.000000
50%        14.000000
75%        30.000000
max       508.000000
Name: fat, dtype: float64


<Figure size 1200x400 with 2 Axes>

Outliers detectados: 25


La distribución está muy sesgada a la derecha y presenta valores inflados, como 508g, probablemente por errores de escala (ej. 50.8 → 508), similares a los observados en fiber y carbs.

In [419]:
# Filtramos outliers de fat con imagen disponible
outliers_fat = meals[(meals['fat'] > 60) & (meals['image_path'].notna())].head(7)
outliers_fat

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
51240,2023-09-14 19:54:00,73.666667,100.0,97.0,1.73030,13.0,Dinner,1860.0,102.0,10.2,72.0,9.0,100.0,photos/00000040-PHOTO-2023-9-14-19-54-0.jpg,4
56914,2023-09-18 18:28:00,89.666667,100.6,89.0,1.73030,13.0,Dinner,1592.0,114.5,27.0,67.0,16.0,100.0,photos/00000091-PHOTO-2023-9-18-18-28-0.jpg,4
58345,2023-09-19 18:19:00,86.666667,101.0,95.0,1.73030,13.0,Dinner,2250.0,124.0,11.6,141.0,16.0,100.0,photos/00000101-PHOTO-2023-9-19-18-19-0.jpg,4
97198,2023-01-31 20:40:00,84.800000,94.4,81.0,1.17821,11.0,dinner,1680.0,146.5,50.0,110.0,0.0,100.0,photos/00000013-PHOTO-2023-1-31-20-40-0.jpg,8
105724,2023-02-06 18:46:00,82.400000,95.6,76.0,1.17821,11.0,dinner,2015.0,42.0,12.0,86.0,0.0,100.0,photos/00000050-PHOTO-2023-2-6-18-46-0.jpg,8
107111,2023-02-07 17:53:00,76.200000,116.0,86.0,1.28532,12.0,dinner,1734.0,78.0,14.8,91.0,15.0,100.0,photos/00000056-PHOTO-2023-2-7-17-53-0.jpg,8
123514,2020-09-24 19:29:00,81.800000,99.0,89.0,1.29857,13.0,dinner,1254.0,133.0,32.0,67.0,8.0,100.0,photos/00000080-PHOTO-2020-9-24-19-29-0.jpg,9


In [420]:
# Visualización de imágenes correspondientes a los outliers en grasa
fig, axs = plt.subplots(1, len(outliers_fat), figsize=(5 * len(outliers_fat), 5))

# Asegurar que axs sea iterable
if len(outliers_fat) == 1:
    axs = [axs]

# Mostrar cada imagen con su valor de grasa
for i, (idx, row) in enumerate(outliers_fat.iterrows()):
    patient = int(row['patient'])
    img_path = row['image_path']
    full_path = f'CGMacros/CGMacros-0{patient:02d}/{img_path}'

    try:
        img = Image.open(full_path)
        axs[i].imshow(img)
        axs[i].axis('off')
        axs[i].set_title(f'Paciente {patient:02d}\nGrasa: {row["fat"]:.1f} g', fontsize=10, pad=6)
    except FileNotFoundError:
        axs[i].axis('off')
        axs[i].set_title("Imagen no encontrada", fontsize=10, pad=6)

plt.tight_layout()
plt.show()

<Figure size 3500x500 with 7 Axes>

Se pueden observar alimentos que pueden ser altos en grasa, como la pizza, o la hamburguesa, pero las escalas son muy grandes comparadas con valores normales de grasa en estos tipos de alimentos.

Por esto la limpieza de esta variable es como sigue:
1. Valores extremadamente altos (>120g) se dividen entre 100, para mantener valores altos pero coherentes.
2. Aquellos valores altos para un rango común en comidas, se divide entre 2.
3. Posteriormente se hace capping según el rango intercuartil.

In [421]:
def limpiar_fat(x):
    if pd.isna(x):
        return x
    elif x > 120:
        return x / 10  
    elif x > 70:
        return x / 2    

In [422]:
# Capping por IQR
Q1 = meals['fat'].quantile(0.25)
Q3 = meals['fat'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

meals['fat'] = meals['fat'].apply(lambda x: min(x, upper_limit) if pd.notna(x) else x)


In [423]:
outliers_fat = analizar_macronutriente(meals, 'fat')


Análisis de 'fat'
count    1706.000000
mean       19.228664
std        16.024484
min         0.000000
25%        10.000000
50%        14.000000
75%        30.000000
max        60.000000
Name: fat, dtype: float64


<Figure size 1200x400 with 2 Axes>

Outliers detectados: 0


In [424]:
meals['fat'].mode()[0]

10.5

De esta forma, `fat` queda con una curva limpia, sin valores absurdos como los 500g iniciales. El boxplot está completamente dentro del rango lógico.
Se conservaron valores altos reales (como los ~40g en platos fritos o con queso), pero sin exageraciones.

Se observa una distribución con sesgo positivo, media de 19g, mediana de 14g y moda de 10.5g.

#### Resumen de los macronutrientes

In [425]:
# Selección de columnas y descripción estadística
summary_df = meals[['fiber', 'carbs', 'protein', 'fat']].describe()

# Exportar a CSV
summary_df.to_csv('macronutrient_summary.csv', index=True)
summary_df

,fiber,carbs,protein,fat
count,1706.000000,1706.000000,1706.000000,1706.000000
mean,4.673206,50.410615,25.520457,19.228664
std,5.181006,31.428291,19.899525,16.024484
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,24.000000,10.000000,10.000000
50%,4.000000,49.000000,22.000000,14.000000
75%,7.000000,73.000000,38.000000,30.000000
max,17.500000,146.500000,69.000000,60.000000


### `meal_type`
Esta variable representa el tipo de evento de comida que realiza el sujeto, puede ser de cuatro tipos:
* Desayuno
* Comida
* Cena
* Snack

In [426]:
meals['meal_type'].value_counts()

meal_type
dinner       418
snack        300
lunch        272
breakfast    266
Breakfast    170
Lunch        163
Dinner        74
Snacks        38
Snack          4
snack 1        1
Name: count, dtype: int64

In [427]:
def limpiar_y_mapear_tipo_comida(valor):

    if pd.isna(valor):
        return 'unknown'
    
    # Limpieza
    valor = str(valor).strip().lower()
    
    # Reglas por prefijo
    if valor.startswith('b'):
        return 'breakfast'
    elif valor.startswith('l'):
        return 'lunch'
    elif valor.startswith('d'):
        return 'dinner'
    elif valor.startswith('s'):
        return 'snack'
    else:
        return 'unknown'

In [428]:
# Aplicamos la función al campo 'meal_type'
meals['meal_type'] = meals['meal_type'].apply(limpiar_y_mapear_tipo_comida)

# Verificamos resultado
print(meals['meal_type'].value_counts())

meal_type
dinner       492
breakfast    436
lunch        435
snack        343
Name: count, dtype: int64


Ahora todos los registros de `meal_type` están bien registrados. Por lo que se procede con el análisis

In [429]:
# Porcentaje
print("\nDistribución porcentual:")
print(meals['meal_type'].value_counts(normalize=True) * 100)


Distribución porcentual:
meal_type
dinner       28.839390
breakfast    25.556858
lunch        25.498242
snack        20.105510
Name: proportion, dtype: float64


Los tipos de comida tienen un porcentaje similar, siendo la cena (dinner) la que mayor porcentaje hay (28.8%), y snack la que menos (20.1%). 

In [430]:
# Conteo de registros por tipo de comida
plt.figure(figsize=(7, 4))
sns.countplot(
    data=meals,
    x='meal_type',
    order=meals['meal_type'].value_counts().index,
    color='#4C72B0'
)

plt.title("Cantidad de registros por tipo de comida", fontsize=12)
plt.xlabel("Tipo de comida", fontsize=10)
plt.ylabel("Número de registros", fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


<Figure size 700x400 with 1 Axes>

Análisis de `meal_type` y los macronutrientes

In [431]:
meals_macros = meals.groupby('meal_type')[['carbs', 'fat', 'protein', 'fiber']].mean().round(2)
meals_macros.to_csv('meals_macros.csv', index=True)
meals_macros

,carbs,fat,protein,fiber
meal_type,,,,
breakfast,58.74,22.85,35.42,1.32
dinner,47.23,18.73,22.70,5.20
lunch,64.34,25.57,33.74,8.54
snack,26.71,7.30,6.56,3.28


Se puede observar que las comidas (lunch) son las comidas que tienen en general más carbohidratos, grasa, proteínas (el segundo) y fibra, es decir, suele ser el alimento más fuerte del día para los pacientes de este estudio, al menos en valores medios.

El desayuno (breakfast) suele ser el segundo más fuerte, siguiendo de la cena (dinner) y finalmente las colaciones (snack), que tiene sentido porque suelen ser comidas más pequeñas, entre comidas grandes.

In [432]:
# Distribución de carbohidratos por tipo de comida
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=meals,
    x='meal_type',
    y='carbs',
    color='#4C72B0'
)

plt.title("Distribución de carbohidratos por tipo de comida", fontsize=12)
plt.xlabel("Tipo de comida", fontsize=10)
plt.ylabel("Carbohidratos (g)", fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


<Figure size 800x500 with 1 Axes>

In [433]:
# Ver cuántos valores únicos de carbs hay en breakfast
meals[meals['meal_type'] == 'breakfast']


,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
1308,2020-05-02 08:18:00,88.400000,101.8,81.0,3.14520,30.0,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000010-PHOTO-2020-5-2-8-18-0.jpg,1
2797,2020-05-03 09:07:00,86.266667,95.2,93.0,1.57260,15.0,breakfast,608.0,66.0,66.0,10.5,0.0,100.0,photos/00000019-PHOTO-2020-5-3-9-7-0.jpg,1
4247,2020-05-04 09:17:00,88.000000,106.4,92.0,6.91944,66.0,breakfast,712.0,66.0,22.0,42.0,0.0,100.0,photos/00000030-PHOTO-2020-5-4-9-17-0.jpg,1
5692,2020-05-05 09:22:00,90.666667,105.2,91.0,3.56456,34.0,breakfast,902.0,73.0,66.0,42.0,7.0,100.0,photos/00000040-PHOTO-2020-5-5-9-22-0.jpg,1
7104,2020-05-06 08:54:00,94.200000,110.0,80.0,1.36292,13.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000047-PHOTO-2020-5-6-8-54-0.jpg,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
680803,2025-05-17 07:11:00,148.200000,153.0,85.0,1.02443,11.0,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000056-PHOTO-2025-5-17-7-11-0.jpg,49
682351,2025-05-18 08:59:00,134.600000,153.2,80.0,0.93130,10.0,breakfast,608.0,66.0,66.0,10.5,0.0,100.0,photos/00000063-PHOTO-2025-5-18-8-59-0.jpg,49
683678,2025-05-19 07:06:00,132.600000,164.8,80.0,0.93130,10.0,breakfast,712.0,66.0,22.0,42.0,0.0,100.0,photos/00000073-PHOTO-2025-5-19-7-6-0.jpg,49
685118,2025-05-20 07:06:00,138.333333,152.6,81.0,2.23512,24.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000085-PHOTO-2025-5-20-7-6-0.jpg,49


El almuerzo tiene alta energía y gran variedad de comidas. La cena muestra energía media pero con mucha variabilidad, desde cenas ligeras hasta muy pesadas. Los snacks suelen ser bajos en energía, aunque con algunos picos. El desayuno también muestra alta energía, pero con muy poca variabilidad.

Se verificó que no hay registros duplicados, y la limpieza de los valores de la variable de carbohidratos no hace que registros tomen valor de 66, lo cual es un dato que fue registrado de esta manera.

### `calories`

Esta variable indica cuántas calorías tiene el alimento en cuestión.

Por el análisis anterior, sabemos que no tiene ningún valor nulo.

In [434]:
meals['calories'].describe()

count    1706.000000
mean      505.858968
std       329.540743
min         0.000000
25%       268.000000
50%       448.000000
75%       712.000000
max      2826.000000
Name: calories, dtype: float64

In [435]:
# Distribución y boxplot de calorías
plt.figure(figsize=(12, 5))

# Histograma + KDE
plt.subplot(1, 2, 1)
sns.histplot(meals['calories'], kde=True, bins=50, color='#4C72B0')
plt.title("Distribución de calorías", fontsize=12)
plt.xlabel("Calorías", fontsize=10)
plt.ylabel("Frecuencia", fontsize=10)

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=meals['calories'], color='#4C72B0')
plt.title("Boxplot de calorías", fontsize=12)
plt.xlabel("Calorías", fontsize=10)

plt.tight_layout()
plt.show()

<Figure size 1200x500 with 2 Axes>

In [436]:
Q1 = meals['calories'].quantile(0.25)
Q3 = meals['calories'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_calories = meals[(meals['calories'] < lower) | (meals['calories'] > upper)]
print(f"Outliers detectados: {outliers_calories.shape[0]}")


Outliers detectados: 17


`calories` presenta una distribución asimétrica positiva, con densidad alta entre 200 y 800 kcal y varios valores extremos por encima de 1378 kcal, incluyendo uno poco realista de 2826 kcal. Aunque en general está bien distribuida, algunos outliers no fisiológicos (mayores a 2000 kcal) podrían afectar los modelos. La mediana (448 kcal) y la moda visual (300–500 kcal) sugieren que la mayoría de las comidas están en un rango razonable.}

De cualquier manera, se procede con la limpieza de outliers analizando los registros.

In [437]:
# Número de imágenes a mostrar (puedes cambiarlo)
n = 5
subset = outliers_calories.head(n)

# Mostrar imágenes
fig, axs = plt.subplots(1, n, figsize=(4 * n, 4))

for i, (_, row) in enumerate(subset.iterrows()):
    img_path = f"CGMacros/CGMacros-0{int(row['patient']):02d}/{row['image_path']}"
    try:
        img = Image.open(img_path)
        axs[i].imshow(img)
        axs[i].axis('off')
        axs[i].set_title(f"Paciente {int(row['patient'])}\nCalories: {row['calories']} kcal")
    except Exception as e:
        axs[i].text(0.5, 0.5, 'Error al cargar', ha='center', va='center')
        axs[i].axis('off')

plt.tight_layout()
plt.show()

<Figure size 2000x400 with 5 Axes>

Aunque algunas de estas comidas sí parecen energéticas, muchas de estas cifras superan con creces los valores comunes para una comida típica (~400 a 900 kcal).

Es probable que estén sobreestimadas o mal registradas (especialmente valores >2000 kcal).

Para la limpieza de los outliers en este caso, preservando valores altos, pero manteniendolos en un margen fisiológico posible, se hara capping hasta el límite intercuartil.

In [438]:
# Límite superior
calories_cap = meals['calories'].quantile(0.75) + 1.5 * (meals['calories'].quantile(0.75) - meals['calories'].quantile(0.25))

# Aplicar capping
meals['calories'] = meals['calories'].clip(upper=calories_cap)

In [439]:
# Distribución y boxplot de calorías
plt.figure(figsize=(12, 5))

# Histograma + KDE
plt.subplot(1, 2, 1)
sns.histplot(meals['calories'], kde=True, bins=50, color='#4C72B0')
plt.title("Distribución de calorías", fontsize=12)
plt.xlabel("Calorías", fontsize=10)
plt.ylabel("Frecuencia", fontsize=10)

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=meals['calories'], color='#4C72B0')
plt.title("Boxplot de calorías", fontsize=12)
plt.xlabel("Calorías", fontsize=10)

plt.tight_layout()
plt.show()

# Mostrar la moda
print(f"La moda de la variable 'calories' es: {meals['calories'].mode()[0]}")

# Descripción estadística
meals['calories'].describe()


<Figure size 1200x500 with 2 Axes>

La moda de la variable 'calories' es: 268.0


count    1706.000000
mean      501.802696
std       314.084677
min         0.000000
25%       268.000000
50%       448.000000
75%       712.000000
max      1378.000000
Name: calories, dtype: float64

Finalmente tenemos la variable `calories` sin outliers, que parece tener una distribución con un sesgo a la derecha, teniendo una media de 501 calorias, mediana de 448 cal y moda en 268, que de hecho corresponde con el cuartil 1.

La distribución ahora es más controlada: se conserva la forma asimétrica, pero sin valores extremos que distorsionen el gráfico.
La moda de 268 kcal resalta un pico bajo en la ingesta calórica, probablemente asociada a snacks o desayunos ligeros.

### Análisis multvariado entre variables nutricionales

In [440]:
# Matriz de correlación entre variables nutricionales
target_vars = ['calories', 'carbs', 'protein', 'fat', 'fiber']
corr = meals[target_vars].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)

plt.title("Matriz de correlación entre variables nutricionales", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


<Figure size 800x600 with 2 Axes>

Con este mapa de correlaciones entre las variables nutricionales, podemos observar que:
- Las variables `carbs`, `protein` y `fat` tienen una correlación positiva moderada entre sí, lo que es esperado ya que son macronutrientes que suelen estar presentes en las comidas.
- La variable `fiber` tiene una correlación positiva más débil con `carbs`, `protein` y `fat`, lo que sugiere que las comidas con más fibra no necesariamente tienen más de los otros macronutrientes.
- La variable `calories` está más correlacionada con la variable `fat` y con `carbs`, lo que indica que a mayor cantidad de carbohidratos y de grasas, más calorías se ven en los alimentos, lo que tiene mucho sentido nutricional.

Ahora el dataset de meals está completamente ordenado, excepto por los valores de sensores continuos, que se analizarán más adelante.


## Conjunto de datos: `bio`

Este conjunto de datos contiene información de perfil clínico y demográfico de los participantes al inicio del estudio.

In [441]:
bio

,subject,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2
0,1,27,M,22.265239,133.8,65.00,Hispanic/Latino,5.4,91,2.5,67,216,74,142,130,13,2.9,11:06:00 AM,89,9:40,73,12:11,81,13:18
1,2,49,F,30.946742,169.2,62.00,Hispanic/Latino,5.5,93,14.8,61,181,91,90,78,12,2.0,7:38:00 AM,91,7:52,123,9:21,80,10:22
2,3,59,F,26.948690,157.0,64.00,Hispanic/Latino,6.5,118,17.4,154,190,74,116,90,31,2.6,7:25:00 AM,119,7:38,166,9:23,98,10:23
3,4,33,F,42.384279,262.6,66.00,Hispanic/Latino,5.5,105,19.4,300,267,46,221,164,60,5.8,7:20:00 AM,109,7:37,110,9:04,90,10:01
4,5,51,F,30.957534,172.0,62.50,Hispanic/Latino,6.6,144,12.9,392,269,38,231,157,78,7.1,7:45:00 AM,139,8:59,215,10:52,130,11:54
5,6,51,F,29.303451,197.0,68.75,White,5.2,96,6.4,75,203,72,131,118,15,2.8,7:45:00 AM,98,9:04,97,10:54,70,11:56
6,7,66,F,27.070327,199.6,72.00,Hispanic/Latino,5.9,108,15.9,92,128,43,85,67,18,3.0,8:00:00 AM,115,8:40,157,10:22,95,11:24
7,8,54,M,39.945440,218.4,62.00,Hispanic/Latino,5.8,112,17.7,145,180,60,120,95,29,3.0,8:55:00 AM,110,9:07,156,10:25,94,11:26
8,9,34,F,37.001506,183.2,59.00,Hispanic/Latino,5.7,122,25.7,312,202,40,162,108,62,5.1,7:32:00 AM,119,7:46,128,9:32,96,10:29
9,10,54,F,35.811892,195.8,62.00,Hispanic/Latino,5.7,100,15.3,101,183,55,128,110,20,3.3,7:44:00 AM,100,8:01,160,9:35,108,10:36


In [442]:
bio.info()
bio.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   subject                     45 non-null     int64  
 1   age                         45 non-null     int64  
 2   gender                      45 non-null     object 
 3   bmi                         45 non-null     float64
 4   body_weight                 45 non-null     float64
 5   height                      45 non-null     float64
 6   self_identify               45 non-null     object 
 7   a1c_pdl_(lab)               45 non-null     float64
 8   fasting_glu_pdl_(lab)       45 non-null     int64  
 9   insulin                     45 non-null     float64
 10  triglycerides               45 non-null     int64  
 11  cholesterol                 45 non-null     int64  
 12  hdl                         45 non-null     int64  
 13  non_hdl                     45 non-nu

,subject,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2
count,45.000000,45.000000,45,45.000000,45.000000,45.000000,45,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45,45.000000,45,45.000000,45,45.000000,45
unique,NaN,NaN,2,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33,NaN,34,NaN,40,NaN,38
top,NaN,NaN,F,NaN,NaN,NaN,Hispanic/Latino,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7:48:00 AM,NaN,7:38,NaN,9:17,NaN,10:47
freq,NaN,NaN,29,NaN,NaN,NaN,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,3,NaN,2,NaN,3
mean,24.422222,48.111111,NaN,31.149765,182.502222,64.376667,NaN,6.122222,120.688889,14.166667,160.022222,191.644444,51.911111,139.733333,128.955556,35.777778,12.655556,NaN,121.244444,NaN,160.711111,NaN,120.488889,NaN
std,14.627945,12.703296,NaN,6.728244,35.990090,3.277742,NaN,0.908017,30.231548,8.263611,169.430797,46.711959,15.428460,47.230768,109.994412,57.609536,59.066771,NaN,29.339049,NaN,60.514695,NaN,46.660584,NaN
min,1.000000,18.000000,NaN,20.689962,116.800000,59.000000,NaN,4.600000,79.000000,2.500000,40.000000,91.000000,24.000000,38.000000,21.000000,8.000000,1.700000,NaN,80.000000,NaN,73.000000,NaN,67.000000,NaN
25%,12.000000,40.000000,NaN,26.922934,157.000000,62.000000,NaN,5.500000,100.000000,9.300000,83.000000,168.000000,42.000000,109.000000,90.000000,17.000000,3.000000,NaN,100.000000,NaN,111.000000,NaN,92.000000,NaN
50%,23.000000,51.000000,NaN,30.038349,180.000000,64.000000,NaN,5.900000,109.000000,13.500000,121.000000,187.000000,51.000000,137.000000,115.000000,24.000000,3.800000,NaN,117.000000,NaN,157.000000,NaN,101.000000,NaN
75%,36.000000,58.000000,NaN,35.918461,202.000000,67.000000,NaN,6.900000,142.000000,17.800000,154.000000,208.000000,60.000000,162.000000,133.000000,31.000000,4.800000,NaN,139.000000,NaN,193.000000,NaN,131.000000,NaN


No presenta ningun valor nulo en ninguna columna, cuenta con la información de los 45 pacientes o sujetos del estudio.

Hay algunas variables que no están en el formato adecuado, que son las variables relacionadas con tiempo: `collection_time_pdl_(lab)`, `time_(t)`, `time_(t)_1`, `time_(t)_2`.

In [443]:
cols_tiempo = ['collection_time_pdl_(lab)', 'time_(t)', 'time_(t)_1', 'time_(t)_2']

for col in cols_tiempo:
    bio[col] = pd.to_datetime(bio[col], errors='coerce')

In [444]:
bio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   subject                     45 non-null     int64         
 1   age                         45 non-null     int64         
 2   gender                      45 non-null     object        
 3   bmi                         45 non-null     float64       
 4   body_weight                 45 non-null     float64       
 5   height                      45 non-null     float64       
 6   self_identify               45 non-null     object        
 7   a1c_pdl_(lab)               45 non-null     float64       
 8   fasting_glu_pdl_(lab)       45 non-null     int64         
 9   insulin                     45 non-null     float64       
 10  triglycerides               45 non-null     int64         
 11  cholesterol                 45 non-null     int64         
 

Ahora pasamos a realizar el análisis de las variables numéricas del dataset bio:

In [445]:
# Función para graficar una variable numérica: histograma + boxplot + outliers
def graficar_numerica(df, columna):
    
    if columna not in df.columns:
        print(f"La columna '{columna}' no está en el DataFrame.")
        return

    if not pd.api.types.is_numeric_dtype(df[columna]):
        print(f"La columna '{columna}' no es numérica.")
        return

    serie = df[columna].dropna()

    # Cálculo de IQR y detección de outliers
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = serie[(serie < lower) | (serie > upper)]

    # Mostrar resumen estadístico
    print(f"\nAnálisis de la variable: '{columna}'")
    print(serie.describe())
    print(f"\nOutliers detectados: {len(outliers)}")
    print(f"Rango intercuartílico considerado: [{lower:.2f}, {upper:.2f}]")

    # Visualización: histograma + boxplot
    plt.figure(figsize=(14, 5))

    # Histograma con KDE
    plt.subplot(1, 2, 1)
    sns.histplot(serie, kde=True, bins=30, color='#4C72B0')
    plt.axvline(lower, color='red', linestyle='--', label='Límite inferior IQR')
    plt.axvline(upper, color='red', linestyle='--', label='Límite superior IQR')
    plt.title(f"Distribución de {columna}", fontsize=12)
    plt.xlabel(columna, fontsize=10)
    plt.ylabel("Frecuencia", fontsize=10)
    plt.legend()

    # Boxplot
    plt.subplot(1, 2, 2)
    sns.boxplot(x=serie, color='#4C72B0')
    plt.title(f"Boxplot de {columna}", fontsize=12)
    plt.xlabel(columna, fontsize=10)

    plt.tight_layout()
    plt.show()



### `age`

Esta variable representa la edad de los sujetos del estudio

In [446]:
graficar_numerica(bio, 'age')


Análisis de la variable: 'age'
count    45.000000
mean     48.111111
std      12.703296
min      18.000000
25%      40.000000
50%      51.000000
75%      58.000000
max      69.000000
Name: age, dtype: float64

Outliers detectados: 0
Rango intercuartílico considerado: [13.00, 85.00]


<Figure size 1400x500 with 2 Axes>

Los pacientes del estudio cuentan con una edad entre 18 y 69 años, con media de edad de 48 años, mediana de 51 años. Tienen una distribución normal con una leve desvación hacia la izquierda. 

### `bmi`

La variable bmi es para "body mass index" o índice de masa corporal, que es una medida que se usa para estimar si una persona tiene un peso saludable en relación con su altura. Usualmente está dada por el peso dividido por la altura.

Según la OMS: 

* < 18.5 → Bajo peso
* 18.5 – 24.9 → Normal
* 25 – 29.9 → Sobrepeso
* ≥ 30 → Obesidad (dividida en grados)



In [447]:
graficar_numerica(bio, 'bmi')


Análisis de la variable: 'bmi'
count    45.000000
mean     31.149765
std       6.728244
min      20.689962
25%      26.922934
50%      30.038349
75%      35.918461
max      49.088236
Name: bmi, dtype: float64

Outliers detectados: 0
Rango intercuartílico considerado: [13.43, 49.41]


<Figure size 1400x500 with 2 Axes>

Tiene una distribución normal, un leve sesgo positivo. Su máximo es 49, que representa una persona obesa, y el mínimo es 20. La media está en 31, y la mediana en 30, siendo la moda 27.

### `body_weigth`

Es el peso de los pacientes del estudio, está dada en libras.

In [448]:
graficar_numerica(bio, 'body_weight')


Análisis de la variable: 'body_weight'
count     45.000000
mean     182.502222
std       35.990090
min      116.800000
25%      157.000000
50%      180.000000
75%      202.000000
max      284.600000
Name: body_weight, dtype: float64

Outliers detectados: 1
Rango intercuartílico considerado: [89.50, 269.50]


<Figure size 1400x500 with 2 Axes>

Muestra una distribución moderadamente simétrica, centrada entre 150 y 225 lb. Solo se detectó un outlier, por encima del límite superior del IQR (269.5 lb), lo que indica que el peso corporal está mayormente dentro de un rango esperado. La dispersión general es amplia, pero sin valores extremos significativos que afecten gravemente la calidad de los datos.

Peso promedio de 182.5 lb y una mediana de 180 lb, lo que indica una distribución relativamente equilibrada. El rango intercuartílico va de 157 a 202 lb, con un mínimo de 116.8 lb y un máximo de 284.6 lb, este último coincidiendo con el outlier detectado previamente. La desviación estándar es de 36 lb, lo que refleja una variabilidad moderada en los datos.

### `height`

Es la altura de los participantes del estudio

In [449]:
graficar_numerica(bio, 'height')


Análisis de la variable: 'height'
count    45.000000
mean     64.376667
std       3.277742
min      59.000000
25%      62.000000
50%      64.000000
75%      67.000000
max      72.000000
Name: height, dtype: float64

Outliers detectados: 0
Rango intercuartílico considerado: [54.50, 74.50]


<Figure size 1400x500 with 2 Axes>

La variable height tiene 45 registros, con una media de 64.38 pulgadas y una mediana de 64 pulgadas, lo que sugiere una distribución centrada. El rango intercuartílico va de 62 a 67 pulgadas, con valores mínimos y máximos de 59 y 72 pulgadas, respectivamente. La desviación estándar es baja (3.28 pulgadas), indicando poca dispersión. No se detectaron outliers, y tanto el histograma como el boxplot confirman una distribución compacta y sin valores atípicos.

In [450]:
bio.columns

Index(['subject', 'age', 'gender', 'bmi', 'body_weight', 'height',
       'self_identify', 'a1c_pdl_(lab)', 'fasting_glu_pdl_(lab)', 'insulin',
       'triglycerides', 'cholesterol', 'hdl', 'non_hdl', 'ldl_(cal)',
       'vldl_(cal)', 'cho/hdl_ratio', 'collection_time_pdl_(lab)',
       '#1_contour_fingerstick_glu', 'time_(t)', '#2_contour_fingerstick_glu',
       'time_(t)_1', '#3_contour_fingerstick_glu', 'time_(t)_2'],
      dtype='object')

### `a1c_pdl_(lab)`
Indicador robusto de control glucémico (HbA1c o hemoglobina glucosilada). Mide el promedio de glucosa en sangre durante los últimos 2–3 meses. Se basa en la cantidad de glucosa adherida a la hemoglobina en los glóbulos rojos. 

Sus valores típicos:
| Valor (%)  | Interpretación                   |
| ---------- | -------------------------------- |
| < 5.7%     | Normal                           |
| 5.7 – 6.4% | Prediabetes                      |
| ≥ 6.5%     | Diabetes (diagnóstico clínico)   |
| < 7.0%     | Meta recomendada para diabéticos |



In [451]:
graficar_numerica(bio, 'a1c_pdl_(lab)')


Análisis de la variable: 'a1c_pdl_(lab)'
count    45.000000
mean      6.122222
std       0.908017
min       4.600000
25%       5.500000
50%       5.900000
75%       6.900000
max       8.500000
Name: a1c_pdl_(lab), dtype: float64

Outliers detectados: 0
Rango intercuartílico considerado: [3.40, 9.00]


<Figure size 1400x500 with 2 Axes>

Muestra una distribución ligeramente asimétrica hacia la derecha, sin presencia de outliers.
Su media es 6.12 %, la mediana: 5.90 %, mínimo de 4.6% y máximo de 8.5%.

La mayoría de los valores se sitúan en el rango normal-alto o prediabético (5.7–6.4 %), aunque algunos se acercan a niveles diagnósticos de diabetes (>6.5 %).

El valor máximo (8.5 %) sigue estando dentro de un rango clínicamente posible, por lo que no hay indicios de errores en los datos.

La dispersión es moderada, y los datos están bien contenidos dentro del rango fisiológico esperado.

### `fasting_glu_pdl_(lab)`

Representa la glucosa en sangre en ayunas, medida en laboratorio, y es un indicador clave del metabolismo de la glucosa.

Rangos de referencia (mg_dL):
| Glucosa en ayunas | Interpretación         |
| ----------------- | ---------------------- |
| < 100 mg/dL       | Normal                 |
| 100–125 mg/dL     | Prediabetes            |
| ≥ 126 mg/dL       | Diabetes (diagnóstico) |


In [452]:
graficar_numerica(bio, 'fasting_glu_pdl_(lab)')


Análisis de la variable: 'fasting_glu_pdl_(lab)'
count     45.000000
mean     120.688889
std       30.231548
min       79.000000
25%      100.000000
50%      109.000000
75%      142.000000
max      218.000000
Name: fasting_glu_pdl_(lab), dtype: float64

Outliers detectados: 1
Rango intercuartílico considerado: [37.00, 205.00]


<Figure size 1400x500 with 2 Axes>

Muestra una distribución sesgada a la derecha, con una media de 120.7 mg/dL y una mediana de 109 mg/dL, lo que sugiere que varios valores están por encima del umbral normal (100 mg/dL).

* Una proporción considerable de individuos podría estar en estado de prediabetes o diabetes, ya que más del 50% tienen niveles > 100 mg/dL.
* El valor máximo de 218 mg/dL es un outlier clínicamente plausible.
* La dispersión es alta, lo que indica variabilidad relevante para análisis de salud metabólica.

### `insulin`

Representa la concentración de insulina en sangre, una hormona clave para el metabolismo de la glucosa. Se suele medir en μU/mL (microunidades por mililitro).

Valores típicos en ayunas: 

| Rango (μU/mL) | Interpretación                         |
| ------------- | -------------------------------------- |
| 2 – 25        | Normal                                 |
| > 25          | Hiperinsulinemia (posible resistencia) |
| < 2           | Hiposecreción (poco común en sanos)    |


In [453]:
graficar_numerica(bio, 'insulin')


Análisis de la variable: 'insulin'
count    45.000000
mean     14.166667
std       8.263611
min       2.500000
25%       9.300000
50%      13.500000
75%      17.800000
max      46.400000
Name: insulin, dtype: float64

Outliers detectados: 1
Rango intercuartílico considerado: [-3.45, 30.55]


<Figure size 1400x500 with 2 Axes>

La variable insulin (niveles de insulina en sangre, en ayunas y en μU/mL) muestra una distribución asimétrica hacia la derecha, con un valor atípico identificado.

* La mayoría de los valores están dentro del rango normal (2–25 μU/mL).
* El valor de 46.4 μU/mL se identifica como un outlier, lo que podría indicar un caso de hiperinsulinemia o una medición fuera de ayuno.
* La forma de la distribución y el boxplot confirman que este outlier está influyendo ligeramente en la media.

### `triglycerides` 

Representa los niveles de triglicéridos en sangre, que son un tipo de grasa (lípido) que el cuerpo almacena y usa como fuente de energía. Es un biomarcador clave en el análisis de salud metabólica y cardiovascular.

Rangos de referencia en ayunas:
| Nivel (mg/dL) | Interpretación           |
| ------------- | ------------------------ |
| < 150         | Normal                   |
| 150 – 199     | Límite alto              |
| 200 – 499     | Alto                     |
| ≥ 500         | Muy alto (riesgo severo) |


In [454]:
graficar_numerica(bio, 'triglycerides')


Análisis de la variable: 'triglycerides'
count      45.000000
mean      160.022222
std       169.430797
min        40.000000
25%        83.000000
50%       121.000000
75%       154.000000
max      1150.000000
Name: triglycerides, dtype: float64

Outliers detectados: 4
Rango intercuartílico considerado: [-23.50, 260.50]


<Figure size 1400x500 with 2 Axes>

Presenta una distribución fuertemente sesgada a la derecha, con varios valores elevados que actúan como outliers clínicamente relevantes.

* La mayoría de los valores están en el rango normal o ligeramente elevado.
* Hay 4 valores que superan los 260 mg/dL, incluyendo un caso extremo de 1150 mg/dL, que probablemente representa una hipertrigliceridemia severa o un error de registro.
* El sesgo a la derecha sugiere que aplicar transformaciones (como log) o capping puede ser útil.

In [455]:
# Cálculo de los límites IQR
Q1 = bio['triglycerides'].quantile(0.25)
Q3 = bio['triglycerides'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

# Filtrar registros outliers (por encima del límite superior)
outliers_triglycerides = bio[bio['triglycerides'] > upper]

# Mostrar resultados
print(f"Número de outliers por encima de {upper:.2f}: {outliers_triglycerides.shape[0]}")
display(outliers_triglycerides)


Número de outliers por encima de 260.50: 4


,subject,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2
3,4,33,F,42.384279,262.6,66.0,Hispanic/Latino,5.5,105,19.4,300,267,46,221,164,60,5.8,2025-06-01 07:20:00,109,2025-06-01 07:37:00,110,2025-06-01 09:04:00,90,2025-06-01 10:01:00
4,5,51,F,30.957534,172.0,62.5,Hispanic/Latino,6.6,144,12.9,392,269,38,231,157,78,7.1,2025-06-01 07:45:00,139,2025-06-01 08:59:00,215,2025-06-01 10:52:00,130,2025-06-01 11:54:00
8,9,34,F,37.001506,183.2,59.0,Hispanic/Latino,5.7,122,25.7,312,202,40,162,108,62,5.1,2025-06-01 07:32:00,119,2025-06-01 07:46:00,128,2025-06-01 09:32:00,96,2025-06-01 10:29:00
11,12,52,M,30.272873,205.0,69.0,Hispanic/Latino,7.1,179,13.5,1150,213,24,189,800,400,400.0,2025-06-01 07:17:00,186,2025-06-01 07:38:00,279,2025-06-01 09:17:00,215,2025-06-01 10:17:00


Según lo observado en los registros outliers, en los primeros tres casos, colesterol total, LDL, HDL y BMI son también elevados, lo cual da soporte a que los valores puedan ser reales.

El cuarto sujeto (con 1150) muestra:
* LDL de 800, probablemente imposible fisiológicamente.
* cho/hdl_ratio de 400, que claramente es un error de entrada (un valor normal va de 3 a 6 aprox).

Por lo tanto la limpieza de outliers se realizará como sigue:
* Capping suave: valores por encima del umbral se recortan al límite permitido.

In [456]:
bio['triglycerides'] = bio['triglycerides'].apply(lambda x: min(x, upper if x > 600 else x))


In [457]:
graficar_numerica(bio, 'triglycerides')


Análisis de la variable: 'triglycerides'
count     45.000000
mean     140.255556
std       79.136865
min       40.000000
25%       83.000000
50%      121.000000
75%      154.000000
max      392.000000
Name: triglycerides, dtype: float64

Outliers detectados: 3
Rango intercuartílico considerado: [-23.50, 260.50]


<Figure size 1400x500 with 2 Axes>

### `cholesterol`

Mide los niveles de colesterol total en sangre en ayunas, un biomarcador clave para evaluar el riesgo cardiovascular y el estado metabólico de una persona.

Rangos de referencia (mg/dL)
| Nivel   | Interpretación        |
| ------- | --------------------- |
| < 200   | Deseable (normal)     |
| 200–239 | Límite alto           |
| ≥ 240   | Alto (riesgo elevado) |


In [458]:
graficar_numerica(bio, 'cholesterol')


Análisis de la variable: 'cholesterol'
count     45.000000
mean     191.644444
std       46.711959
min       91.000000
25%      168.000000
50%      187.000000
75%      208.000000
max      345.000000
Name: cholesterol, dtype: float64

Outliers detectados: 4
Rango intercuartílico considerado: [108.00, 268.00]


<Figure size 1400x500 with 2 Axes>

Muestra una distribución ligeramente sesgada a la derecha, con valores concentrados alrededor del rango saludable pero con algunos casos extremos.
La mayoría de los individuos están en el rango normal o límite alto (entre 150–240 mg/dL).El valor máximo de 345 mg/dL indica colesterol muy elevado, posiblemente reflejando riesgo cardiovascular alto o un caso clínico. La dispersión es moderada, con unos pocos valores fuera del rango esperado.

Los outliers que se muestran con fisiológicamente posibles.

### `hdl`

Representa los niveles de colesterol HDL (del inglés High-Density Lipoprotein), conocido como el “colesterol bueno”. Es una lipoproteína que transporta el exceso de colesterol desde los tejidos y arterias hacia el hígado para su eliminación.

Rangos de referencia
| Nivel de HDL (mg/dL) | Interpretación                              |
| -------------------- | ------------------------------------------- |
| < 40 (hombres)       | Riesgo elevado de enfermedad cardiovascular |
| < 50 (mujeres)       | Riesgo elevado                              |
| ≥ 60                 | Protector (ideal)                           |


In [459]:
graficar_numerica(bio, 'hdl')


Análisis de la variable: 'hdl'
count     45.000000
mean      51.911111
std       15.428460
min       24.000000
25%       42.000000
50%       51.000000
75%       60.000000
max      106.000000
Name: hdl, dtype: float64

Outliers detectados: 2
Rango intercuartílico considerado: [15.00, 87.00]


<Figure size 1400x500 with 2 Axes>

mMestra una distribución ligeramente simétrica con algunos valores elevados considerados outliers. La mayoría de los valores están en el rango normal o protector (≥ 40 mg/dL en hombres, ≥ 50 mg/dL en mujeres).
Los valores outlier por arriba de 87 mg/dL, aunque poco comunes, son fisiológicamente posibles y suelen asociarse con menor riesgo cardiovascular.
La distribución es bastante equilibrada y no presenta problemas severos de dispersión.


### `non_hdl`

Representa el colesterol no-HDL, es decir, todo el colesterol “malo” o aterogénico, que puede contribuir a la formación de placas en las arterias.

| Nivel de non-HDL (mg/dL) | Interpretación |
| ------------------------ | -------------- |
| < 130                    | Deseable       |
| 130 – 159                | Límite alto    |
| 160 – 189                | Alto           |
| ≥ 190                    | Muy alto       |


In [460]:
graficar_numerica(bio, 'non_hdl')


Análisis de la variable: 'non_hdl'
count     45.000000
mean     139.733333
std       47.230768
min       38.000000
25%      109.000000
50%      137.000000
75%      162.000000
max      283.000000
Name: non_hdl, dtype: float64

Outliers detectados: 2
Rango intercuartílico considerado: [29.50, 241.50]


<Figure size 1400x500 with 2 Axes>

Presenta una distribución ligeramente sesgada a la derecha, con dos outliers elevados.

La mayoría de los valores se encuentran entre 100 y 160 mg/dL, lo que indica un perfil intermedio a ligeramente elevado, común en poblaciones con riesgo metabólico moderado.
Los outliers reflejan casos con colesterol no-HDL muy alto (≥ 240 mg/dL), lo cual incrementa el riesgo cardiovascular.
La forma de la distribución sugiere que los datos son clínicamente válidos, aunque con variabilidad importante.

### `cho/hdl_ratio`

Representa el índice colesterol total / HDL, también conocido como índice aterogénico. Es un indicador clave del riesgo cardiovascular. 
Compara el colesterol total con el colesterol HDL (el “bueno”) para evaluar el balance entre colesterol dañino y protector.

Rangos de referencia:
| Valor del ratio | Interpretación                        |
| --------------- | ------------------------------------- |
| < 3.5           | Bajo riesgo cardiovascular (ideal)    |
| 3.5 – 5.0       | Riesgo moderado                       |
| > 5.0           | Riesgo elevado de enfermedad cardíaca |


In [461]:
graficar_numerica(bio, 'cho/hdl_ratio')


Análisis de la variable: 'cho/hdl_ratio'
count     45.000000
mean      12.655556
std       59.066771
min        1.700000
25%        3.000000
50%        3.800000
75%        4.800000
max      400.000000
Name: cho/hdl_ratio, dtype: float64

Outliers detectados: 1
Rango intercuartílico considerado: [0.30, 7.50]


<Figure size 1400x500 with 2 Axes>

La mayoría de los individuos tienen un índice en el rango 3–5, lo cual es clínicamente aceptable o ligeramente elevado.

El valor de 400 es claramente un error o caso anómalo, ya que no es fisiológicamente plausible.
La media está completamente distorsionada por ese único valor atípico.

In [462]:
# Calcular IQR
Q1 = bio['cho/hdl_ratio'].quantile(0.25)
Q3 = bio['cho/hdl_ratio'].quantile(0.75)
IQR = Q3 - Q1

# Límites para el capping
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Aplicar capping
bio['cho/hdl_ratio'] = bio['cho/hdl_ratio'].apply(lambda x: upper_bound if x > upper_bound else (lower_bound if x < lower_bound else x))


In [463]:
graficar_numerica(bio, 'cho/hdl_ratio')


Análisis de la variable: 'cho/hdl_ratio'
count    45.000000
mean      3.933333
std       1.333655
min       1.700000
25%       3.000000
50%       3.800000
75%       4.800000
max       7.500000
Name: cho/hdl_ratio, dtype: float64

Outliers detectados: 0
Rango intercuartílico considerado: [0.30, 7.50]


<Figure size 1400x500 with 2 Axes>

Con el trabajo realizado, los datos son simétricos, sin distorsión por outliers, y el boxplot refleja una distribución limpia y contenida.
La mediana es muy cercana a la media, lo que indica consistencia en los valores.

En cuanto a las demás columnas del dataset "bio", no se analizan porque son lecturas puntuales que de hecho pueden aportar ruido en modelos.

## Conjunto de datos: `gut_scores_imputed`

In [464]:
gut_scores_imputed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 23 columns):
 #   Column                                     Non-Null Count  Dtype
---  ------                                     --------------  -----
 0   gut_lining_health                          45 non-null     int32
 1   lps_biosynthesis_pathways                  45 non-null     int32
 2   biofilm_chemotaxis_and_virulence_pathways  45 non-null     int32
 3   tma_production_pathways                    45 non-null     int32
 4   ammonia_production_pathways                45 non-null     int32
 5   metabolic_fitness                          45 non-null     int32
 6   active_microbial_diversity                 45 non-null     int32
 7   butyrate_production_pathways               45 non-null     int32
 8   flagellar_assembly_pathways                45 non-null     int32
 9   putrescine_production_pathways             45 non-null     int32
 10  uric_acid_production_pathways              45 non-nu

In [465]:
# Visualización de la distribución de variables de gut scores
gut_data = gut_scores_imputed.drop(columns=["subject"])

n_cols = 4
n_rows = int(np.ceil(len(gut_data.columns) / n_cols))
fig, axs = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))

for i, col in enumerate(gut_data.columns):
    ax = axs[i // n_cols, i % n_cols]
    sns.countplot(x=gut_data[col], ax=ax, color='#4C72B0')
    ax.set_title(col.replace("_", " ").title(), fontsize=11)
    ax.set_xlabel("Puntuación (1 = No óptima, 3 = Buena)", fontsize=9)
    ax.set_ylabel("Frecuencia", fontsize=9)
    ax.tick_params(axis='x', labelrotation=0)

# Ajustes generales
plt.tight_layout()
plt.suptitle("Distribuciones de Gut Scores (1 = No óptima, 2 = Promedio, 3 = Buena)", 
             y=1.02, fontsize=16)
plt.show()

<Figure size 2000x3000 with 24 Axes>

El dataset contiene 22 variables ordinales (valores de 1 a 3 que representan estados "Not Optimal", "Average" y "Good"), revela que la mayoría de las puntuaciones se concentran en el nivel intermedio (2), indicando que la salud intestinal de los sujetos tiende a ser promedio. Sin embargo, en varias variables relevantes como `Butyrate Production`, `Metabolic Fitness` y `Digestive Efficiency`, las puntuaciones "Good" son escasas, lo cual sugiere que pocos participantes presentan un perfil microbiómico altamente favorable. A la vez, hay otras variables como `Biofilm Chemotaxis` y `Virulence Pathways`, `TMA Production`, `Gas Production` y `Sulfide Gas Production` donde predominan puntuaciones bajas (1), lo que podría reflejar una actividad microbiana potencialmente perjudicial en varios sujetos.

Estas observaciones permiten identificar posibles rutas microbianas candidatas para correlacionar con métricas de respuesta glucémica como el área incremental bajo la curva (iAUC) o el cambio máximo postprandial. Variables como `Metabolic Fitness`, `Gut Microbiome Health`, `Inflammatory Activity` o `Butyrate Production` son especialmente relevantes debido a su vínculo conocido con el metabolismo de la glucosa, la resistencia a la insulina y la inflamación sistémica.

In [466]:
# Matriz de correlación entre variables de gut scores
plt.figure(figsize=(18, 14))
sns.heatmap(
    gut_data.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title("Matriz de correlación entre Gut Scores", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


<Figure size 1800x1400 with 2 Axes>

La matriz de correlación entre los gut scores muestra asociaciones relevantes entre diversas funciones microbianas intestinales. 

Se observa una fuerte correlación positiva entre variables funcionales clave como `digestive_efficiency`, `protein_fermentation`, `metabolic_fitness`, `gut_microbiome_health` e `inflammatory_activity`, lo que sugiere que estas dimensiones del microbioma suelen mejorar o deteriorarse en conjunto. Por ejemplo, `digestive_efficiency` presenta correlaciones ≥0.70 con `gut_microbiome_health` (0.74), `metabolic_fitness` (0.76) y `protein_fermentation` (0.78), consolidando su rol central en la salud gastrointestinal general. También destaca la fuerte correlación de `microbiome_induced_stress` con `putrescine_production_pathways` (0.69) y `inflammatory_activity` (0.60), lo cual podría reflejar una asociación entre estrés microbiano, inflamación y fermentación proteica. 

En contraste, variables como `lps_biosynthesis_pathways`, `tma_production_pathways` y `sulfide_gas_production_pathways` tienden a correlacionarse negativamente con funciones protectoras del microbioma, lo que refuerza su vínculo con estados menos saludables.

## Conclusión del análisis exploratorio de datos hecho a los conjuntos de datos

Se realizó el EDA para los conjuntos de datos:
* meals
* bio
* gut_scores_imputed

Se realizó la limpieza de outliers correspondiente y se analizaron las variables.

# **7. Reemplazo de los datos en el conjunto de datos original** <a class="anchor" id="7"></a>

[Tabla de Contenidos](#0.1)

Ahora que se tienen conjuntos de datos limpios, es decir, sin valores nulos, sin outliers incoherentes, y analizados, se procede a reemplazar los registros originales del dataset df (que es el dataset original de las mediciones de glucosa y los eventos de comida) por los registros que se preprocesaron en el dataset meals.


In [467]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687580 entries, 0 to 687579
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   timestamp            687580 non-null  object 
 1   libre_gl             687360 non-null  float64
 2   dexcom_gl            629825 non-null  float64
 3   hr                   610256 non-null  float64
 4   calories_(activity)  652134 non-null  float64
 5   mets                 501078 non-null  float64
 6   meal_type            1706 non-null    object 
 7   calories             1706 non-null    float64
 8   carbs                1706 non-null    float64
 9   protein              1706 non-null    float64
 10  fat                  1706 non-null    float64
 11  fiber                1705 non-null    float64
 12  amount_consumed      1642 non-null    float64
 13  image_path           3197 non-null    object 
 14  patient              687580 non-null  int64  
dtypes: float64(11), i

In [468]:
meals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1706 entries, 233 to 686614
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp            1706 non-null   datetime64[ns]
 1   libre_gl             1705 non-null   float64       
 2   dexcom_gl            1684 non-null   float64       
 3   hr                   1625 non-null   float64       
 4   calories_(activity)  1684 non-null   float64       
 5   mets                 1320 non-null   float64       
 6   meal_type            1706 non-null   object        
 7   calories             1706 non-null   float64       
 8   carbs                1706 non-null   float64       
 9   protein              1706 non-null   float64       
 10  fat                  1706 non-null   float64       
 11  fiber                1706 non-null   float64       
 12  amount_consumed      1706 non-null   float64       
 13  image_path           1644 non-null

Para poder tener un identificador claro del evento de comida, se toma la columna `patient` y la columna `timestamp`, pero en el dataset df esta no está como variable de tiempo:

In [469]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [470]:
# Definir la clave compuesta
meals_key = set(zip(meals['patient'], meals['timestamp']))

# Filtrar df para eliminar las filas que serán reemplazadas
df_filtered = df[~df[['patient', 'timestamp']].apply(tuple, axis=1).isin(meals_key)]

# Añadir meals (preprocesado) al dataframe original
df_updated = pd.concat([df_filtered, meals], ignore_index=True)

# Ordenar para mantener coherencia temporal
df_updated = df_updated.sort_values(by=['patient', 'timestamp']).reset_index(drop=True)

In [471]:
df_updated['meal_type'].value_counts()

meal_type
dinner       492
breakfast    436
lunch        435
snack        343
Name: count, dtype: int64

Ahora se tiene el dataset original, con todas las mediciones de glucosa, a partir del cual se pueden obtener las curvas de glucosa por la respuesta glucémica postpandrial, con los eventos de comida correctamente preprocesados.

# **8. Respuesta Glucémica Postprandial** <a class="anchor" id="8"></a>


[Tabla de Contenidos](#0.1)

En este punto del análisis se tienen cuatro conuntos de datos:
- `df_updated`: Tiene la información de las mediciones de los sensores de glucosa y los eventos de comida.
- `bio`: Contiene información de la salud general de los sujetos del estudio.
- `mcirobes_pca`: Contiene tres componentes principales del microbioma intestinal de los sujetos del estudio.
- `gut_scores_imputed`: Son calificaciones ordinales de la salud general del paciente, relacionada con su salud intestinal.

Ahora se realiza el análisis de la respuesta glucémica postpandrial, es decir, el cambio en los níveles de glucosa que sucede en cada sujeto tras cada evento de comida. 

Para esto:
1. Se toman los registros en los que `meal_type` no es nulo, que representa cuando cada paciente registró el inicio de una comida, y este punto se toma así mismo, como el inicio de una comida o alimentación.
2. Se toma el intervalo de tiempo de dos horas posteriores al inicio de la comida para generar la curva de glucosa post-comida o "Respuesta glucémica postpandrial", en inglés conocida como "Postpandrial Glucemic Response" (PPGR).
3. Se grafican las curvas de cada una de las comidas realizadas y registradas por cada paciente.
4. También se obtiene información relevante a cada curva de respuesta glucémica obtenida.
5. Se crea un dataset que obtiene toda esta información, que será el principal dataset del estudio.


In [472]:
# Por motivos de practicidad, se actuaiza la variable df para referirse al DataFrame actualizado (df_updated)
df = df_updated

In [473]:
# Filtrar el DataFrame para solo los puntos donde existe meal_type, que es el inicio de cada comida
meal_start = df[df['meal_type'].notna()]
meal_start

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
233,2020-05-01 14:23:00,69.800000,109.4,95.0,4.61296,44.0,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,photos/00000005-PHOTO-2020-5-1-14-23-0.jpg,1
618,2020-05-01 20:48:00,84.800000,114.8,81.0,1.36292,13.0,dinner,80.0,18.0,0.0,0.0,0.0,100.0,photos/00000007-PHOTO-2020-5-1-20-48-0.jpg,1
825,2020-05-02 00:15:00,81.000000,97.4,78.0,4.40328,42.0,snack,110.0,24.0,0.0,2.0,0.0,100.0,NaN,1
1308,2020-05-02 08:18:00,88.400000,101.8,81.0,3.14520,30.0,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000010-PHOTO-2020-5-2-8-18-0.jpg,1
1530,2020-05-02 12:00:00,82.000000,80.0,84.0,2.93552,28.0,lunch,840.0,89.0,17.0,42.0,3.0,100.0,photos/00000012-PHOTO-2020-5-2-12-0-0.jpg,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684069,2025-05-19 13:37:00,82.933333,94.8,93.0,1.02443,11.0,lunch,445.0,43.0,20.0,20.0,13.0,100.0,photos/00000082-PHOTO-2025-5-19-13-37-0.jpg,49
684473,2025-05-19 20:21:00,88.466667,106.0,78.0,0.93130,10.0,dinner,370.0,38.0,26.0,12.0,2.0,100.0,photos/00000084-PHOTO-2025-5-19-20-21-0.jpg,49
685118,2025-05-20 07:06:00,138.333333,152.6,81.0,2.23512,24.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000085-PHOTO-2025-5-20-7-6-0.jpg,49
685465,2025-05-20 12:53:00,84.000000,100.0,84.0,1.02443,11.0,lunch,725.0,94.0,44.0,20.0,4.0,100.0,photos/00000098-PHOTO-2025-5-20-12-53-0.jpg,49


Se puede observar que existen 1706 eventos de comida.

Para entender el conjunto de datos completo se debe tomar en cuenta lo sigueinte:
Este conjunto de datos pertenece a un estudio en el cual a cada paciente se le monitorizo por 10 días completos, teniendo al menos tres comidas cada día.

Entonces, es posible crear una gráfica que permita visualizar la glucosa en el paciente a lo largo de los 10 días del estudio:

In [474]:
# Evolución de la glucosa para un paciente específico
patient_id = 1
df_patient = df[df['patient'] == patient_id]

# Configuración del gráfico
plt.figure(figsize=(20, 6))
plt.plot(df_patient['timestamp'], df_patient['libre_gl'], label='Libre GL', color='#C44E52')   # rojo
plt.plot(df_patient['timestamp'], df_patient['dexcom_gl'], label='Dexcom GL', color='#55A868') # verde

# Personalización visual
plt.title(f'Evolución de glucosa en el tiempo - Paciente {patient_id:02d}', fontsize=14)
plt.xlabel('Tiempo', fontsize=12)
plt.ylabel('Glucosa (mg/dL)', fontsize=12)
plt.legend(title='Sensor', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


<Figure size 2000x600 with 1 Axes>

También es posible ver un día en específico:

In [475]:
# Zoom en un día específico para un paciente
zoom_day = "2020-05-05"

# Filtrar datos del paciente para el día seleccionado
df_patient_day = df_patient[df_patient['timestamp'].dt.date == pd.to_datetime(zoom_day).date()]

# Configuración del gráfico
plt.figure(figsize=(15, 6))
plt.plot(df_patient_day['timestamp'], df_patient_day['libre_gl'], label='Libre GL', color='#C44E52', alpha=0.7)
plt.plot(df_patient_day['timestamp'], df_patient_day['dexcom_gl'], label='Dexcom GL', color='#55A868', alpha=0.7)

# Personalización visual
plt.title(f'Evolución de glucosa - Paciente {patient_id:02d} - Día {zoom_day}', fontsize=14)
plt.xlabel('Hora', fontsize=12)
plt.ylabel('Glucosa (mg/dL)', fontsize=12)
plt.legend(title='Sensor', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


<Figure size 1500x600 with 1 Axes>

Y se pueden agregar marcadores que permiten identificar cuándo sucede el inicio de un evento de comida, de acuerdo con el indicador establecido anteriormente (`meal_type` no es nulo).

In [476]:
# Filtrar el DataFrame para ese día
df_patient_day = df_patient[df_patient['timestamp'].dt.date == pd.to_datetime(zoom_day).date()]

# Configuración del gráfico
plt.figure(figsize=(15, 6))

# Líneas de glucosa
plt.plot(df_patient_day['timestamp'], df_patient_day['libre_gl'], label='Libre GL', color='#C44E52', alpha=0.8)
plt.plot(df_patient_day['timestamp'], df_patient_day['dexcom_gl'], label='Dexcom GL', color='#55A868', alpha=0.8)

# Marcadores para eventos de comida
mask_meal = df_patient_day['meal_type'].notna()
plt.scatter(
    df_patient_day['timestamp'][mask_meal],
    df_patient_day['libre_gl'][mask_meal],
    color='#4C72B0', label='Comida (Libre GL)', marker='o', s=60, edgecolor='white'
)
plt.scatter(
    df_patient_day['timestamp'][mask_meal],
    df_patient_day['dexcom_gl'][mask_meal],
    color='#4C72B0', label='Comida (Dexcom GL)', marker='x', s=60
)

# Personalización
plt.title(f'Evolución de glucosa - Paciente {patient_id:02d} - Día {zoom_day}', fontsize=14)
plt.xlabel('Hora', fontsize=12)
plt.ylabel('Glucosa (mg/dL)', fontsize=12)
plt.legend(title='Lecturas', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


<Figure size 1500x600 with 1 Axes>

Las curvas de ambos sensores presentan fluctuaciones que reflejan el ritmo glucémico natural, influenciado por comidas, actividad y metabolismo basal.

Se observan varios picos de glucosa tras las comidas, especialmente notables alrededor de las 09:00, 13:00 y 20:00 horas.

En cuanto a reacción postpandrial:
- Justo después de algunos marcadores de comida, la glucosa se eleva claramente (por ejemplo, en la tarde y noche), lo que indica una respuesta glucémica esperada.
- El pico más alto se observa cerca de las 18:00, coincidiendo con una subida rápida tanto en Dexcom como en Libre, aunque más pronunciada en Dexcom.

También se observa que el sensor Dexcom GL tiende a reportar valores ligeramente más altos que Libre GL durante la mayor parte del día.
Ambas series son coherentes en forma, aunque con desplazamientos pequeños en magnitud. En general, la tendencia es similar, lo cual es positivo en cuanto a consistencia entre dispositivos.

Durante las primeras horas del día (00:00–06:00), los niveles se mantienen relativamente estables con un descenso progresivo.
También se observan caídas marcadas después de los picos postcomida, especialmente entre 19:00 y 21:00, lo cual puede reflejar una respuesta insulinémica adecuada.

**Análisis por evento de comida**

También es posible analizar cada evento de comida específico, a continuación se grafican algunos eventos de comida individuales para poder observar más detenidamente su respuesta glucémica postpandrial.

Para esto vamos a limitar el tiempo de análisis de cada evento de la siguiente manera:
* La respuesta glucémica postpandrial se suele medir dos horas después del inicio de la comida, porque es el tiempo de respuesta fisiológico.

In [477]:
# Definición de ventanas pre y postprandial
interval_pre = pd.Timedelta(hours=1)
interval_post = pd.Timedelta(hours=2)
min_delay = pd.Timedelta(minutes=15)  # Se ignoran picos muy tempranos

# Configuración de límite de gráficos
max_plots = 50
plot_count = 0

# Iterar sobre eventos de comida
for index, row in meals.iterrows():
    if plot_count >= max_plots:
        print(f"Se alcanzó el máximo de {max_plots} gráficos.")
        break

    meal_time = row['timestamp']
    start_time = meal_time - interval_pre
    end_time = meal_time + interval_post

    # Filtrar datos por ventana y paciente
    mask_window = (
        (df['timestamp'] >= start_time) &
        (df['timestamp'] <= end_time) &
        (df['patient'] == row['patient'])
    )
    sub_data = df[mask_window]

    # Solo buscar picos a partir de 15 minutos después de la comida
    delayed_data = sub_data[sub_data['timestamp'] >= (meal_time + min_delay)]

    # Detectar picos
    peak_libre, peak_time_libre = None, None
    peak_dexcom, peak_time_dexcom = None, None

    if not delayed_data['libre_gl'].isnull().all():
        peak_libre = delayed_data['libre_gl'].max()
        peak_time_libre = delayed_data[delayed_data['libre_gl'] == peak_libre]['timestamp'].iloc[0]

    if not delayed_data['dexcom_gl'].isnull().all():
        peak_dexcom = delayed_data['dexcom_gl'].max()
        peak_time_dexcom = delayed_data[delayed_data['dexcom_gl'] == peak_dexcom]['timestamp'].iloc[0]

    # Si hay al menos un pico, graficar
    if peak_libre is not None or peak_dexcom is not None:
        plt.figure(figsize=(12, 5))

        # Libre GL
        if peak_libre is not None:
            plt.plot(sub_data['timestamp'], sub_data['libre_gl'], label='Libre GL', color='#C44E52')
            plt.scatter(peak_time_libre, peak_libre, color='darkred', label='Pico (Libre GL)', zorder=5)
            plt.scatter(meal_time, row['libre_gl'], color='purple', marker='D', label='Inicio Meal (Libre GL)', zorder=5)

        # Dexcom GL
        if peak_dexcom is not None:
            plt.plot(sub_data['timestamp'], sub_data['dexcom_gl'], label='Dexcom GL', color='#55A868')
            plt.scatter(peak_time_dexcom, peak_dexcom, color='darkorange', label='Pico (Dexcom GL)', zorder=5)
            plt.scatter(meal_time, row['dexcom_gl'], color='cyan', marker='D', label='Inicio Meal (Dexcom GL)', zorder=5)

        # Personalización
        plt.title(
            f"Curva de glucosa pre y postprandial\nPaciente {int(row['patient']):02d} – {meal_time.strftime('%Y-%m-%d %H:%M')}",
            fontsize=13
        )
        plt.xlabel('Tiempo', fontsize=11)
        plt.ylabel('Glucosa (mg/dL)', fontsize=11)
        plt.legend(fontsize=9)
        plt.grid(True, linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()

        plot_count += 1


<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

...

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

Se alcanzó el máximo de 50 gráficos.


Hay picos postprandiales evidentes:
* En la mayoría de los eventos (especialmente el del 01/05 a las 20:48 y el del 02/05 a las 16:37), ambas curvas muestran un aumento claro de glucosa, con un pico glucémico entre 30 y 60 minutos después de la comida, como es fisiológicamente esperado.

La magnitud del pico varía:
* El sensor Dexcom tiende a registrar picos más altos que Libre.
Ejemplo claro: el evento del 01/05 20:48 muestra un pico de ~155 mg/dL en Dexcom vs. ~113 mg/dL en Libre.

Esto sugiere una mayor sensibilidad del sensor Dexcom o diferencias en calibración.

Las curvas fisiológicas son coherentes:
* Las curvas tienen forma convexa (ascenso y posterior descenso o estabilización), lo cual refleja una respuesta insulínica funcional.
* No se observan hipoglucemias (<70 mg/dL) ni patrones anómalos.

Se observa consistencia entre sensores:
* Aunque hay diferencias en los valores absolutos, las tendencias (forma de la curva y momento del pico) son coherentes entre sensores, lo cual valida la calidad del registro.

Se observan también eventos con respuesta leve o retardada:
* En el evento del 03/05 a las 14:51, la respuesta es más moderada, con picos por debajo de 105 mg/dL. Esto podría corresponder a una comida baja en carbohidratos o bien a un retardo glucémico por composición mixta (proteínas/grasas).

# **9. Filtrar los eventos de comida** <a class="anchor" id="9"></a>

[Tabla de Contenidos](#0.1)

Para garantizar la calidad de los datos para el entrenamiento del modelo se establecen reglas para la obtención de la curva de respuesta glucémica que cada evento de comida debe cumplir para poder ser un dato válido para el modelo:
* No debe ocurrir un evento de comida 1 hora antes del evento de comida que se analiza: El cruce de dos eventos de comida afecta directamente a la respuesta glucémica.
* No debe ocurrir un evento de comida 2 horas después del evento de comida que se analiza.

Además, para evitar errores de registro en el tiempo del inicio del evento de comida se añade el siguiente procedimiento:
* Revisar si el punto marcado como inicio de comida coincide con un máximo local de glucosa. Si es así, eso sugiere que el paciente registró el evento después de haber comido: Evaluar si el valor de glucosa en ese punto es el máximo entre -60 y +120 minutos (usando las curvas individuales).
* Cuando esto suceda, ajustar meal_time (inicio del evento de comida) hacia atrás: Si lo es, considerar que el registro fue hecho tarde, y cambiar el meal_time a 60 minutos antes (o buscar el último mínimo local antes del pico).

Definimos los lapsos de tiempo de respuesta glucémica

In [478]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687580 entries, 0 to 687579
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   timestamp            687580 non-null  datetime64[ns]
 1   libre_gl             687360 non-null  float64       
 2   dexcom_gl            629825 non-null  float64       
 3   hr                   610256 non-null  float64       
 4   calories_(activity)  652134 non-null  float64       
 5   mets                 501078 non-null  float64       
 6   meal_type            1706 non-null    object        
 7   calories             1706 non-null    float64       
 8   carbs                1706 non-null    float64       
 9   protein              1706 non-null    float64       
 10  fat                  1706 non-null    float64       
 11  fiber                1706 non-null    float64       
 12  amount_consumed      1706 non-null    float64       
 13  image_path    

In [479]:
meals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1706 entries, 233 to 686614
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp            1706 non-null   datetime64[ns]
 1   libre_gl             1705 non-null   float64       
 2   dexcom_gl            1684 non-null   float64       
 3   hr                   1625 non-null   float64       
 4   calories_(activity)  1684 non-null   float64       
 5   mets                 1320 non-null   float64       
 6   meal_type            1706 non-null   object        
 7   calories             1706 non-null   float64       
 8   carbs                1706 non-null   float64       
 9   protein              1706 non-null   float64       
 10  fat                  1706 non-null   float64       
 11  fiber                1706 non-null   float64       
 12  amount_consumed      1706 non-null   float64       
 13  image_path           1644 non-null

In [480]:
# Parámetros de tiempo
interval_post = pd.Timedelta(hours=2) # Tiempo en el que se mide la respuesta postpandrial
interval_min_gap_before = pd.Timedelta(hours=1) # Tiempo previo al inicio de la comida, no debe haber comidas en este intervalo
interval_min_gap_after = pd.Timedelta(hours=2) # Tiempo después de la comida, no debe haber comidas en este intervalo

Función para encontrar el mínimo local anterior al inicio del evento de comida mal registrado (registrado en un máximo local)

In [481]:
# Función para encontrar el mínimo local antes de un pico de glucosa
def find_local_minimum_before_peak(glucose_data, peak_idx, max_lookback_minutes=60):
    """
    Encuentra el mínimo local antes de un pico de glucosa, revisando hacia atrás 
    hasta un máximo de `max_lookback_minutes` registros.

    Parámetros:
    - glucose_data: DataFrame con una columna 'glucose'
    - peak_idx: Índice (int) del pico dentro del DataFrame
    - max_lookback_minutes: Número máximo de registros hacia atrás para buscar (por defecto: 60)

    Retorna:
    - Índice entero del mínimo local antes del pico, o del valor mínimo si no se encuentra uno local
    """
    # Validaciones básicas
    if glucose_data.empty or peak_idx >= len(glucose_data):
        return None
    if peak_idx == 0:
        return 0

    # Definir el rango de búsqueda hacia atrás
    start_idx = max(0, peak_idx - max_lookback_minutes)
    segment = glucose_data.iloc[start_idx:peak_idx + 1]

    # Invertir la señal para detectar mínimos como si fueran picos
    inverted_signal = -segment['glucose'].values
    peaks, _ = find_peaks(inverted_signal, distance=5)

    if len(peaks) > 0:
        # Devolver el último mínimo local (más cercano al pico)
        local_min_idx = peaks[-1] + start_idx
        return local_min_idx
    else:
        # Si no hay mínimo local, devolver el índice del mínimo absoluto
        return segment['glucose'].idxmin()

Función para detectar cuando un evento de comida:
* El `timestamp` de inicio de evento de comida es al momento de máximo local del evento de comida completo (1 hr antes y 2 hrs después)
* Corrige el registro, asignando el `timestamp` al tiempo que se detecta el mínimo local anterior.

Si el registro tiene muy pocos datos, se añade directamente, más adelante se tratarán los datos faltantes.

In [482]:
# Función para detectar y corregir timestamps de comidas mal registradas
def detect_and_correct_meal_timestamps(df, meals, verbose=False):
    """
    Detecta comidas registradas en el momento del pico de glucosa (posiblemente tardías) 
    y corrige su timestamp al mínimo local anterior.

    Parámetros:
    - df: DataFrame general con series de tiempo por paciente (contiene glucosa y eventos).
    - meals: Subset del DataFrame que contiene solo registros de comidas.
    - verbose: Si es True, imprime detalles de cada corrección aplicada.

    Retorna:
    - DataFrame con los eventos de comida (corregidos o no), incluyendo metadatos de la corrección.
    """
    
    corrected_meals = []
    cols_to_move = [
        'meal_type', 'calories', 'carbs', 'protein', 'fat',
        'fiber', 'amount_consumed', 'calories_(activity)', 'image_path'
    ]
    glucose_sources = ['libre_gl', 'dexcom_gl']

    for _, meal in meals.iterrows():
        patient_id = meal['patient']
        original_time = meal['timestamp']

        # Ventana de análisis
        start_analysis = original_time - pd.Timedelta(hours=1)
        end_analysis = original_time + pd.Timedelta(hours=2)

        # Base para filtrar datos por paciente y tiempo
        base_mask = (
            (df['patient'] == patient_id) &
            (df['timestamp'] >= start_analysis) &
            (df['timestamp'] <= end_analysis)
        )

        # Usar el primer sensor con al menos 10 registros
        for glucose_col in glucose_sources:
            sensor_mask = base_mask & df[glucose_col].notna()
            analysis_data = df[sensor_mask].copy().sort_values('timestamp').reset_index(drop=True)
            if len(analysis_data) >= 10:
                break
        else:
            meal_copy = meal.copy()
            meal_copy['correction_applied'] = False
            corrected_meals.append(meal_copy)
            continue

        # Ubicar el índice más cercano al tiempo original
        time_diffs = abs(analysis_data['timestamp'] - original_time)
        original_idx = time_diffs.idxmin()

        # Usar columna de glucosa estandarizada para análisis
        analysis_data = analysis_data.rename(columns={glucose_col: 'glucose'})

        original_glucose = analysis_data.loc[original_idx, 'glucose']
        max_glucose = analysis_data['glucose'].max()
        max_glucose_idx = analysis_data['glucose'].values.argmax()

        # Verificar si el evento está en o cerca del pico
        is_at_peak = original_glucose >= (0.95 * max_glucose)

        if is_at_peak:
            corrected_idx = find_local_minimum_before_peak(analysis_data, max_glucose_idx)
            corrected_time = analysis_data.loc[corrected_idx, 'timestamp']
            corrected_glucose = analysis_data.loc[corrected_idx, 'glucose']

            if verbose:
                print(f"\nPaciente {patient_id} - Sensor: {glucose_col}")
                print(f"  Tiempo original: {original_time}")
                print(f"  Glucosa en registro: {original_glucose:.1f}")
                print(f"  Glucosa máxima: {max_glucose:.1f}")
                print(f"  Tiempo corregido: {corrected_time}")
                print(f"  Glucosa corregida: {corrected_glucose:.1f}")
                print(f"  Diferencia temporal: {(original_time - corrected_time).total_seconds() / 60:.1f} minutos")
                print("-" * 50)

            # Reasignar evento si el timestamp corregido está suficientemente cerca
            patient_data = df[df['patient'] == patient_id]
            time_diffs = abs(patient_data['timestamp'] - corrected_time)
            closest_idx = time_diffs.idxmin()
            closest_timestamp = patient_data.loc[closest_idx, 'timestamp']
            time_diff_minutes = abs((closest_timestamp - corrected_time).total_seconds()) / 60

            if time_diff_minutes <= 5:
                if pd.isna(df.at[closest_idx, 'meal_type']):
                    # Vaciar columnas en el timestamp original
                    original_idx = df[(df['patient'] == patient_id) & (df['timestamp'] == original_time)].index
                    if len(original_idx) == 1:
                        df.loc[original_idx, cols_to_move] = pd.NA

                    # Reasignar en el nuevo timestamp
                    for col in cols_to_move:
                        df.at[closest_idx, col] = meal[col]

                    # Guardar versión corregida
                    corrected_meal = meal.copy()
                    corrected_meal['timestamp'] = corrected_time
                    corrected_meal['original_timestamp'] = original_time
                    corrected_meal['correction_applied'] = True
                    corrected_meal['original_glucose'] = original_glucose
                    corrected_meal['corrected_glucose'] = corrected_glucose
                    corrected_meal['sensor_used'] = glucose_col
                    corrected_meals.append(corrected_meal)
                else:
                    if verbose:
                        print("No se reasignó: el nuevo timestamp ya contiene un evento de comida.")
                    meal_copy = meal.copy()
                    meal_copy['correction_applied'] = False
                    corrected_meals.append(meal_copy)
            else:
                if verbose:
                    print(f"No se reasignó: diferencia temporal demasiado grande ({time_diff_minutes:.1f} min).")
                meal_copy = meal.copy()
                meal_copy['correction_applied'] = False
                corrected_meals.append(meal_copy)
        else:
            meal_copy = meal.copy()
            meal_copy['correction_applied'] = False
            corrected_meals.append(meal_copy)

    return pd.DataFrame(corrected_meals)


In [483]:
df[df['meal_type'].notna()]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
233,2020-05-01 14:23:00,69.800000,109.4,95.0,4.61296,44.0,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,photos/00000005-PHOTO-2020-5-1-14-23-0.jpg,1
618,2020-05-01 20:48:00,84.800000,114.8,81.0,1.36292,13.0,dinner,80.0,18.0,0.0,0.0,0.0,100.0,photos/00000007-PHOTO-2020-5-1-20-48-0.jpg,1
825,2020-05-02 00:15:00,81.000000,97.4,78.0,4.40328,42.0,snack,110.0,24.0,0.0,2.0,0.0,100.0,NaN,1
1308,2020-05-02 08:18:00,88.400000,101.8,81.0,3.14520,30.0,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000010-PHOTO-2020-5-2-8-18-0.jpg,1
1530,2020-05-02 12:00:00,82.000000,80.0,84.0,2.93552,28.0,lunch,840.0,89.0,17.0,42.0,3.0,100.0,photos/00000012-PHOTO-2020-5-2-12-0-0.jpg,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684069,2025-05-19 13:37:00,82.933333,94.8,93.0,1.02443,11.0,lunch,445.0,43.0,20.0,20.0,13.0,100.0,photos/00000082-PHOTO-2025-5-19-13-37-0.jpg,49
684473,2025-05-19 20:21:00,88.466667,106.0,78.0,0.93130,10.0,dinner,370.0,38.0,26.0,12.0,2.0,100.0,photos/00000084-PHOTO-2025-5-19-20-21-0.jpg,49
685118,2025-05-20 07:06:00,138.333333,152.6,81.0,2.23512,24.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000085-PHOTO-2025-5-20-7-6-0.jpg,49
685465,2025-05-20 12:53:00,84.000000,100.0,84.0,1.02443,11.0,lunch,725.0,94.0,44.0,20.0,4.0,100.0,photos/00000098-PHOTO-2025-5-20-12-53-0.jpg,49


In [484]:
# Aplicar la detección y corrección de comidas registradas en el pico
corrected_meals_df = detect_and_correct_meal_timestamps(df, meals)

In [485]:
corrected_meals_df[corrected_meals_df['correction_applied'] == True]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient,correction_applied,original_timestamp,original_glucose,corrected_glucose,sensor_used
2002,2020-05-02 19:15:00,94.266667,94.6,88.0,3.56456,34.0,dinner,330.0,32.0,4.0,22.0,4.0,100.0,photos/00000016-PHOTO-2020-5-2-19-52-0.jpg,1,True,2020-05-02 19:52:00,94.266667,87.000000,libre_gl
5926,2020-05-05 12:30:00,96.666667,108.0,88.0,3.14520,30.0,lunch,425.0,28.0,27.0,23.0,3.0,100.0,photos/00000042-PHOTO-2020-5-5-13-16-0.jpg,1,True,2020-05-05 13:16:00,96.666667,83.000000,libre_gl
10799,2020-05-08 21:52:00,107.800000,138.0,83.0,3.14520,30.0,snack,120.0,28.0,1.0,0.0,0.0,100.0,photos/00000072-PHOTO-2020-5-8-22-29-0.jpg,1,True,2020-05-08 22:29:00,107.800000,104.000000,libre_gl
11996,2020-05-09 17:45:00,118.400000,120.4,79.0,1.36292,13.0,dinner,120.0,28.0,1.0,0.0,0.0,100.0,photos/00000079-PHOTO-2020-5-9-18-26-0.jpg,1,True,2020-05-09 18:26:00,118.400000,82.000000,libre_gl
23338,2019-11-21 18:40:00,141.533333,204.6,NaN,0.93120,10.0,dinner,41.0,3.0,2.0,2.0,0.0,80.0,photos/00000064-PHOTO-2019-11-21-19-40-0.jpg,2,True,2019-11-21 19:40:00,141.533333,80.533333,libre_gl
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633791,2025-05-07 17:47:00,261.866667,268.0,94.0,1.60170,NaN,dinner,642.0,80.0,8.0,33.0,3.0,100.0,photos/00000065-PHOTO-2025-5-7-18-33-0.jpg,46,True,2025-05-07 18:33:00,261.866667,229.000000,libre_gl
634821,2025-05-08 10:43:00,128.400000,139.0,90.0,1.38814,NaN,lunch,585.0,40.0,38.0,17.0,13.0,100.0,photos/00000068-PHOTO-2025-5-8-11-43-0.jpg,46,True,2025-05-08 11:43:00,128.400000,108.266667,libre_gl
652265,2025-11-08 08:59:00,209.333333,242.8,107.0,4.72692,44.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000058-PHOTO-2025-11-8-9-59-0.jpg,47,True,2025-11-08 09:59:00,209.333333,211.000000,libre_gl
682994,2025-05-18 19:38:00,180.800000,210.4,73.0,0.93130,10.0,dinner,590.0,69.0,31.0,42.0,9.0,100.0,photos/00000072-PHOTO-2025-5-18-19-42-0.jpg,49,True,2025-05-18 19:42:00,180.800000,180.000000,libre_gl


In [486]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687580 entries, 0 to 687579
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   timestamp            687580 non-null  datetime64[ns]
 1   libre_gl             687360 non-null  float64       
 2   dexcom_gl            629825 non-null  float64       
 3   hr                   610256 non-null  float64       
 4   calories_(activity)  652052 non-null  float64       
 5   mets                 501078 non-null  float64       
 6   meal_type            1706 non-null    object        
 7   calories             1706 non-null    float64       
 8   carbs                1706 non-null    float64       
 9   protein              1706 non-null    float64       
 10  fat                  1706 non-null    float64       
 11  fiber                1706 non-null    float64       
 12  amount_consumed      1706 non-null    float64       
 13  image_path    

In [487]:
df[df['meal_type'].notna()]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
233,2020-05-01 14:23:00,69.800000,109.4,95.0,4.61296,44.0,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,photos/00000005-PHOTO-2020-5-1-14-23-0.jpg,1
618,2020-05-01 20:48:00,84.800000,114.8,81.0,1.36292,13.0,dinner,80.0,18.0,0.0,0.0,0.0,100.0,photos/00000007-PHOTO-2020-5-1-20-48-0.jpg,1
825,2020-05-02 00:15:00,81.000000,97.4,78.0,4.40328,42.0,snack,110.0,24.0,0.0,2.0,0.0,100.0,NaN,1
1308,2020-05-02 08:18:00,88.400000,101.8,81.0,3.14520,30.0,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000010-PHOTO-2020-5-2-8-18-0.jpg,1
1530,2020-05-02 12:00:00,82.000000,80.0,84.0,2.93552,28.0,lunch,840.0,89.0,17.0,42.0,3.0,100.0,photos/00000012-PHOTO-2020-5-2-12-0-0.jpg,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684069,2025-05-19 13:37:00,82.933333,94.8,93.0,1.02443,11.0,lunch,445.0,43.0,20.0,20.0,13.0,100.0,photos/00000082-PHOTO-2025-5-19-13-37-0.jpg,49
684473,2025-05-19 20:21:00,88.466667,106.0,78.0,0.93130,10.0,dinner,370.0,38.0,26.0,12.0,2.0,100.0,photos/00000084-PHOTO-2025-5-19-20-21-0.jpg,49
685118,2025-05-20 07:06:00,138.333333,152.6,81.0,2.23512,24.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000085-PHOTO-2025-5-20-7-6-0.jpg,49
685465,2025-05-20 12:53:00,84.000000,100.0,84.0,1.02443,11.0,lunch,725.0,94.0,44.0,20.0,4.0,100.0,photos/00000098-PHOTO-2025-5-20-12-53-0.jpg,49


Función para validar las correcciones, para poder analizar si las correcciones tienen sentido o no, calculando el iAUC simple para verificar un cambio en el registro y cómo puede afectar en el cálculo futuro.

In [488]:
# Función para validar métricas antes y después de la corrección del timestamp
def validate_corrections(df, corrected_meals):
    """
    Valida las correcciones de timestamp analizando la respuesta glucémica
    antes y después del ajuste.

    Para cada corrección, calcula:
    - Área incremental bajo la curva (iAUC)
    - Pico de glucosa

    Parámetros:
    - df: DataFrame completo con datos de glucosa
    - corrected_meals: DataFrame con información de comidas corregidas (de detect_and_correct_meal_timestamps)
    """
    print("Validación de correcciones:")
    
    corrections = corrected_meals[corrected_meals['correction_applied'] == True]
    
    for _, meal in corrections.iterrows():
        patient_id = meal['patient']
        original_time = meal['original_timestamp']
        corrected_time = meal['timestamp']
        glucose_col = meal['sensor_used']  # 'libre_gl' o 'dexcom_gl'

        print(f"\nPaciente {int(patient_id):02d} (Sensor: {glucose_col})")
        print(f"  Timestamp original:  {original_time}")
        print(f"  Timestamp corregido: {corrected_time}")

        for time_label, timestamp in [("Original", original_time), ("Corregido", corrected_time)]:
            end_time = timestamp + pd.Timedelta(hours=2)

            # Filtrado de datos glucémicos para esa ventana
            mask = (
                (df['patient'] == patient_id) &
                (df['timestamp'] >= timestamp) &
                (df['timestamp'] <= end_time) &
                (df[glucose_col].notna())
            )
            glucose_data = df[mask].copy().sort_values('timestamp')

            if len(glucose_data) > 5:
                # Cálculo de iAUC y pico
                baseline = glucose_data[glucose_col].iloc[0]
                glucose_data['time_minutes'] = (glucose_data['timestamp'] - timestamp).dt.total_seconds() / 60
                glucose_data['incremental'] = (glucose_data[glucose_col] - baseline).clip(lower=0)

                iauc = np.trapz(glucose_data['incremental'], glucose_data['time_minutes'])
                peak = glucose_data[glucose_col].max()

                print(f"  {time_label}: iAUC = {iauc:.1f}, Pico = {peak:.1f} mg/dL")
            else:
                print(f"  {time_label}: Insuficientes datos ({len(glucose_data)}) para calcular métricas.")


Se corrigen los registros tardíos para todos los eventos de comida del dataset original

In [489]:
# Validar las correcciones
validate_corrections(df, corrected_meals_df)

print(f"\nResumen:")
print(f"Comidas originales: {len(meals)}")
print(f"Correcciones aplicadas: {corrected_meals_df['correction_applied'].sum()}")
print(f"Porcentaje corregido: {(corrected_meals_df['correction_applied'].sum() / len(corrected_meals_df)) * 100:.1f}%")

Validación de correcciones:

Paciente 01 (Sensor: libre_gl)
  Timestamp original:  2020-05-02 19:52:00
  Timestamp corregido: 2020-05-02 19:15:00
  Original: iAUC = 202.9, Pico = 99.0 mg/dL
  Corregido: iAUC = 422.1, Pico = 99.0 mg/dL

Paciente 01 (Sensor: libre_gl)
  Timestamp original:  2020-05-05 13:16:00
  Timestamp corregido: 2020-05-05 12:30:00
  Original: iAUC = 0.0, Pico = 96.7 mg/dL
  Corregido: iAUC = 615.0, Pico = 97.0 mg/dL

Paciente 01 (Sensor: libre_gl)
  Timestamp original:  2020-05-08 22:29:00
  Timestamp corregido: 2020-05-08 21:52:00
  Original: iAUC = 0.2, Pico = 108.0 mg/dL
  Corregido: iAUC = 75.3, Pico = 108.0 mg/dL

Paciente 01 (Sensor: libre_gl)
  Timestamp original:  2020-05-09 18:26:00
  Timestamp corregido: 2020-05-09 17:45:00
  Original: iAUC = 40.7, Pico = 124.0 mg/dL
  Corregido: iAUC = 2160.0, Pico = 124.0 mg/dL

Paciente 02 (Sensor: libre_gl)
  Timestamp original:  2019-11-21 19:40:00
  Timestamp corregido: 2019-11-21 18:40:00
  Original: iAUC = 0.0, Pic

Función para filtar los eventos de comida, tomando en cuenta los rangos de tiempo definidos anteriormente.

In [490]:
corrected_meals_df

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient,correction_applied,original_timestamp,original_glucose,corrected_glucose,sensor_used
233,2020-05-01 14:23:00,69.800000,109.4,95.0,4.61296,44.0,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,photos/00000005-PHOTO-2020-5-1-14-23-0.jpg,1,False,NaT,NaN,NaN,NaN
618,2020-05-01 20:48:00,84.800000,114.8,81.0,1.36292,13.0,dinner,80.0,18.0,0.0,0.0,0.0,100.0,photos/00000007-PHOTO-2020-5-1-20-48-0.jpg,1,False,NaT,NaN,NaN,NaN
825,2020-05-02 00:15:00,81.000000,97.4,78.0,4.40328,42.0,snack,110.0,24.0,0.0,2.0,0.0,100.0,NaN,1,False,NaT,NaN,NaN,NaN
1308,2020-05-02 08:18:00,88.400000,101.8,81.0,3.14520,30.0,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000010-PHOTO-2020-5-2-8-18-0.jpg,1,False,NaT,NaN,NaN,NaN
1530,2020-05-02 12:00:00,82.000000,80.0,84.0,2.93552,28.0,lunch,840.0,89.0,17.0,42.0,3.0,100.0,photos/00000012-PHOTO-2020-5-2-12-0-0.jpg,1,False,NaT,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684069,2025-05-19 13:37:00,82.933333,94.8,93.0,1.02443,11.0,lunch,445.0,43.0,20.0,20.0,13.0,100.0,photos/00000082-PHOTO-2025-5-19-13-37-0.jpg,49,False,NaT,NaN,NaN,NaN
684473,2025-05-19 20:21:00,88.466667,106.0,78.0,0.93130,10.0,dinner,370.0,38.0,26.0,12.0,2.0,100.0,photos/00000084-PHOTO-2025-5-19-20-21-0.jpg,49,False,NaT,NaN,NaN,NaN
685118,2025-05-20 07:06:00,138.333333,152.6,81.0,2.23512,24.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000085-PHOTO-2025-5-20-7-6-0.jpg,49,False,NaT,NaN,NaN,NaN
685465,2025-05-20 12:53:00,84.000000,100.0,84.0,1.02443,11.0,lunch,725.0,94.0,44.0,20.0,4.0,100.0,photos/00000098-PHOTO-2025-5-20-12-53-0.jpg,49,False,NaT,NaN,NaN,NaN


In [491]:
meals

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient
233,2020-05-01 14:23:00,69.800000,109.4,95.0,4.61296,44.0,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,photos/00000005-PHOTO-2020-5-1-14-23-0.jpg,1
618,2020-05-01 20:48:00,84.800000,114.8,81.0,1.36292,13.0,dinner,80.0,18.0,0.0,0.0,0.0,100.0,photos/00000007-PHOTO-2020-5-1-20-48-0.jpg,1
825,2020-05-02 00:15:00,81.000000,97.4,78.0,4.40328,42.0,snack,110.0,24.0,0.0,2.0,0.0,100.0,NaN,1
1308,2020-05-02 08:18:00,88.400000,101.8,81.0,3.14520,30.0,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000010-PHOTO-2020-5-2-8-18-0.jpg,1
1530,2020-05-02 12:00:00,82.000000,80.0,84.0,2.93552,28.0,lunch,840.0,89.0,17.0,42.0,3.0,100.0,photos/00000012-PHOTO-2020-5-2-12-0-0.jpg,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684069,2025-05-19 13:37:00,82.933333,94.8,93.0,1.02443,11.0,lunch,445.0,43.0,20.0,20.0,13.0,100.0,photos/00000082-PHOTO-2025-5-19-13-37-0.jpg,49
684473,2025-05-19 20:21:00,88.466667,106.0,78.0,0.93130,10.0,dinner,370.0,38.0,26.0,12.0,2.0,100.0,photos/00000084-PHOTO-2025-5-19-20-21-0.jpg,49
685118,2025-05-20 07:06:00,138.333333,152.6,81.0,2.23512,24.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000085-PHOTO-2025-5-20-7-6-0.jpg,49
685465,2025-05-20 12:53:00,84.000000,100.0,84.0,1.02443,11.0,lunch,725.0,94.0,44.0,20.0,4.0,100.0,photos/00000098-PHOTO-2025-5-20-12-53-0.jpg,49


In [492]:
meals = df[df['meal_type'].notna()]

In [493]:
meals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1706 entries, 233 to 686560
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp            1706 non-null   datetime64[ns]
 1   libre_gl             1705 non-null   float64       
 2   dexcom_gl            1685 non-null   float64       
 3   hr                   1626 non-null   float64       
 4   calories_(activity)  1684 non-null   float64       
 5   mets                 1320 non-null   float64       
 6   meal_type            1706 non-null   object        
 7   calories             1706 non-null   float64       
 8   carbs                1706 non-null   float64       
 9   protein              1706 non-null   float64       
 10  fat                  1706 non-null   float64       
 11  fiber                1706 non-null   float64       
 12  amount_consumed      1706 non-null   float64       
 13  image_path           1644 non-null

In [494]:
# Ordenar y preparar
meals = meals.sort_values(['patient', 'timestamp']).reset_index(drop=True)
eventos_validos_idx = []

# Filtrado estricto
for i, row in tqdm(meals.iterrows(), total=len(meals)):
    pid = row['patient']
    ts = row['timestamp']
    
    comparables = meals[(meals['patient'] == pid) & (meals.index != i)]
    time_diffs = (comparables['timestamp'] - ts).dt.total_seconds() / 60

    hay_solapamiento = (
        ((time_diffs > -60) & (time_diffs < 0)) |   # Comidas en 1h anterior
        ((time_diffs > 0) & (time_diffs < 120))     # Comidas en 2h posterior (estricto)
    ).any()
    
    if not hay_solapamiento:
        eventos_validos_idx.append(i)

# Dataset final filtrado
meals_filtrados = meals.loc[eventos_validos_idx].reset_index(drop=True)


100%|██████████| 1706/1706 [00:02<00:00, 755.81it/s]


In [495]:
print(f"Eventos de comida originales: {len(meals)}")
print(f"Eventos de comida filtrados: {len(meals_filtrados)}")


Eventos de comida originales: 1706
Eventos de comida filtrados: 1330


In [496]:
problemas = []
for _, row in meals_filtrados.iterrows():
    posteriores = df[
        (df['patient'] == row['patient']) &
        (df['timestamp'] > row['timestamp']) &
        (df['timestamp'] < row['timestamp'] + pd.Timedelta(hours=2)) &  
        (df['meal_type'].notna())
    ]
    if len(posteriores) > 0:
        problemas.append((row['patient'], row['timestamp']))

print(f"Eventos de meals_filtrados con otra comida en las 2h posteriores: {len(problemas)}")

Eventos de meals_filtrados con otra comida en las 2h posteriores: 0


In [497]:
# Crear lista para almacenar los datos postprandiales
postprandial_rows = []

# Recorrer cada evento de comida válido
for _, row in tqdm(meals_filtrados.iterrows(), total=len(meals_filtrados)):
    patient_id = row['patient']
    start_time = row['timestamp']
    end_time = start_time + pd.Timedelta(minutes=120)

    # Extraer ventana postprandial de df
    window = df[
        (df['patient'] == patient_id) &
        (df['timestamp'] >= start_time) &
        (df['timestamp'] <= end_time)
    ].copy()

    # Agregar columna para identificar el inicio del evento
    window['meal_timestamp'] = start_time
    postprandial_rows.append(window)

# Concatenar todos los fragmentos
postprandial_data = pd.concat(postprandial_rows).reset_index(drop=True)


100%|██████████| 1330/1330 [00:07<00:00, 187.86it/s]


In [498]:
postprandial_data.loc[
    postprandial_data['timestamp'] != postprandial_data['meal_timestamp'],
    ['meal_type', 'calories', 'carbs', 'protein', 'fat', 'fiber', 'amount_consumed', 'image_path']
] = pd.NA

In [499]:
postprandial_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159700 entries, 0 to 159699
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   timestamp            159700 non-null  datetime64[ns]
 1   libre_gl             159590 non-null  float64       
 2   dexcom_gl            156336 non-null  float64       
 3   hr                   150637 non-null  float64       
 4   calories_(activity)  158070 non-null  float64       
 5   mets                 122594 non-null  float64       
 6   meal_type            1330 non-null    object        
 7   calories             1330 non-null    float64       
 8   carbs                1330 non-null    float64       
 9   protein              1330 non-null    float64       
 10  fat                  1330 non-null    float64       
 11  fiber                1330 non-null    float64       
 12  amount_consumed      1330 non-null    float64       
 13  image_path    

In [500]:
postprandial_data[postprandial_data['meal_type'].notna()]

,timestamp,libre_gl,dexcom_gl,hr,calories_(activity),mets,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,image_path,patient,meal_timestamp
0,2020-05-01 14:23:00,69.800000,109.4,95.0,4.61296,44.0,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,photos/00000005-PHOTO-2020-5-1-14-23-0.jpg,1,2020-05-01 14:23:00
121,2020-05-01 20:48:00,84.800000,114.8,81.0,1.36292,13.0,dinner,80.0,18.0,0.0,0.0,0.0,100.0,photos/00000007-PHOTO-2020-5-1-20-48-0.jpg,1,2020-05-01 20:48:00
242,2020-05-02 00:15:00,81.000000,97.4,78.0,4.40328,42.0,snack,110.0,24.0,0.0,2.0,0.0,100.0,NaN,1,2020-05-02 00:15:00
363,2020-05-02 08:18:00,88.400000,101.8,81.0,3.14520,30.0,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,photos/00000010-PHOTO-2020-5-2-8-18-0.jpg,1,2020-05-02 08:18:00
484,2020-05-02 12:00:00,82.000000,80.0,84.0,2.93552,28.0,lunch,840.0,89.0,17.0,42.0,3.0,100.0,photos/00000012-PHOTO-2020-5-2-12-0-0.jpg,1,2020-05-02 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159095,2025-05-19 13:37:00,82.933333,94.8,93.0,1.02443,11.0,lunch,445.0,43.0,20.0,20.0,13.0,100.0,photos/00000082-PHOTO-2025-5-19-13-37-0.jpg,49,2025-05-19 13:37:00
159216,2025-05-19 20:21:00,88.466667,106.0,78.0,0.93130,10.0,dinner,370.0,38.0,26.0,12.0,2.0,100.0,photos/00000084-PHOTO-2025-5-19-20-21-0.jpg,49,2025-05-19 20:21:00
159337,2025-05-20 07:06:00,138.333333,152.6,81.0,2.23512,24.0,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,photos/00000085-PHOTO-2025-5-20-7-6-0.jpg,49,2025-05-20 07:06:00
159458,2025-05-20 12:53:00,84.000000,100.0,84.0,1.02443,11.0,lunch,725.0,94.0,44.0,20.0,4.0,100.0,photos/00000098-PHOTO-2025-5-20-12-53-0.jpg,49,2025-05-20 12:53:00


Ahora se guardan los registros, ya filtrados, junto a su ventana de tiempo, en un dataset que contiene solamente la información de los alimentos, sin tomar en cuenta otras mediciones de glucosa innecesarias para el análisis.

En este punto conseguimos tener 1330 eventos de comida que cumplen con las condiciones iniciales:
* No debe ocurrir un evento de comida 1 hora antes del evento de comida que se analiza: El cruce de dos eventos de comida afecta directamente a la respuesta glucémica.
* No debe ocurrir un evento de comida 2 horas después del evento de comida que se analiza.

Además todos los registros de glucosa están asignados "correctamente en el tiempo", y no "inician" al mismo tiempo que hay un máximo local.

Ahora, según los datos faltantes de mediciones de glucosa de sensores, se decide si se hara una imputación de su medición, o se eliminará ese registro de comida.

In [501]:
# Lista para guardar resultados
meals_with_missing_data = []

# Filtrar filas donde meal_type no es nulo (es decir, inicio de evento de comida)
meal_starts = postprandial_data[postprandial_data['meal_type'].notna()]

# Iterar por cada inicio de comida
for _, meal_row in meal_starts.iterrows():
    patient_id = meal_row['patient']
    meal_time = meal_row['timestamp']
    end_time = meal_time + interval_post

    # Filtrar registros de ese paciente dentro de la ventana de 2 horas
    mask = (
        (postprandial_data['patient'] == patient_id) &
        (postprandial_data['timestamp'] >= meal_time) &
        (postprandial_data['timestamp'] <= end_time)
    )
    window_data = postprandial_data[mask]

    # Verificar si hay datos faltantes
    missing_dexcom = window_data['dexcom_gl'].isna().any()
    missing_libre = window_data['libre_gl'].isna().any()

    # Guardar info
    meals_with_missing_data.append({
        'patient': patient_id,
        'meal_time': meal_time,
        'meal_type': meal_row['meal_type'],
        'n_rows': len(window_data),
        'missing_dexcom': missing_dexcom,
        'missing_libre': missing_libre,
        'n_missing_dexcom': window_data['dexcom_gl'].isna().sum(),
        'n_missing_libre': window_data['libre_gl'].isna().sum()
    })

# Convertimos a DataFrame
missing_df = pd.DataFrame(meals_with_missing_data)


In [502]:
missing_df

,patient,meal_time,meal_type,n_rows,missing_dexcom,missing_libre,n_missing_dexcom,n_missing_libre
0,1,2020-05-01 14:23:00,lunch,121,False,False,0,0
1,1,2020-05-01 20:48:00,dinner,121,False,False,0,0
2,1,2020-05-02 00:15:00,snack,121,False,False,0,0
3,1,2020-05-02 08:18:00,breakfast,121,False,False,0,0
4,1,2020-05-02 12:00:00,lunch,121,False,False,0,0
...,...,...,...,...,...,...,...,...
1325,49,2025-05-19 13:37:00,lunch,121,False,False,0,0
1326,49,2025-05-19 20:21:00,dinner,121,False,False,0,0
1327,49,2025-05-20 07:06:00,breakfast,121,False,False,0,0
1328,49,2025-05-20 12:53:00,lunch,121,False,False,0,0


In [503]:
len(missing_df[(missing_df['n_missing_dexcom'] > 0) | (missing_df['n_missing_libre'] > 0)])

40

Se tiene que existen 40 registros los cuales tienen datos faltantes en las mediciones, vamos a explorar cómo ocurren

In [504]:
missing_df[(missing_df['n_missing_dexcom'] > 0) | (missing_df['n_missing_libre'] > 0)]

,patient,meal_time,meal_type,n_rows,missing_dexcom,missing_libre,n_missing_dexcom,n_missing_libre
33,1,2020-05-11 10:58:00,breakfast,121,True,False,121,0
56,2,2019-11-25 20:23:00,breakfast,121,True,False,121,0
89,3,2020-03-21 07:56:00,breakfast,121,True,False,108,0
110,4,2023-09-18 08:57:00,breakfast,121,False,True,0,110
149,5,2020-08-27 09:06:00,breakfast,121,True,False,29,0
195,8,2023-01-30 12:34:00,lunch,121,True,False,20,0
217,8,2023-02-06 18:46:00,dinner,121,True,False,30,0
224,8,2023-02-09 08:12:00,breakfast,121,True,False,75,0
324,11,2020-10-13 07:31:00,breakfast,121,True,False,68,0
357,12,2023-03-08 09:33:00,breakfast,106,True,False,106,0


Ahora que analizamos los datos faltantes, vemos que la imputación de valores es muy complicada, porque hay eventos de comida completos sin registros en un sensor (`dexcom_gl`). Además cuando hay datos faltantes, se entiende que son continuos por una desconexión del sensor.
Además se observa que la mayoría de las veces (excepto una) el sensor que no tiene registros es el `dexcom_gl`.

Por este hecho se procede de la siguiente manera:
* Se mantiene un dataframe en el que se eliminan todos estos registros que tienen al menos un registro nulo en alguna de las mediciones de los sensores.
* Se crea otro dataframe en el que se elimina por completo la columna `dexcom_gl`, se borra el dato con registros nulos de `libre_gl` y se trabaja únicamente con los datos de este sensor.

De esta forma podremos analizar:
* Un modelo que tiene las mediciones de ambos sensores.
* Un modelo que tiene les mediciones de un único sensor.

In [505]:
# Filtrar filas donde meal_type no es nulo = inicio de evento de comida
meal_starts = postprandial_data[postprandial_data['meal_type'].notna()]

# Lista para guardar los registros de eventos completos
complete_event_records = []

# Iterar por cada inicio de evento de comida
for _, meal_row in meal_starts.iterrows():
    patient_id = meal_row['patient']
    meal_time = meal_row['timestamp']
    end_time = meal_time + interval_post

    # Filtrar registros postprandiales de ese paciente
    mask = (
        (postprandial_data['patient'] == patient_id) &
        (postprandial_data['timestamp'] >= meal_time) &
        (postprandial_data['timestamp'] <= end_time)
    )
    event_data = postprandial_data[mask]

    # Verificar que NO hay ningún valor nulo en los sensores
    if event_data['dexcom_gl'].isna().any() or event_data['libre_gl'].isna().any():
        continue  # Saltamos eventos con faltantes en cualquiera de los sensores

    # Si es completo, lo añadimos al resultado
    complete_event_records.append(event_data)

# Concatenar todos los eventos completos
glucose_both_df = pd.concat(complete_event_records, ignore_index=True)

In [506]:
len(glucose_both_df[glucose_both_df['meal_type'].notna()])

1290

In [507]:
glucose_both_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155011 entries, 0 to 155010
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   timestamp            155011 non-null  datetime64[ns]
 1   libre_gl             155011 non-null  float64       
 2   dexcom_gl            155011 non-null  float64       
 3   hr                   146380 non-null  float64       
 4   calories_(activity)  153624 non-null  float64       
 5   mets                 118983 non-null  float64       
 6   meal_type            1290 non-null    object        
 7   calories             1290 non-null    float64       
 8   carbs                1290 non-null    float64       
 9   protein              1290 non-null    float64       
 10  fat                  1290 non-null    float64       
 11  fiber                1290 non-null    float64       
 12  amount_consumed      1290 non-null    float64       
 13  image_path    

El conjunto de datos `glucose_both_df` tiene los registros de glucosa de 1290 eventos de comida, con las mediciones completas de ambos sensores de glucosa.

Ahora obtenemos el dataset que contiene los registros de glucosa para aquellos eventos de comida que tienen los registros completos para el sensor `libre_gl`

In [508]:
# Identificar inicios de eventos de comida
meal_starts = postprandial_data[postprandial_data['meal_type'].notna()]

# Lista para almacenar los eventos válidos
libre_only_event_records = []

# Iterar por cada comida
for _, meal_row in meal_starts.iterrows():
    patient_id = meal_row['patient']
    meal_time = meal_row['timestamp']
    end_time = meal_time + interval_post

    # Filtrar los registros de glucosa para ese paciente en la ventana de 1h
    mask = (
        (postprandial_data['patient'] == patient_id) &
        (postprandial_data['timestamp'] >= meal_time) &
        (postprandial_data['timestamp'] <= end_time)
    )
    event_data = postprandial_data[mask]

    # Verificamos que NO hay nulos en libre_gl (es decir, está completo)
    if event_data['libre_gl'].isna().any():
        continue  # Saltamos eventos con datos faltantes en Libre

    # Guardamos los registros válidos
    libre_only_event_records.append(event_data)

# Concatenamos los eventos válidos
glucose_libre_only_df = pd.concat(libre_only_event_records, ignore_index=True)

# Eliminamos la columna dexcom_gl
glucose_libre_only_df = glucose_libre_only_df.drop(columns=['dexcom_gl'])


In [509]:
len(glucose_libre_only_df[glucose_libre_only_df['meal_type'].notna()])

1329

El conjunto de datos `glucose_libre_only_df` contiene los registros de medición de glucosa para los eventos de comida que tienen los registros completos para el sensor `libre_gl`.

Así tenemos dos dataframes para trabajar:
* `glucose_both_df`
* `glucose_libre_only_df` 

Sigue poder observar estas curvas válidas que se crean en los dataframes

# **10. Visualización de curvas válidas** <a class="anchor" id="10"></a>

[Tabla de Contenidos](#0.1)

In [510]:
# Definir ventana postprandial y condiciones de análisis
interval_post = pd.Timedelta(hours=2)
min_delay = pd.Timedelta(minutes=15)
max_plots = 50
plot_count = 0

# Filtrar eventos de comida con meal_type definido
meal_starts = postprandial_data[postprandial_data['meal_type'].notna()]

# Iterar sobre eventos de comida
for _, row in meal_starts.iterrows():
    if plot_count >= max_plots:
        print(f"Se alcanzó el máximo de {max_plots} gráficos.")
        break

    start_time = row['timestamp']
    end_time = start_time + interval_post
    patient_id = row['patient']

    # Filtrado por paciente y ventana temporal
    mask = (
        (glucose_both_df['timestamp'] >= start_time) &
        (glucose_both_df['timestamp'] <= end_time) &
        (glucose_both_df['patient'] == patient_id)
    )
    sub_data = glucose_both_df[mask]
    delayed_data = sub_data[sub_data['timestamp'] >= (start_time + min_delay)]

    # Inicializar picos
    peak_libre, peak_time_libre = None, None
    peak_dexcom, peak_time_dexcom = None, None

    if not delayed_data['libre_gl'].isnull().all():
        peak_libre = delayed_data['libre_gl'].max()
        peak_time_libre = delayed_data[delayed_data['libre_gl'] == peak_libre]['timestamp'].iloc[0]

    if not delayed_data['dexcom_gl'].isnull().all():
        peak_dexcom = delayed_data['dexcom_gl'].max()
        peak_time_dexcom = delayed_data[delayed_data['dexcom_gl'] == peak_dexcom]['timestamp'].iloc[0]

    if peak_libre is not None or peak_dexcom is not None:
        # Crear figura con 2 subplots: curva + imagen
        fig = plt.figure(figsize=(14, 6))
        gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1])
        ax_main = fig.add_subplot(gs[0])

        # Libre GL
        if peak_libre is not None:
            ax_main.plot(sub_data['timestamp'], sub_data['libre_gl'], label='Libre GL', color='#C44E52', marker='o', markersize=3)
            ax_main.scatter(peak_time_libre, peak_libre, color='darkred', s=100, label='Pico (Libre GL)')
            if not pd.isna(row['libre_gl']):
                ax_main.scatter(start_time, row['libre_gl'], color='purple', marker='D', s=100, label='Inicio Meal (Libre GL)')

        # Dexcom GL
        if peak_dexcom is not None:
            ax_main.plot(sub_data['timestamp'], sub_data['dexcom_gl'], label='Dexcom GL', color='#55A868', marker='s', markersize=3)
            ax_main.scatter(peak_time_dexcom, peak_dexcom, color='darkorange', s=100, label='Pico (Dexcom GL)')
            if not pd.isna(row['dexcom_gl']):
                ax_main.scatter(start_time, row['dexcom_gl'], color='cyan', marker='D', s=100, label='Inicio Meal (Dexcom GL)')

        ax_main.set_title(f"Curva de glucosa - Paciente {int(patient_id):02d} - {start_time.strftime('%Y-%m-%d %H:%M')}", fontsize=13)
        ax_main.set_xlabel("Tiempo", fontsize=11)
        ax_main.set_ylabel("Glucosa (mg/dL)", fontsize=11)
        ax_main.legend(fontsize=9)
        ax_main.grid(True, linestyle='--', alpha=0.4)
        ax_main.tick_params(axis='x', rotation=45)

        # Subplot para imagen asociada
        img_path = row.get('image_path')
        if img_path:
            try:
                patient_path = f'CGMacros/CGMacros-{int(patient_id):03d}/'
                path_final = os.path.join(patient_path, img_path)
                if os.path.exists(path_final):
                    ax_img = fig.add_subplot(gs[1])
                    img = plt.imread(path_final)
                    ax_img.imshow(img)
                    ax_img.axis('off')
                    ax_img.set_title(f'Comida del Paciente {int(patient_id):02d}', fontsize=11)
                else:
                    print(f"Imagen no encontrada: {path_final}")
            except Exception as e:
                print(f"Error al cargar la imagen: {e}")

        plt.tight_layout()
        plt.show()
        plot_count += 1


<Figure size 1400x600 with 2 Axes>

<Figure size 1400x600 with 2 Axes>

Error al cargar la imagen: join() argument must be str, bytes, or os.PathLike object, not 'float'


<Figure size 1400x600 with 1 Axes>

<Figure size 1400x600 with 2 Axes>

...

<Figure size 1400x600 with 2 Axes>

<Figure size 1400x600 with 2 Axes>

<Figure size 1400x600 with 2 Axes>

<Figure size 1400x600 with 2 Axes>

Se alcanzó el máximo de 50 gráficos.


Para el análisis, se cálcula el área incremental bajo la curva:

In [511]:
# Parámetros de visualización
interval_post = pd.Timedelta(hours=2)
max_plots = 25
plot_count = 0

# Iterar sobre eventos de comida
for _, row in meal_starts.iterrows():
    if plot_count >= max_plots:
        print(f"Se alcanzó el máximo de {max_plots} gráficos.")
        break

    start_time = row['timestamp']
    end_time = start_time + interval_post
    patient_id = row['patient']

    # Filtrar datos del paciente para esa ventana
    mask = (
        (glucose_both_df['timestamp'] >= start_time) &
        (glucose_both_df['timestamp'] <= end_time) &
        (glucose_both_df['patient'] == patient_id)
    )
    sub_data = glucose_both_df[mask]

    plt.figure(figsize=(12, 5))

    # LIBRE GL
    if pd.notna(row['libre_gl']):
        baseline_libre = row['libre_gl']
        libre_data = sub_data[['timestamp', 'libre_gl']].dropna()

        if len(libre_data) > 1:
            libre_times = (libre_data['timestamp'] - start_time).dt.total_seconds() / 60
            libre_values = libre_data['libre_gl'].values
            libre_above_baseline = np.maximum(0, libre_values - baseline_libre)

            plt.plot(libre_data['timestamp'], libre_values, label='Libre GL', color='#C44E52')
            plt.axhline(baseline_libre, linestyle='--', color='#C44E52', alpha=0.6, label='Línea base (Libre GL)')
            plt.fill_between(
                libre_data['timestamp'], baseline_libre, libre_values,
                where=libre_values > baseline_libre,
                interpolate=True,
                color='#C44E52', alpha=0.2, label='iAUC (Libre)'
            )

    # DEXCOM GL
    if pd.notna(row['dexcom_gl']):
        baseline_dexcom = row['dexcom_gl']
        dexcom_data = sub_data[['timestamp', 'dexcom_gl']].dropna()

        if len(dexcom_data) > 1:
            dexcom_times = (dexcom_data['timestamp'] - start_time).dt.total_seconds() / 60
            dexcom_values = dexcom_data['dexcom_gl'].values
            dexcom_above_baseline = np.maximum(0, dexcom_values - baseline_dexcom)

            plt.plot(dexcom_data['timestamp'], dexcom_values, label='Dexcom GL', color='#55A868')
            plt.axhline(baseline_dexcom, linestyle='--', color='#55A868', alpha=0.6, label='Línea base (Dexcom GL)')
            plt.fill_between(
                dexcom_data['timestamp'], baseline_dexcom, dexcom_values,
                where=dexcom_values > baseline_dexcom,
                interpolate=True,
                color='#55A868', alpha=0.2, label='iAUC (Dexcom)'
            )

    # Personalización del gráfico
    plt.title(f"Área Incremental Bajo la Curva (iAUC)\nPaciente {int(patient_id):02d} - {start_time.strftime('%Y-%m-%d %H:%M')}", fontsize=13)
    plt.xlabel("Tiempo", fontsize=11)
    plt.ylabel("Glucosa (mg/dL)", fontsize=11)
    plt.legend(fontsize=9)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

    plot_count += 1


<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

...

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

<Figure size 1200x500 with 1 Axes>

Se alcanzó el máximo de 25 gráficos.


# **11. Área incremental bajo la curva (iAUC)** <a class="anchor" id="11"></a>


[Tabla de Contenidos](#0.1)

El iAUC (Incremental Area Under the Curve) mide el área por encima del valor basal de glucosa durante un intervalo postprandial, en este caso 2 horas. Se excluye cualquier parte del área por debajo de la línea base.

Este cálculo indica la cantidad adicional de glucosa que se acumula en el cuerpo entre dos puntos de tiempo específicos.

Otras métricas que se calculan porque permiten entender mejor el comportamiento de la respuesta glucémica postpandrial:

* **Glucose Peak Value (Valor máximo)**: Refleja la magnitud máxima de la respuesta.
* **Time to Peak (TTP)**: Tiempo entre el inicio de la comida y el pico de glucosa (Alto valor de TTP → absorción lenta (más grasa/proteína) mientras que najo TTP → absorción rápida (altos carbohidratos simples)).
* **Glycemic Delta (ΔG)**: Diferencia entre el valor pico y el basal.
* **Early iAUC (ej. 0–30 min)**: Sirve para diferenciar comidas de absorción rápida (snacks) de otra.
* **Variabilidad postprandial**: Rango o desviación estándar en los 120 min.

In [512]:
glucose_both_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155011 entries, 0 to 155010
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   timestamp            155011 non-null  datetime64[ns]
 1   libre_gl             155011 non-null  float64       
 2   dexcom_gl            155011 non-null  float64       
 3   hr                   146380 non-null  float64       
 4   calories_(activity)  153624 non-null  float64       
 5   mets                 118983 non-null  float64       
 6   meal_type            1290 non-null    object        
 7   calories             1290 non-null    float64       
 8   carbs                1290 non-null    float64       
 9   protein              1290 non-null    float64       
 10  fat                  1290 non-null    float64       
 11  fiber                1290 non-null    float64       
 12  amount_consumed      1290 non-null    float64       
 13  image_path    

In [513]:
len(glucose_both_df)

155011

Imputación de la variable `calories_(activity)` para añadirla como variable informativa en cada evento de comida, con su promedio.

In [514]:
df_1 = glucose_both_df.copy()
df_1 = df_1.sort_values(by=['patient', 'timestamp'])

# Lista de variables a imputar dentro de ventanas post comida
variables_a_imputar = ['calories_(activity)', 'hr', 'mets']

# Imputación dentro de las ventanas de 2 horas post comida
for var in variables_a_imputar:
    for _, meal_row in df_1[df_1['meal_type'].notna()].iterrows():
        patient_id = meal_row['patient']
        start_time = meal_row['timestamp']
        end_time = start_time + pd.Timedelta(hours=2)

        mask = (
            (df_1['patient'] == patient_id) &
            (df_1['timestamp'] >= start_time) &
            (df_1['timestamp'] <= end_time)
        )

        # Interpolación + ffill + bfill
        temp = df_1.loc[mask, var].copy()
        temp = temp.interpolate(method='linear', limit_direction='both')
        temp = temp.fillna(method='ffill').fillna(method='bfill')
        df_1.loc[mask, var] = temp

    # Imputación global: promedio por paciente
    df_1[var] = df_1[var].fillna(
        df_1.groupby('patient')[var].transform('mean')
    )

    # Si aún quedan nulos: promedio general
    df_1[var] = df_1[var].fillna(df_1[var].mean())



In [515]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155011 entries, 0 to 155010
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   timestamp            155011 non-null  datetime64[ns]
 1   libre_gl             155011 non-null  float64       
 2   dexcom_gl            155011 non-null  float64       
 3   hr                   155011 non-null  float64       
 4   calories_(activity)  155011 non-null  float64       
 5   mets                 155011 non-null  float64       
 6   meal_type            1290 non-null    object        
 7   calories             1290 non-null    float64       
 8   carbs                1290 non-null    float64       
 9   protein              1290 non-null    float64       
 10  fat                  1290 non-null    float64       
 11  fiber                1290 non-null    float64       
 12  amount_consumed      1290 non-null    float64       
 13  image_path    

In [516]:
glucose_both_df = df_1.copy()

In [517]:
glucose_libre_only_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159579 entries, 0 to 159578
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   timestamp            159579 non-null  datetime64[ns]
 1   libre_gl             159579 non-null  float64       
 2   hr                   150516 non-null  float64       
 3   calories_(activity)  157949 non-null  float64       
 4   mets                 122473 non-null  float64       
 5   meal_type            1329 non-null    object        
 6   calories             1329 non-null    float64       
 7   carbs                1329 non-null    float64       
 8   protein              1329 non-null    float64       
 9   fat                  1329 non-null    float64       
 10  fiber                1329 non-null    float64       
 11  amount_consumed      1329 non-null    float64       
 12  image_path           1282 non-null    object        
 13  patient       

In [518]:
df_2 = glucose_libre_only_df.copy()
df_2 = df_2.sort_values(by=['patient', 'timestamp'])

# Lista de variables a imputar dentro de ventanas post comida
variables_a_imputar = ['calories_(activity)', 'hr', 'mets']

# Imputación dentro de las ventanas de 2 horas post comida
for var in variables_a_imputar:
    for _, meal_row in df_2[df_2['meal_type'].notna()].iterrows():
        patient_id = meal_row['patient']
        start_time = meal_row['timestamp']
        end_time = start_time + pd.Timedelta(hours=2)

        mask = (
            (df_2['patient'] == patient_id) &
            (df_2['timestamp'] >= start_time) &
            (df_2['timestamp'] <= end_time)
        )

        # Interpolación + ffill + bfill
        temp = df_2.loc[mask, var].copy()
        temp = temp.interpolate(method='linear', limit_direction='both')
        temp = temp.fillna(method='ffill').fillna(method='bfill')
        df_2.loc[mask, var] = temp

    # Imputación global: promedio por paciente
    df_2[var] = df_2[var].fillna(
        df_2.groupby('patient')[var].transform('mean')
    )

    # Si aún quedan nulos: promedio general
    df_2[var] = df_2[var].fillna(df_2[var].mean())



In [519]:
df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159579 entries, 0 to 159578
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   timestamp            159579 non-null  datetime64[ns]
 1   libre_gl             159579 non-null  float64       
 2   hr                   159579 non-null  float64       
 3   calories_(activity)  159579 non-null  float64       
 4   mets                 159579 non-null  float64       
 5   meal_type            1329 non-null    object        
 6   calories             1329 non-null    float64       
 7   carbs                1329 non-null    float64       
 8   protein              1329 non-null    float64       
 9   fat                  1329 non-null    float64       
 10  fiber                1329 non-null    float64       
 11  amount_consumed      1329 non-null    float64       
 12  image_path           1282 non-null    object        
 13  patient       

In [520]:
glucose_libre_only_df = df_2.copy()

Con todos los datos completos, podemos agregar promedios al dataset que se utiliza para los cálculos y modelos.

In [521]:
# Lista donde se guardarán las métricas por evento
results = []

# Identificar los inicios de evento de comida
meal_starts = glucose_both_df[glucose_both_df['meal_type'].notna()]

# Iterar por cada evento
for _, meal_row in meal_starts.iterrows():
    patient_id = meal_row['patient']
    start_time = meal_row['timestamp']
    end_time = start_time + pd.Timedelta(hours=2)

    # Subset del evento actual
    event_data = glucose_both_df[
        (glucose_both_df['patient'] == patient_id) &
        (glucose_both_df['timestamp'] >= start_time) &
        (glucose_both_df['timestamp'] <= end_time)
    ]

    result = {
        'patient': patient_id,
        'meal_time': start_time,
        'meal_type': meal_row['meal_type'],
        'calories': meal_row['calories'],
        'carbs': meal_row['carbs'],
        'protein': meal_row['protein'],
        'fat': meal_row['fat'],
        'fiber': meal_row['fiber'],
        'amount_consumed': meal_row['amount_consumed']
    }

    # Libre GL
    libre_data = event_data[['timestamp', 'libre_gl']].copy()
    times = (libre_data['timestamp'] - start_time).dt.total_seconds() / 60
    values = libre_data['libre_gl'].values
    baseline = meal_row['libre_gl']
    above_baseline = np.maximum(0, values - baseline)

    result['iauc_libre'] = np.trapz(above_baseline, times)
    result['peak_value_libre'] = values.max()
    result['delta_libre'] = values.max() - baseline
    peak_time = libre_data[libre_data['libre_gl'] == values.max()]['timestamp'].iloc[0]
    result['time_to_peak_libre'] = (peak_time - start_time).total_seconds() / 60
    result['variability_libre'] = libre_data['libre_gl'].std()

    # Early iAUC (0–30 min)
    early_mask = (libre_data['timestamp'] - start_time).dt.total_seconds() <= 1800
    libre_early = libre_data[early_mask]
    if len(libre_early) > 1:
        times_early = (libre_early['timestamp'] - start_time).dt.total_seconds() / 60
        values_early = libre_early['libre_gl'].values
        above_baseline_early = np.maximum(0, values_early - baseline)
        result['early_iauc_libre'] = np.trapz(above_baseline_early, times_early)

    # Dexcom GL
    dexcom_data = event_data[['timestamp', 'dexcom_gl']].copy()
    times = (dexcom_data['timestamp'] - start_time).dt.total_seconds() / 60
    values = dexcom_data['dexcom_gl'].values
    baseline = meal_row['dexcom_gl']
    above_baseline = np.maximum(0, values - baseline)

    result['iauc_dexcom'] = np.trapz(above_baseline, times)
    result['peak_value_dexcom'] = values.max()
    result['delta_dexcom'] = values.max() - baseline
    peak_time = dexcom_data[dexcom_data['dexcom_gl'] == values.max()]['timestamp'].iloc[0]
    result['time_to_peak_dexcom'] = (peak_time - start_time).total_seconds() / 60
    result['variability_dexcom'] = dexcom_data['dexcom_gl'].std()

    # Early iAUC (0–30 min)
    early_mask = (dexcom_data['timestamp'] - start_time).dt.total_seconds() <= 1800
    dexcom_early = dexcom_data[early_mask]
    if len(dexcom_early) > 1:
        times_early = (dexcom_early['timestamp'] - start_time).dt.total_seconds() / 60
        values_early = dexcom_early['dexcom_gl'].values
        above_baseline_early = np.maximum(0, values_early - baseline)
        result['early_iauc_dexcom'] = np.trapz(above_baseline_early, times_early)


    # Promedios durante las 2 horas postprandiales
    result['mean_calories_activity'] = event_data['calories_(activity)'].mean()
    result['mean_hr'] = event_data['hr'].mean()
    result['mean_mets'] = event_data['mets'].mean()


    # Guardar resultado
    results.append(result)

# Convertir a DataFrame final
metrics_both_df = pd.DataFrame(results)

In [522]:
metrics_both_df

,patient,meal_time,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,iauc_libre,peak_value_libre,delta_libre,time_to_peak_libre,variability_libre,early_iauc_libre,iauc_dexcom,peak_value_dexcom,delta_dexcom,time_to_peak_dexcom,variability_dexcom,early_iauc_dexcom,mean_calories_activity,mean_hr,mean_mets
0,1,2020-05-01 14:23:00,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,1526.600000,93.0,23.200000,97.0,8.387954,17.533333,783.6,134.0,24.6,101.0,6.988962,145.6,2.709378,84.504132,25.842975
1,1,2020-05-01 20:48:00,dinner,80.0,18.0,0.0,0.0,0.0,100.0,1626.600000,114.0,29.200000,42.0,8.192057,393.600000,1422.9,154.0,39.2,36.0,10.717140,184.1,1.243350,64.793388,11.859504
2,1,2020-05-02 00:15:00,snack,110.0,24.0,0.0,2.0,0.0,100.0,1650.000000,113.0,32.000000,45.0,8.930279,270.000000,1861.6,133.0,35.6,39.0,9.151036,282.6,2.214637,82.851240,21.123967
3,1,2020-05-02 08:18:00,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,1388.400000,118.0,29.600000,42.0,11.223976,345.600000,1273.4,142.0,40.2,36.0,15.503717,136.3,3.231845,90.578512,30.826446
4,1,2020-05-02 12:00:00,lunch,840.0,89.0,17.0,42.0,3.0,100.0,1039.700000,103.0,21.000000,30.0,8.422984,247.500000,1902.7,135.0,55.0,44.0,27.186248,522.5,1.564802,78.239669,14.925620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1285,49,2025-05-19 07:06:00,breakfast,712.0,66.0,22.0,42.0,0.0,100.0,13439.366667,295.0,162.400000,77.0,50.646609,1018.433333,12835.7,331.0,166.2,77.0,60.219531,461.0,3.217988,98.768595,34.553719
1286,49,2025-05-19 13:37:00,lunch,445.0,43.0,20.0,20.0,13.0,100.0,5523.300000,149.0,66.066667,91.0,23.106481,298.100000,6276.0,187.0,92.2,106.0,33.690988,170.1,1.665565,83.603306,17.884298
1287,49,2025-05-19 20:21:00,dinner,370.0,38.0,26.0,12.0,2.0,100.0,8834.166667,192.0,103.533333,62.0,32.078574,708.700000,10814.2,236.0,130.0,72.0,46.664682,447.7,1.276112,72.528926,13.702479
1288,49,2025-05-20 07:06:00,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,5935.500000,229.0,90.666667,47.0,26.563904,792.533333,6138.3,255.0,102.4,47.0,30.363261,539.4,3.599744,93.925620,38.652893


In [523]:
metrics_both_df.isnull().any(axis=1).sum()

0

In [524]:
metrics_both_df[(metrics_both_df['iauc_libre'] == 0) & (metrics_both_df['iauc_dexcom'] == 0)]

,patient,meal_time,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,iauc_libre,peak_value_libre,delta_libre,time_to_peak_libre,variability_libre,early_iauc_libre,iauc_dexcom,peak_value_dexcom,delta_dexcom,time_to_peak_dexcom,variability_dexcom,early_iauc_dexcom,mean_calories_activity,mean_hr,mean_mets
127,5,2020-08-20 20:22:00,dinner,418.0,48.0,29.0,12.0,6.0,100.0,0.0,130.000000,0.0,0.0,15.937914,0.0,0.0,158.6,0.0,0.0,18.993165,0.0,1.250090,89.066116,13.347107
255,10,2020-06-22 17:36:00,dinner,401.0,32.0,24.0,7.0,2.0,100.0,0.0,129.000000,0.0,0.0,10.193030,0.0,0.0,170.0,0.0,0.0,13.771235,0.0,2.157227,87.942149,21.619835
344,12,2023-03-06 21:24:00,dinner,138.0,60.0,3.0,7.0,0.0,100.0,0.0,231.666667,0.0,0.0,9.800825,0.0,0.0,180.6,0.0,0.0,3.918597,0.0,1.230500,83.215161,10.000000
417,15,2024-01-28 15:32:00,lunch,555.0,94.0,12.0,13.0,5.0,100.0,0.0,76.333333,0.0,0.0,6.413054,0.0,0.0,143.6,0.0,0.0,6.825198,0.0,0.856972,83.421488,10.636364
429,15,2024-02-02 12:51:00,lunch,445.0,43.0,20.0,20.0,13.0,100.0,0.0,74.000000,0.0,0.0,9.443122,0.0,0.0,162.0,0.0,0.0,12.922641,0.0,0.947838,76.018868,11.764151
477,17,2023-10-16 02:25:00,snack,8.0,2.0,0.0,0.0,0.0,100.0,0.0,99.533333,0.0,0.0,11.143173,0.0,0.0,165.8,0.0,0.0,12.814975,0.0,1.045969,57.024793,11.066116
523,19,2020-10-01 13:04:00,lunch,435.0,16.0,66.0,14.0,4.0,90.0,0.0,84.466667,0.0,0.0,1.747142,0.0,0.0,132.0,0.0,0.0,4.677730,0.0,2.000094,86.644628,18.694215
563,20,2024-03-25 20:48:00,snack,320.0,16.0,10.0,26.0,2.0,90.0,0.0,139.200000,0.0,0.0,8.669124,0.0,0.0,185.0,0.0,0.0,7.272757,0.0,1.219708,93.338843,11.842975
785,30,2021-01-23 11:24:00,lunch,1180.0,81.0,44.0,54.5,17.5,90.0,0.0,192.200000,0.0,0.0,32.415233,0.0,0.0,174.0,0.0,0.0,8.224922,0.0,1.401544,86.702479,15.033058
790,30,2021-01-25 11:44:00,lunch,435.0,16.0,66.0,14.0,4.0,75.0,0.0,238.000000,0.0,0.0,44.864804,0.0,0.0,163.0,0.0,0.0,17.576495,0.0,1.098888,79.793388,11.702479


De esta forma obtenemos `metrics_both_df`, el conjunto de datos que contiene la información de las curvas de glucosa de ambos sensores de glucosa en los sujetos del estudio. Este conjunto de datos no contiene valores nulos, y lleva consigo la información completa de los eventos de comida.

Ahora se obtiene el dataframe con medidas de la iAUC, pero solamente con la información de `libre_gl`.

In [525]:
# Lista donde se guardarán las métricas por evento
results = []

# Identificar los inicios de evento de comida
meal_starts = glucose_libre_only_df[glucose_libre_only_df['meal_type'].notna()]

# Iterar por cada evento
for _, meal_row in meal_starts.iterrows():
    patient_id = meal_row['patient']
    start_time = meal_row['timestamp']
    end_time = start_time + pd.Timedelta(hours=2)

    # Subset del evento actual
    event_data = glucose_libre_only_df[
        (glucose_libre_only_df['patient'] == patient_id) &
        (glucose_libre_only_df['timestamp'] >= start_time) &
        (glucose_libre_only_df['timestamp'] <= end_time)
    ]

    result = {
        'patient': patient_id,
        'meal_time': start_time,
        'meal_type': meal_row['meal_type'],
        'calories': meal_row['calories'],
        'carbs': meal_row['carbs'],
        'protein': meal_row['protein'],
        'fat': meal_row['fat'],
        'fiber': meal_row['fiber'],
        'amount_consumed': meal_row['amount_consumed']
    }

    # Libre GL
    libre_data = event_data[['timestamp', 'libre_gl']].copy()
    times = (libre_data['timestamp'] - start_time).dt.total_seconds() / 60
    values = libre_data['libre_gl'].values
    baseline = meal_row['libre_gl']
    above_baseline = np.maximum(0, values - baseline)

    result['iauc_libre'] = np.trapz(above_baseline, times)
    result['peak_value_libre'] = values.max()
    result['delta_libre'] = values.max() - baseline
    peak_time = libre_data[libre_data['libre_gl'] == values.max()]['timestamp'].iloc[0]
    result['time_to_peak_libre'] = (peak_time - start_time).total_seconds() / 60
    result['variability_libre'] = libre_data['libre_gl'].std()

    # Early iAUC (0–30 min)
    early_mask = (libre_data['timestamp'] - start_time).dt.total_seconds() <= 1800
    libre_early = libre_data[early_mask]
    if len(libre_early) > 1:
        times_early = (libre_early['timestamp'] - start_time).dt.total_seconds() / 60
        values_early = libre_early['libre_gl'].values
        above_baseline_early = np.maximum(0, values_early - baseline)
        result['early_iauc_libre'] = np.trapz(above_baseline_early, times_early)

    # Promedios durante las 2 horas postprandiales
    result['mean_calories_activity'] = event_data['calories_(activity)'].mean()
    result['mean_hr'] = event_data['hr'].mean()
    result['mean_mets'] = event_data['mets'].mean()

    # Guardar resultado
    results.append(result)

# Convertir a DataFrame final
metrics_libre_df = pd.DataFrame(results)

In [526]:
metrics_libre_df

,patient,meal_time,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,iauc_libre,peak_value_libre,delta_libre,time_to_peak_libre,variability_libre,early_iauc_libre,mean_calories_activity,mean_hr,mean_mets
0,1,2020-05-01 14:23:00,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,1526.600000,93.0,23.200000,97.0,8.387954,17.533333,2.709378,84.504132,25.842975
1,1,2020-05-01 20:48:00,dinner,80.0,18.0,0.0,0.0,0.0,100.0,1626.600000,114.0,29.200000,42.0,8.192057,393.600000,1.243350,64.793388,11.859504
2,1,2020-05-02 00:15:00,snack,110.0,24.0,0.0,2.0,0.0,100.0,1650.000000,113.0,32.000000,45.0,8.930279,270.000000,2.214637,82.851240,21.123967
3,1,2020-05-02 08:18:00,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,1388.400000,118.0,29.600000,42.0,11.223976,345.600000,3.231845,90.578512,30.826446
4,1,2020-05-02 12:00:00,lunch,840.0,89.0,17.0,42.0,3.0,100.0,1039.700000,103.0,21.000000,30.0,8.422984,247.500000,1.564802,78.239669,14.925620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1324,49,2025-05-19 13:37:00,lunch,445.0,43.0,20.0,20.0,13.0,100.0,5523.300000,149.0,66.066667,91.0,23.106481,298.100000,1.665565,83.603306,17.884298
1325,49,2025-05-19 20:21:00,dinner,370.0,38.0,26.0,12.0,2.0,100.0,8834.166667,192.0,103.533333,62.0,32.078574,708.700000,1.276112,72.528926,13.702479
1326,49,2025-05-20 07:06:00,breakfast,268.0,24.0,22.0,10.5,0.0,100.0,5935.500000,229.0,90.666667,47.0,26.563904,792.533333,3.599744,93.925620,38.652893
1327,49,2025-05-20 12:53:00,lunch,725.0,94.0,44.0,20.0,4.0,100.0,9735.000000,230.0,146.000000,120.0,50.570916,412.500000,1.380017,78.578512,14.818182


In [527]:
metrics_libre_df.isnull().any(axis=1).sum()

0

Finalmente tenemos dos conjuntos de datos que corresponden de la siguiente manera:
* `metrics_both_df`: Contiene medidas relacionadas al iAUC de los sensores `dexcom_gl` y `libre_gl`.
* `metrics_libre_df`: Contiene medidas relacionadas al iAUC del sensor `libre_gl`.

# **12. Merge de los conjuntos de datos para modelos** <a class="anchor" id="12"></a>


[Tabla de Contenidos](#0.1)

## `metrics_both_df`

In [528]:
# Merge sucesivo
metrics_both_df = metrics_both_df.merge(bio, left_on='patient', right_on='subject', how='left')
metrics_both_df = metrics_both_df.merge(gut_scores_imputed, on='subject', how='left')
metrics_both_df = metrics_both_df.merge(microbes_pca_df, on='subject', how='left')

# Eliminar columna 'subject' ya que no es necesaria
metrics_both_df.drop(columns='subject', inplace=True)

print("Dimensiones del DataFrame final:", metrics_both_df.shape)
metrics_both_df.head()

Dimensiones del DataFrame final: (1290, 72)


,patient,meal_time,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,iauc_libre,peak_value_libre,delta_libre,time_to_peak_libre,variability_libre,early_iauc_libre,iauc_dexcom,peak_value_dexcom,delta_dexcom,time_to_peak_dexcom,variability_dexcom,early_iauc_dexcom,mean_calories_activity,mean_hr,mean_mets,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2,gut_lining_health,lps_biosynthesis_pathways,biofilm_chemotaxis_and_virulence_pathways,tma_production_pathways,ammonia_production_pathways,metabolic_fitness,active_microbial_diversity,butyrate_production_pathways,flagellar_assembly_pathways,putrescine_production_pathways,uric_acid_production_pathways,bile_acid_metabolism_pathways,inflammatory_activity,gut_microbiome_health,digestive_efficiency,protein_fermentation,gas_production,methane_gas_production_pathways,sulfide_gas_production_pathways,oxalate_metabolism_pathways,salt_stress_pathways,microbiome_induced_stress,microbe_PC1,microbe_PC2,microbe_PC3
0,1,2020-05-01 14:23:00,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,1526.6,93.0,23.2,97.0,8.387954,17.533333,783.6,134.0,24.6,101.0,6.988962,145.6,2.709378,84.504132,25.842975,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
1,1,2020-05-01 20:48:00,dinner,80.0,18.0,0.0,0.0,0.0,100.0,1626.6,114.0,29.2,42.0,8.192057,393.600000,1422.9,154.0,39.2,36.0,10.717140,184.1,1.243350,64.793388,11.859504,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
2,1,2020-05-02 00:15:00,snack,110.0,24.0,0.0,2.0,0.0,100.0,1650.0,113.0,32.0,45.0,8.930279,270.000000,1861.6,133.0,35.6,39.0,9.151036,282.6,2.214637,82.851240,21.123967,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
3,1,2020-05-02 08:18:00,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,1388.4,118.0,29.6,42.0,11.223976,345.600000,1273.4,142.0,40.2,36.0,15.503717,136.3,3.231845,90.578512,30.826446,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
4,1,2020-05-02 12:00:00,lunch,840.0,89.0,17.0,42.0,3.0,100.0,1039.7,103.0,21.0,30.0,8.422984,247.500000,1902.7,135.0,55.0,44.0,27.186248,522.5,1.564802,78.239669,14.925620,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317


In [529]:
metrics_both_df.columns

Index(['patient', 'meal_time', 'meal_type', 'calories', 'carbs', 'protein',
       'fat', 'fiber', 'amount_consumed', 'iauc_libre', 'peak_value_libre',
       'delta_libre', 'time_to_peak_libre', 'variability_libre',
       'early_iauc_libre', 'iauc_dexcom', 'peak_value_dexcom', 'delta_dexcom',
       'time_to_peak_dexcom', 'variability_dexcom', 'early_iauc_dexcom',
       'mean_calories_activity', 'mean_hr', 'mean_mets', 'age', 'gender',
       'bmi', 'body_weight', 'height', 'self_identify', 'a1c_pdl_(lab)',
       'fasting_glu_pdl_(lab)', 'insulin', 'triglycerides', 'cholesterol',
       'hdl', 'non_hdl', 'ldl_(cal)', 'vldl_(cal)', 'cho/hdl_ratio',
       'collection_time_pdl_(lab)', '#1_contour_fingerstick_glu', 'time_(t)',
       '#2_contour_fingerstick_glu', 'time_(t)_1',
       '#3_contour_fingerstick_glu', 'time_(t)_2', 'gut_lining_health',
       'lps_biosynthesis_pathways',
       'biofilm_chemotaxis_and_virulence_pathways', 'tma_production_pathways',
       'ammonia_producti

In [530]:
metrics_both_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290 entries, 0 to 1289
Data columns (total 72 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   patient                                    1290 non-null   int64         
 1   meal_time                                  1290 non-null   datetime64[ns]
 2   meal_type                                  1290 non-null   object        
 3   calories                                   1290 non-null   float64       
 4   carbs                                      1290 non-null   float64       
 5   protein                                    1290 non-null   float64       
 6   fat                                        1290 non-null   float64       
 7   fiber                                      1290 non-null   float64       
 8   amount_consumed                            1290 non-null   float64       
 9   iauc_libre         

De esta forma, tenemos el dataset que tiene toda la información de los eventos de comida, y la información de todos los sujetos de prueba contenida.

Se puede observar que `metrics_both_df` es un conjunto de datos con 1291 eventos de comida, sin un solo registro nulo.

In [531]:
# Filtrar registros con iAUC igual a 0
sospechosos = metrics_both_df[(metrics_both_df['iauc_dexcom'] == 0) | (metrics_both_df['iauc_libre'] == 0)]

# Seleccionar una muestra aleatoria de 5 registros
sample = sospechosos.sample(5, random_state=42)

# Visualización de cada caso
for _, row in sample.iterrows():
    paciente = row['patient']
    start_time = row['meal_time']
    sensor = 'dexcom_gl' if row['iauc_dexcom'] == 0 else 'libre_gl'
    color = '#55A868' if sensor == 'dexcom_gl' else '#C44E52'

    # Subset de datos para el paciente y rango temporal
    mask = (
        (df['patient'] == paciente) &
        (df['timestamp'] >= start_time - pd.Timedelta(minutes=10)) &
        (df['timestamp'] <= start_time + pd.Timedelta(hours=2))
    )
    sub = df[mask]

    # Gráfico
    plt.figure(figsize=(10, 4))
    plt.plot(sub['timestamp'], sub[sensor], marker='o', color=color)
    plt.axvline(start_time, color='gray', linestyle='--', alpha=0.7, label='Inicio de comida')
    plt.title(f'Paciente {int(paciente):02d} - iAUC = 0 ({sensor})', fontsize=13)
    plt.xlabel("Tiempo", fontsize=11)
    plt.ylabel("Glucosa (mg/dL)", fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()


<Figure size 1000x400 with 1 Axes>

<Figure size 1000x400 with 1 Axes>

<Figure size 1000x400 with 1 Axes>

<Figure size 1000x400 with 1 Axes>

<Figure size 1000x400 with 1 Axes>

## `metrics_libre_df`

Se hace el mismo procedimiento que en el paso anterior, pero esta vez para el dataset que contiene solamente la información de `libre_gl`.

In [532]:
# Merge sucesivo
metrics_libre_df = metrics_libre_df.merge(bio, left_on='patient', right_on='subject', how='left')
metrics_libre_df = metrics_libre_df.merge(gut_scores_imputed, on='subject', how='left')
metrics_libre_df = metrics_libre_df.merge(microbes_pca_df, on='subject', how='left')

# Eliminar columna 'subject' ya que no es necesaria
metrics_libre_df.drop(columns='subject', inplace=True)

print("Dimensiones del DataFrame final:", metrics_libre_df.shape)
metrics_libre_df.head()

Dimensiones del DataFrame final: (1329, 66)


,patient,meal_time,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,iauc_libre,peak_value_libre,delta_libre,time_to_peak_libre,variability_libre,early_iauc_libre,mean_calories_activity,mean_hr,mean_mets,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2,gut_lining_health,lps_biosynthesis_pathways,biofilm_chemotaxis_and_virulence_pathways,tma_production_pathways,ammonia_production_pathways,metabolic_fitness,active_microbial_diversity,butyrate_production_pathways,flagellar_assembly_pathways,putrescine_production_pathways,uric_acid_production_pathways,bile_acid_metabolism_pathways,inflammatory_activity,gut_microbiome_health,digestive_efficiency,protein_fermentation,gas_production,methane_gas_production_pathways,sulfide_gas_production_pathways,oxalate_metabolism_pathways,salt_stress_pathways,microbiome_induced_stress,microbe_PC1,microbe_PC2,microbe_PC3
0,1,2020-05-01 14:23:00,lunch,1170.0,85.0,44.0,54.2,12.0,100.0,1526.6,93.0,23.2,97.0,8.387954,17.533333,2.709378,84.504132,25.842975,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
1,1,2020-05-01 20:48:00,dinner,80.0,18.0,0.0,0.0,0.0,100.0,1626.6,114.0,29.2,42.0,8.192057,393.600000,1.243350,64.793388,11.859504,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
2,1,2020-05-02 00:15:00,snack,110.0,24.0,0.0,2.0,0.0,100.0,1650.0,113.0,32.0,45.0,8.930279,270.000000,2.214637,82.851240,21.123967,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
3,1,2020-05-02 08:18:00,breakfast,448.0,66.0,22.0,10.5,0.0,100.0,1388.4,118.0,29.6,42.0,11.223976,345.600000,3.231845,90.578512,30.826446,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
4,1,2020-05-02 12:00:00,lunch,840.0,89.0,17.0,42.0,3.0,100.0,1039.7,103.0,21.0,30.0,8.422984,247.500000,1.564802,78.239669,14.925620,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317


In [533]:
metrics_libre_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1329 entries, 0 to 1328
Data columns (total 66 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   patient                                    1329 non-null   int64         
 1   meal_time                                  1329 non-null   datetime64[ns]
 2   meal_type                                  1329 non-null   object        
 3   calories                                   1329 non-null   float64       
 4   carbs                                      1329 non-null   float64       
 5   protein                                    1329 non-null   float64       
 6   fat                                        1329 non-null   float64       
 7   fiber                                      1329 non-null   float64       
 8   amount_consumed                            1329 non-null   float64       
 9   iauc_libre         

In [534]:
metrics_libre_df[metrics_libre_df['iauc_libre'] == 0]

,patient,meal_time,meal_type,calories,carbs,protein,fat,fiber,amount_consumed,iauc_libre,peak_value_libre,delta_libre,time_to_peak_libre,variability_libre,early_iauc_libre,mean_calories_activity,mean_hr,mean_mets,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2,gut_lining_health,lps_biosynthesis_pathways,biofilm_chemotaxis_and_virulence_pathways,tma_production_pathways,ammonia_production_pathways,metabolic_fitness,active_microbial_diversity,butyrate_production_pathways,flagellar_assembly_pathways,putrescine_production_pathways,uric_acid_production_pathways,bile_acid_metabolism_pathways,inflammatory_activity,gut_microbiome_health,digestive_efficiency,protein_fermentation,gas_production,methane_gas_production_pathways,sulfide_gas_production_pathways,oxalate_metabolism_pathways,salt_stress_pathways,microbiome_induced_stress,microbe_PC1,microbe_PC2,microbe_PC3
20,1,2020-05-06 20:58:00,dinner,193.0,32.0,5.0,4.0,3.0,100.0,0.0,87.200000,0.0,0.0,3.052724,0.0,1.865459,72.545455,17.793388,27,M,22.265239,133.8,65.00,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,2,1,2,1,3,2,3,2,2,2,3,2,3,1,1,2,3,1,1,3,2,12.073691,-3.625122,10.078317
120,5,2020-08-17 13:08:00,lunch,1180.0,81.0,44.0,54.5,17.5,100.0,0.0,87.800000,0.0,0.0,7.446329,0.0,1.570547,94.462810,16.768595,51,F,30.957534,172.0,62.50,Hispanic/Latino,6.6,144,12.9,392.0,269,38,231,157,78,7.1,2025-06-01 07:45:00,139,2025-06-01 08:59:00,215,2025-06-01 10:52:00,130,2025-06-01 11:54:00,1,1,2,3,1,2,1,2,2,1,1,2,1,1,2,2,3,3,2,2,2,2,7.241053,-6.608969,2.575672
130,5,2020-08-20 20:22:00,dinner,418.0,48.0,29.0,12.0,6.0,100.0,0.0,130.000000,0.0,0.0,15.937914,0.0,1.250090,89.066116,13.347107,51,F,30.957534,172.0,62.50,Hispanic/Latino,6.6,144,12.9,392.0,269,38,231,157,78,7.1,2025-06-01 07:45:00,139,2025-06-01 08:59:00,215,2025-06-01 10:52:00,130,2025-06-01 11:54:00,1,1,2,3,1,2,1,2,2,1,1,2,1,1,2,2,3,3,2,2,2,2,7.241053,-6.608969,2.575672
169,6,2023-04-14 09:08:00,breakfast,608.0,66.0,66.0,10.5,0.0,100.0,0.0,97.666667,0.0,0.0,1.317278,0.0,1.765103,75.214876,16.272727,51,F,29.303451,197.0,68.75,White,5.2,96,6.4,75.0,203,72,131,118,15,2.8,2025-06-01 07:45:00,98,2025-06-01 09:04:00,97,2025-06-01 10:54:00,70,2025-06-01 11:56:00,1,1,1,3,2,2,2,2,2,2,2,1,1,1,2,2,1,1,1,1,2,2,8.087649,-3.566292,-3.655229
173,6,2023-04-15 12:52:00,lunch,445.0,43.0,20.0,20.0,13.0,100.0,0.0,119.800000,0.0,0.0,13.701188,0.0,1.239785,70.057851,11.429752,51,F,29.303451,197.0,68.75,White,5.2,96,6.4,75.0,203,72,131,118,15,2.8,2025-06-01 07:45:00,98,2025-06-01 09:04:00,97,2025-06-01 10:54:00,70,2025-06-01 11:56:00,1,1,1,3,2,2,2,2,2,2,2,1,1,1,2,2,1,1,1,1,2,2,8.087649,-3.566292,-3.655229
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1245,47,2025-11-01 12:19:00,lunch,435.0,16.0,66.0,14.0,4.0,100.0,0.0,155.000000,0.0,0.0,10.373530,0.0,3.458181,79.206612,32.190083,62,M,31.377201,182.8,64.00,Hispanic/Latino,6.9,150,9.7,167.0,168,36,132,103,33,4.7,2025-06-01 07:15:00,157,2025-06-01 07:25:00,211,2025-06-01 08:31:00,148,2025-06-01 09:31:00,1,2,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,1,1,-6.694523,0.332532,0.252850
1251,47,2025-11-03 11:47:00,lunch,355.0,19.0,32.0,15.0,5.0,100.0,0.0,174.600000,0.0,0.0,19.612312,0.0,2.186778,82.876033,20.355372,62,M,31.377201,182.8,64.00,Hispanic/Latino,6.9,150,9.7,167.0,168,36,132,103,33,4.7,2025-06-01 07:15:00,157,2025-06-01 07:25:00,211,2025-06-01 08:31:00,148,2025-06-01 09:31:00,1,2,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,1,1,-6.694523,0.332532,0.252850
12

Se puede observar que este dataset contiene más registros que el anterior, pero también menos columnas. 

Igual que el anterior, no contiene valores nulos.

# **13. Feature Engineering** <a class="anchor" id="13"></a>


[Tabla de Contenidos](#0.1)

Se extraen las características de las variables de tiempo de los conjunto de datos `metrics_both_df` y `metrics_libre_df` con el fin de poder incluirlas en los datos de entrenamiento y prueba.

También se obtiene la cantidad de macronutrientes consumidos realmente en cada evento de comida.

In [535]:
metrics_both_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290 entries, 0 to 1289
Data columns (total 72 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   patient                                    1290 non-null   int64         
 1   meal_time                                  1290 non-null   datetime64[ns]
 2   meal_type                                  1290 non-null   object        
 3   calories                                   1290 non-null   float64       
 4   carbs                                      1290 non-null   float64       
 5   protein                                    1290 non-null   float64       
 6   fat                                        1290 non-null   float64       
 7   fiber                                      1290 non-null   float64       
 8   amount_consumed                            1290 non-null   float64       
 9   iauc_libre         

### `meal_time`

In [536]:
# Función para crear y reemplazar variable cíclica
def add_cyclical_features(df, col_name, max_val):
    df[f'{col_name}_sin'] = np.sin(2 * np.pi * df[col_name] / max_val)
    df[f'{col_name}_cos'] = np.cos(2 * np.pi * df[col_name] / max_val)
    return df.drop(columns=[col_name])

Para `metrics_both_df`

In [537]:
# Extracción de la hora y minuto del evento de comida
metrics_both_df['meal_time_hour'] = metrics_both_df['meal_time'].dt.hour

# Aplicar transformación a cada dataset y reasignar
metrics_both_df = add_cyclical_features(metrics_both_df, 'meal_time_hour', 24)

metrics_both_df.drop(columns='meal_time', inplace=True)

In [538]:
metrics_both_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290 entries, 0 to 1289
Data columns (total 73 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   patient                                    1290 non-null   int64         
 1   meal_type                                  1290 non-null   object        
 2   calories                                   1290 non-null   float64       
 3   carbs                                      1290 non-null   float64       
 4   protein                                    1290 non-null   float64       
 5   fat                                        1290 non-null   float64       
 6   fiber                                      1290 non-null   float64       
 7   amount_consumed                            1290 non-null   float64       
 8   iauc_libre                                 1290 non-null   float64       
 9   peak_value_libre   

Y lo mismo para `metrics_libre_df`

In [539]:
# Extracción de la hora y minuto del evento de comida
metrics_libre_df['meal_time_hour'] = metrics_libre_df['meal_time'].dt.hour

# Aplicar transformación a cada dataset y reasignar
metrics_libre_df = add_cyclical_features(metrics_libre_df, 'meal_time_hour', 24)

metrics_libre_df.drop(columns='meal_time', inplace=True)

In [540]:
metrics_libre_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1329 entries, 0 to 1328
Data columns (total 67 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   patient                                    1329 non-null   int64         
 1   meal_type                                  1329 non-null   object        
 2   calories                                   1329 non-null   float64       
 3   carbs                                      1329 non-null   float64       
 4   protein                                    1329 non-null   float64       
 5   fat                                        1329 non-null   float64       
 6   fiber                                      1329 non-null   float64       
 7   amount_consumed                            1329 non-null   float64       
 8   iauc_libre                                 1329 non-null   float64       
 9   peak_value_libre   

## Macronutrientes reales consumidos

Entre las variables encontradas en los conjuntos de datos, encontramos para cada registro de comida la variable `amount_consumed` que representa la cantidad de alimento consumida de un alimento registrado. Esto quiere decir que para saber con exactitud cuántos macronutrientes (`carbs`, `protein`, `fat`, `fiber`) y calorías (`calories`) fueron consumidos en cada registro, es necesario realizar el cálculo:

In [541]:
def real_nutrients(df, nutrient_col):
    """
    Calcula la cantidad real de nutrientes consumidos en cada evento de comida.
    """
    
    factor = df['amount_consumed'] / 100  # Convertir a proporción
    df[nutrient_col] = df[nutrient_col] * factor
    
    return df

In [542]:
metrics_both_df['carbs'].mean()

54.26690697674418

In [543]:
metrics_both_df = real_nutrients(metrics_both_df, 'calories')
metrics_both_df = real_nutrients(metrics_both_df, 'carbs')
metrics_both_df = real_nutrients(metrics_both_df, 'protein')
metrics_both_df = real_nutrients(metrics_both_df, 'fat')
metrics_both_df = real_nutrients(metrics_both_df, 'fiber')

In [544]:
metrics_both_df['carbs'].mean()

51.82389147286822

In [545]:
metrics_both_df.drop(columns=['amount_consumed'], inplace=True)

Ahora para `metrics_libre_df`

In [546]:
metrics_libre_df['carbs'].mean()

54.658623024830696

In [547]:
metrics_libre_df = real_nutrients(metrics_libre_df, 'calories')
metrics_libre_df = real_nutrients(metrics_libre_df, 'carbs')
metrics_libre_df = real_nutrients(metrics_libre_df, 'protein')
metrics_libre_df = real_nutrients(metrics_libre_df, 'fat')
metrics_libre_df = real_nutrients(metrics_libre_df, 'fiber')
metrics_libre_df['carbs'].mean()

52.24877351392025

In [548]:
metrics_libre_df.drop(columns=['amount_consumed'], inplace=True)

## Agrupación de columnas

Se agrupan indicadores microbianos en índices compuestos:

In [549]:
metrics_both_df['gut_health_index'] = (
    metrics_both_df['gut_lining_health'] +
    metrics_both_df['gut_microbiome_health'] +
    metrics_both_df['digestive_efficiency']
)

metrics_libre_df['gut_health_index'] = (
    metrics_libre_df['gut_lining_health'] +
    metrics_libre_df['gut_microbiome_health'] +
    metrics_libre_df['digestive_efficiency']
)

metrics_both_df['gas_production_index'] = (
    metrics_both_df['gas_production'] +
    metrics_both_df['methane_gas_production_pathways'] +
    metrics_both_df['sulfide_gas_production_pathways']
)

metrics_libre_df['gas_production_index'] = (
    metrics_libre_df['gas_production'] +
    metrics_libre_df['methane_gas_production_pathways'] +
    metrics_libre_df['sulfide_gas_production_pathways']
)

metrics_both_df.drop(columns=['gut_lining_health', 'gut_microbiome_health', 'digestive_efficiency',
                             'gas_production', 'methane_gas_production_pathways', 'sulfide_gas_production_pathways'], inplace=True)

metrics_libre_df.drop(columns=['gut_lining_health', 'gut_microbiome_health', 'digestive_efficiency',
                             'gas_production', 'methane_gas_production_pathways', 'sulfide_gas_production_pathways'], inplace=True)

## Feature engineering de variables derivadas

### delta_ratio = delta_libre / (delta_dexcom + ε)

Esta nueva feature calculará la proporción entre el aumento glucémico medido por Libre y Dexcom. Si hay una gran diferencia, puede indicar sensores que captan distinto tipo de respuesta postprandial o variabilidad por tipo de comida.

Algunos alimentos ricos en carbs generan picos más marcados en un sensor que en otro. Esta relación puede ayudar a diferenciar comidas más o menos glucémicas, sin decirlo explícitamente.

In [550]:
# Definir epsilon para evitar división por cero
epsilon = 1e-3

# Agregar delta_ratio a metrics_both_df
metrics_both_df['delta_ratio'] = metrics_both_df['delta_libre'] / (metrics_both_df['delta_dexcom'] + epsilon)

### variability_ratio = variability_libre / (variability_dexcom + ε)
Esta feature da la variación de glucosa postprandial relativa entre los dos sensores.

Carbohidratos simples producen más oscilaciones. Si la variabilidad es muy distinta entre sensores, puede indicar tipo de carbohidrato o presencia de grasa/fibra que modula la absorción.

In [551]:
# Definir epsilon para evitar división por cero
epsilon = 1e-3

# Agregar variability_ratio a metrics_both_df
metrics_both_df['variability_ratio'] = metrics_both_df['variability_libre'] / (metrics_both_df['variability_dexcom'] + epsilon)

### mean_variability = (variability_libre + variability_dexcom) / 2
Mide la oscilación promedio de glucosa, sin sesgo por sensor.

Mayor variabilidad suele estar asociada a comidas con alto índice glucémico o sin control nutricional. Es un buen indicador indirecto de carbs simples.

In [552]:
# Agregar mean_variability a metrics_both_df
metrics_both_df['mean_variability'] = (
    metrics_both_df['variability_libre'] + metrics_both_df['variability_dexcom']
) / 2

### post_meal_state = mean_hr * mean_mets
Mide el índice de activación fisiológica después de la comida (relación entre ritmo cardíaco y gasto energético).

Un estado de alta activación puede indicar que el cuerpo está metabolizando una comida más pesada, o también podría reflejar actividad física cercana que modula la absorción de carbs.

In [553]:
# Agregar post_meal_state a metrics_both_df
metrics_both_df['post_meal_state'] = metrics_both_df['mean_hr'] * metrics_both_df['mean_mets']

# Agregar post_meal_state a metrics_libre_df 
metrics_libre_df['post_meal_state'] = metrics_libre_df['mean_hr'] * metrics_libre_df['mean_mets']

# **14. EDA de los Conjunto de Datos Final** <a class="anchor" id="14"></a>


[Tabla de Contenidos](#0.1)

Anteriormente se realizó un análisis exploratorio de datos (EDA) con los conjuntos de datos individualizados, este EDA se centra en los nuevos datos creados para los eventos de comida, que incluye la información respecto a las curvas de glucosa, y también es mucho más en un análisis multivariado entre columnas que provienen de distintos conjuntos de datos.

## Análisis de la respusta glucémica postprandial

### `metrics_both_df`

In [554]:
metrics_both_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290 entries, 0 to 1289
Data columns (total 72 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   patient                                    1290 non-null   int64         
 1   meal_type                                  1290 non-null   object        
 2   calories                                   1290 non-null   float64       
 3   carbs                                      1290 non-null   float64       
 4   protein                                    1290 non-null   float64       
 5   fat                                        1290 non-null   float64       
 6   fiber                                      1290 non-null   float64       
 7   iauc_libre                                 1290 non-null   float64       
 8   peak_value_libre                           1290 non-null   float64       
 9   delta_libre        

In [555]:
print("Estadísticas - iAUC Libre:")
print(metrics_both_df['iauc_libre'].describe())
print("\nEstadísticas - iAUC Dexcom:")
print(metrics_both_df['iauc_dexcom'].describe())

Estadísticas - iAUC Libre:
count     1290.000000
mean      2876.150930
std       2861.938671
min          0.000000
25%        748.066667
50%       2025.750000
75%       4034.333333
max      15486.900000
Name: iauc_libre, dtype: float64

Estadísticas - iAUC Dexcom:
count     1290.000000
mean      3337.992248
std       3241.588937
min          0.000000
25%        956.175000
50%       2392.250000
75%       4761.650000
max      20765.000000
Name: iauc_dexcom, dtype: float64


Se puede apreciar en las mediciones de ambos conjuntos que tienen distribución muy asimétrica a la derecha, con alta dispersión y valores extremos. La mediana es mucho menor que la media, lo que confirma la presencia de outliers. 

In [556]:
plt.figure(figsize=(12, 5))

# Histograma iAUC Libre (color rojo claro)
plt.subplot(1, 2, 1)
sns.histplot(metrics_both_df['iauc_libre'], kde=True, bins=50, color='#C44E52')
plt.title('Distribución de iAUC (Libre GL)', fontsize=13)
plt.xlabel('iAUC Libre (mg·min/dL)', fontsize=11)
plt.ylabel('Frecuencia', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)

# Histograma iAUC Dexcom (color verde claro)
plt.subplot(1, 2, 2)
sns.histplot(metrics_both_df['iauc_dexcom'], kde=True, bins=50, color='#55A868')
plt.title('Distribución de iAUC (Dexcom GL)', fontsize=13)
plt.xlabel('iAUC Dexcom (mg·min/dL)', fontsize=11)
plt.ylabel('Frecuencia', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()



<Figure size 1200x500 with 2 Axes>

**iAUC Libre**:
* La mayoría de los valores están concentrados en rangos bajos, especialmente por debajo de 1000 mg·min/dL.
* Se observa una cola larga hacia la derecha, indicando una distribución fuertemente sesgada positivamente.
* Hay una alta frecuencia de valores cercanos a cero, lo cual puede incluir errores, no respuestas o comidas muy ligeras.

**iAUC Dexcom**:
* Presenta un patrón similar al de Libre, pero con una distribución ligeramente más extendida.
* También tiene una gran acumulación de valores bajos, con picos cerca de cero.
* Los valores se extienden hasta más de 7000 mg·min/dL, lo que podría indicar valores extremos o comidas muy altas en carga glucémica.



In [557]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=metrics_both_df[['iauc_libre', 'iauc_dexcom']],
    palette=['#C44E52', '#55A868']
)
plt.title('Boxplot de iAUC por sensor', fontsize=13)
plt.ylabel('iAUC (mg·min/dL)', fontsize=11)
plt.xticks([0, 1], ['Libre GL', 'Dexcom GL'], fontsize=10)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


<Figure size 800x500 with 1 Axes>

El análisis del boxplot revela que tanto `iauc_libre` como `iauc_dexcom` presentan una distribución altamente dispersa y asimétrica, con numerosas observaciones fuera del rango intercuartílico superior, lo que indica una gran cantidad de outliers extremos. Aunque la mediana de ambos sensores se sitúa en torno a los 850–880 mg·min/dL, el rango entre los cuartiles es amplio y sugiere una marcada variabilidad en la respuesta glucémica postprandial entre distintas comidas.

Ambos sensores siguen un patrón similar, pero iauc_dexcom tiende a alcanzar valores máximos más elevados que iauc_libre, lo que podría estar relacionado con diferencias en sensibilidad o frecuencia de muestreo. La fuerte asimetría positiva en ambas métricas confirma que la mayoría de los eventos generan respuestas moderadas, pero algunos pocos provocan aumentos glucémicos muy altos.

### Limpieza de outliers para la variable `iauc_libre` y la variable `iauc_dexcom`:

Anteriormente se realizó limpieza de outliers para otras variables del dataset, pero las variable que se tratan aquí no habían sido tratadas porque se han obtenido a partir de cálculos y cojunción de otros datos (la curva de glucosa por la respuesta glucémica postprandial).

In [558]:
def analisis_out_upperbound(df, col_name):
    Q1 = df[col_name].quantile(0.25)
    Q3 = df[col_name].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return df[df[col_name] > upper_bound][['carbs', 'calories']].describe()

In [559]:
analisis_out_upperbound(metrics_both_df, 'iauc_libre')

,carbs,calories
count,57.000000,57.000000
mean,68.680702,590.302632
std,18.985431,211.008774
min,0.000000,0.000000
25%,66.000000,448.000000
50%,66.000000,608.000000
75%,73.000000,712.000000
max,94.000000,1110.000000


In [560]:
analisis_out_upperbound(metrics_both_df, 'iauc_dexcom')

,carbs,calories
count,46.000000,46.000000
mean,66.506522,581.467391
std,18.132909,204.937602
min,0.000000,0.000000
25%,66.000000,448.000000
50%,66.000000,581.500000
75%,70.500000,712.000000
max,94.000000,1110.000000


Se puede observar que los valores cuya área incremental bajo la curva (iAUC) tienen valores de entre 118 y 14.4 g de carbohidratos, esto tiene sentido en los valores altos de carbohidratos, pero no lo tiene en valores bajos de carbohidratos, por lo que estos valores que son outliers para el área incremental bajo la curva serán eliminados e imputados.

Ahora se analizan los valores que son 0 y/o menos de 50 en iauc:

In [561]:
# Ver características de comidas con iAUC mayor al rango intercuartil
metrics_both_df[metrics_both_df['iauc_libre'] < 50][['carbs', 'calories']].describe()

,carbs,calories
count,93.000000,93.000000
mean,41.045269,464.526882
std,27.811222,267.630839
min,0.000000,0.000000
25%,19.000000,326.250000
50%,40.000000,435.000000
75%,60.000000,585.000000
max,109.000000,1378.000000


In [562]:
metrics_both_df[metrics_both_df['iauc_dexcom'] < 50][['carbs', 'calories']].describe()

,carbs,calories
count,90.000000,90.000000
mean,37.805111,428.716111
std,28.629977,269.428915
min,0.000000,0.000000
25%,17.100000,289.125000
50%,27.000000,396.000000
75%,52.750000,445.000000
max,146.500000,1289.000000


Algunos datos también tienen sentido en esta ocasión, aquellos en los que las comidas no tienen carbohidratos o calorías, se puede entender que no tengan área incremental bajo la curva, pero aquellos valores con 146g de carbohidratos, no pueden explicarse, también serán imputados.

Lo primero es detectar qué valores se imputarán, que en este caso serán:
* Aquellos valores de `iauc` que sean menor a 50, cuando `carbs` es mayor a 15g.
* Valores que salgan del límite superior del rango intercuartil para `iauc`.

In [563]:
# Detectar qué valores deben imputarse
def marcar_para_imputar(df, col_name):
    Q1 = df[col_name].quantile(0.25)
    Q3 = df[col_name].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR

    return ((df[col_name] < 50) & (df['carbs'] > 15)) | (df[col_name] > upper_bound)

Así detectamos estos valores y los marcamos

In [564]:
metrics_both_df['impute_libre'] = marcar_para_imputar(metrics_both_df, 'iauc_libre')

In [565]:
metrics_both_df[metrics_both_df['impute_libre'] == True]

,patient,meal_type,calories,carbs,protein,fat,fiber,iauc_libre,peak_value_libre,delta_libre,time_to_peak_libre,variability_libre,early_iauc_libre,iauc_dexcom,peak_value_dexcom,delta_dexcom,time_to_peak_dexcom,variability_dexcom,early_iauc_dexcom,mean_calories_activity,mean_hr,mean_mets,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2,lps_biosynthesis_pathways,biofilm_chemotaxis_and_virulence_pathways,tma_production_pathways,ammonia_production_pathways,metabolic_fitness,active_microbial_diversity,butyrate_production_pathways,flagellar_assembly_pathways,putrescine_production_pathways,uric_acid_production_pathways,bile_acid_metabolism_pathways,inflammatory_activity,protein_fermentation,oxalate_metabolism_pathways,salt_stress_pathways,microbiome_induced_stress,microbe_PC1,microbe_PC2,microbe_PC3,meal_time_hour_sin,meal_time_hour_cos,gut_health_index,gas_production_index,delta_ratio,variability_ratio,mean_variability,post_meal_state,impute_libre
20,1,dinner,193.00,32.0,5.0,4.00,3.00,0.000000,87.2,0.000000,0.0,3.052724,0.000000,325.6,119.0,20.6,86.0,8.399347,9.0,1.865459,72.545455,17.793388,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,1,2,1,3,2,3,2,2,2,3,2,1,1,3,2,12.073691,-3.625122,10.078317,-8.660254e-01,0.500000,6,6,0.000000,0.363404,5.726036,1290.829452,True
26,1,lunch,585.00,40.0,38.0,17.00,12.00,12.800000,83.0,1.133333,94.0,4.654002,0.000000,7.4,113.0,3.0,3.0,6.885745,7.4,3.425929,83.776860,32.677686,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,1,2,1,3,2,3,2,2,2,3,2,1,1,3,2,12.073691,-3.625122,10.078317,-2.588190e-01,-0.965926,6,6,0.377652,0.675791,5.769873,2737.633905,True
64,3,breakfast,712.00,66.0,22.0,42.00,0.00,9438.300000,261.0,138.133333,58.0,40.704149,1023.333333,8684.8,245.0,132.8,64.0,41.307059,530.4,2.317037,86.900826,26.190083,59,F,26.948690,157.0,64.0,Hispanic/Latino,6.5,118,17.4,154.0,190,74,116,90,31,2.6,2025-06-01 07:25:00,119,2025-06-01 07:38:00,166,2025-06-01 09:23:00,98,2025-06-01 10:23:00,1,1,2,1,2,2,2,2,2,2,2,2,1,1,3,2,12.298997,1.759080,2.869390,7.071068e-01,-0.707107,6,7,1.040153,0.985380,41.005604,2275.939827,True
65,3,lunch,333.00,56.4,7.2,7.80,3.00,10567.500000,216.0,140.000000,90.0,45.559016,570.000000,8578.0,214.0,131.0,101.0,47.230465,89.5,2.039928,84.586777,23.057851,59,F,26.948690,157.0,64.0,Hispanic/Latino,6.5,118,17.4,154.0,190,74,116,90,31,2.6,2025-06-01 07:25:00,119,2025-06-01 07:38:00,166,2025-06-01 09:23:00,98,2025-06-01 10:23:00,1,1,2,1,2,2,2,2,2,2,2,2,1,1,3,2,12.298997,1.759080,2.869390,-2.588190e-01,-0.965926,6,7,1.068694,0.964590,46.394740,1950.389318,True
117,5,lunch,1180.00,81.0,44.0,54.50,17.50,0.000000,87.8,0.000000,0.0,7.446329,0.000000,1.6,117.0,1.0,35.0,24.082340,0.0,1.570547,94.462810,16.768595,51,F,30.957534,172.0,62.5,Hispanic/Latino,6.6,144,12.9,392.0,269,38,231,157,78,7.1,2025-06-01 07:45:00,139,2025-06-01 08:59:00,215,2025-06-01 10:52:00,130,2025-06-01 11:54:00,1,2,3,1,2,1,2,2,1,1,2,1,2,2,2,2,7.241053,-6.608969,2.575672,-2.588190e-01,-0.965926,4,8,0.000000,0.309190,15.764335,1584.008606,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1270,49,breakfast,712.00,66.0,22.0,42.00,0.00,10687.500000,296.0,134.000000,90.0,42.461071,750.000000,12287.5,320.0,158.0,90.0,47.569860,987.5,3.288797,103.884298,35.314050,58,F,36.090876,184.8,60.0

Ahora, se utiliza un modelo de machine learning, específicamente el XGBRegressor, que nos permitirá obtener los valores correspondientes a los registros que están registrados erroneámente.

Esto se logra a partir de la información de ciertas variables que se utilizarán como predictoras para el caso de los carbohidratos, como son los demás macronutrientes, y los datos de salud de los sujetos del estudio.

In [566]:
features = ['carbs', 'calories', 'protein', 'fat', 'fiber', 'bmi', 'age', 'insulin', 'a1c_pdl_(lab)', 'triglycerides', 'non_hdl', 'cho/hdl_ratio', 'microbe_PC1']

# Para iauc_libre
train_libre = metrics_both_df[~metrics_both_df['impute_libre']] # Seleccionar solo las filas donde no se necesita imputar
X_libre = train_libre[features] # Solo las columnas que se usarán como características
y_libre = train_libre['iauc_libre'] # La variable objetivo en esas filas

y_log = np.log1p(y_libre) # Aplicamos transformación logarítmica a la variable objetivo para estabilizar la varianza y mantener valores positivos en las predicciones

model_libre = XGBRegressor(n_estimators=500, max_depth=10, random_state=42)
model_libre.fit(X_libre, y_log)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [567]:
# Datos a imputar
to_impute_libre = metrics_both_df[metrics_both_df['impute_libre']]
X_imp_libre = to_impute_libre[features]

# Predicción
preds_log = model_libre.predict(X_imp_libre)
preds_final = np.expm1(preds_log)

# Asignación
metrics_both_df.loc[to_impute_libre.index, 'iauc_libre'] = preds_final

Fueron imputados los valores que para el sensor `libre_gl` eran erróneos según lo definido.

Ahora se realiza el mismo procedimiento para `dexcom_gl`

In [568]:
# Aplicación para Dexcom
metrics_both_df['impute_dexcom'] = marcar_para_imputar(metrics_both_df, 'iauc_dexcom')

In [569]:
# Filtrar datos válidos para entrenamiento
train_dexcom = metrics_both_df[~metrics_both_df['impute_dexcom']]
X_dexcom = train_dexcom[features]
y_dexcom = train_dexcom['iauc_dexcom']

y_log = np.log1p(y_dexcom)

# Entrenamiento del modelo
model_dexcom = XGBRegressor(n_estimators=500, max_depth=10, random_state=42)
model_dexcom.fit(X_dexcom, y_log)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [570]:
# Filtrar datos a imputar
to_impute_dexcom = metrics_both_df[metrics_both_df['impute_dexcom']]
X_imp_dexcom = to_impute_dexcom[features]

# Predicción
preds_log = model_dexcom.predict(X_imp_dexcom)
preds_final = np.expm1(preds_log)

# Asignación
metrics_both_df.loc[to_impute_dexcom.index, 'iauc_dexcom'] = preds_final

In [571]:
def cap_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)
    return df


metrics_both_df = cap_outliers_iqr(metrics_both_df, 'iauc_libre')
metrics_both_df = cap_outliers_iqr(metrics_both_df, 'iauc_dexcom')


Ahora que los valores fueron reasignados, podemos verificar nuevamente la distribución de las variables de `iauc`

In [572]:
plt.figure(figsize=(12, 5))

# Histograma iAUC Libre
plt.subplot(1, 2, 1)
sns.histplot(metrics_both_df['iauc_libre'], kde=True, bins=50, color='#C44E52')
plt.title('Distribución de iAUC (Libre GL)', fontsize=13)
plt.xlabel('iAUC Libre (mg·min/dL)', fontsize=11)
plt.ylabel('Frecuencia', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)

# Histograma iAUC Dexcom
plt.subplot(1, 2, 2)
sns.histplot(metrics_both_df['iauc_dexcom'], kde=True, bins=50, color='#55A868')
plt.title('Distribución de iAUC (Dexcom GL)', fontsize=13)
plt.xlabel('iAUC Dexcom (mg·min/dL)', fontsize=11)
plt.ylabel('Frecuencia', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


<Figure size 1200x500 with 2 Axes>

In [573]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=metrics_both_df[['iauc_libre', 'iauc_dexcom']],
    palette=['#C44E52', '#55A868']
)

plt.title('Boxplot de iAUC por sensor', fontsize=13)
plt.ylabel('iAUC (mg·min/dL)', fontsize=11)
plt.xticks([0, 1], ['Libre GL', 'Dexcom GL'], fontsize=10)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


<Figure size 800x500 with 1 Axes>

El proceso que se realizó fue:
1. Detectar valores inválidos o erróneos.
2. Aplicar una transformación logarítmica a los datos válidos para el entrenamiento de un modelo XGBRegressor, para mejorar la estabilidad de la distribución y evitar valores negativos.
2. Reasignar los valores erróneos en el dataset.
3. Hacer capping a los outliers que se generaron.

Con esto obtenemos:
- Distribución sesgada positivamente, como es fisiológicamente normal, pero muchísimo menos que antes.
- Las colas largas hacia valores altos lucen razonables y sin outliers extremos marcados.
- No hay outliers

In [574]:
metrics_both_df[metrics_both_df['iauc_libre'] < 50][['carbs', 'calories']].describe()

,carbs,calories
count,14.000000,14.000000
mean,6.678571,150.335714
std,5.872777,137.937357
min,0.000000,0.000000
25%,0.500000,23.500000
50%,6.500000,122.500000
75%,12.000000,268.675000
max,14.700000,391.500000


In [575]:
metrics_both_df[metrics_both_df['iauc_dexcom'] < 50][['carbs', 'calories']].describe()

,carbs,calories
count,13.000000,13.000000
mean,6.953846,148.134615
std,6.166119,142.481893
min,0.000000,0.000000
25%,0.000000,0.000000
50%,7.000000,140.000000
75%,13.000000,261.000000
max,14.400000,391.500000


Las comidas con iAUC bajo ahora tienen pocos carbohidratos (lo que es fisiológicamente coherente).

Ya no hay comidas altas en CH o calorías con respuesta glucémica casi nula como ocurría antes.

### Análisis de las variables

In [576]:
# Definir columnas
macros = ['carbs', 'protein', 'fat', 'fiber']
metricas_libre = ['iauc_libre', 'delta_libre', 'peak_value_libre', 'time_to_peak_libre', 'variability_libre']
metricas_dexcom = ['iauc_dexcom', 'delta_dexcom', 'peak_value_dexcom', 'time_to_peak_dexcom', 'variability_dexcom']
otras_cols = ['bmi', 'age', 'insulin', 'a1c_pdl_(lab)', 'triglycerides', 'non_hdl',
              'mean_mets', 'mean_calories_activity', 'mean_hr', 'microbe_PC1']

# Calcular matrices de correlación
corr_libre = metrics_both_df[macros + metricas_libre].corr().loc[macros, metricas_libre]
corr_dexcom = metrics_both_df[macros + metricas_dexcom].corr().loc[macros, metricas_dexcom]
corr_otras = metrics_both_df[macros + otras_cols].corr().loc[macros, otras_cols]

# Mapa de calor para Libre
plt.figure(figsize=(10, 4))
sns.heatmap(corr_libre, annot=True, cmap=sns.light_palette("#C44E52", as_cmap=True), vmin=-1, vmax=1, fmt=".2f",
            annot_kws={"fontsize": 8})
plt.title('Correlación entre macronutrientes y métricas (Libre GL)', fontsize=13)
plt.xlabel('Métricas de glucosa', fontsize=11)
plt.ylabel('Macronutrientes', fontsize=11)
plt.tight_layout()
plt.show()

# Mapa de calor para Dexcom
plt.figure(figsize=(10, 4))
sns.heatmap(corr_dexcom, annot=True, cmap=sns.light_palette("#55A868", as_cmap=True), vmin=-1, vmax=1, fmt=".2f",
            annot_kws={"fontsize": 8})
plt.title('Correlación entre macronutrientes y métricas (Dexcom GL)', fontsize=13)
plt.xlabel('Métricas de glucosa', fontsize=11)
plt.ylabel('Macronutrientes', fontsize=11)
plt.tight_layout()
plt.show()

# Mapa de calor para otras métricas
plt.figure(figsize=(10, 4))
sns.heatmap(corr_otras, annot=True, cmap=sns.light_palette("#4C72B0", as_cmap=True), vmin=-1, vmax=1, fmt=".2f",
            annot_kws={"fontsize": 8})
plt.title('Correlación entre macronutrientes y otras métricas', fontsize=13)
plt.xlabel('Otras métricas', fontsize=11)
plt.ylabel('Macronutrientes', fontsize=11)
plt.tight_layout()
plt.show()



<Figure size 1000x400 with 2 Axes>

<Figure size 1000x400 with 2 Axes>

<Figure size 1000x400 with 2 Axes>

En ambos casos, se observa que los carbohidratos son el nutriente más consistentemente asociado con el aumento glucémico, especialmente con iauc, delta y variability, con coeficientes de correlación entre 0.20 y 0.25. Esto sugiere que a mayor cantidad de carbohidratos en una comida, mayor es la respuesta glucémica medida por ambas tecnologías.

Por el contrario, la fibra muestra una correlación negativa con todas las métricas, alcanzando hasta -0.23, lo que indica que su presencia podría estar asociada con una menor respuesta glucémica, posiblemente por su efecto modulador en la absorción de glucosa. 

En cambio, las correlaciones de proteína y grasa con las métricas son muy bajas o cercanas a cero, lo que indica un efecto mucho menos relevante o indirecto en la respuesta postprandial medida por los sensores. 

En resumen, los carbohidratos y la fibra son los principales determinantes nutricionales de las curvas glucémicas, mientras que grasa y proteína parecen tener un papel secundario en este análisis.

Por el lado de otras métricas interesantes, el ritmo cardíaco medio, tiene la mayor relación con los macronutrientes, aunque en verdad es baja.

In [577]:
plt.figure(figsize=(12, 10))

for i, macro in enumerate(macros):
    plt.subplot(2, 2, i + 1)
    
    # Scatter Libre GL
    sns.scatterplot(
        data=metrics_both_df,
        x=macro,
        y='iauc_libre',
        label='Libre GL',
        color='#C44E52',
        alpha=0.5
    )
    
    # Scatter Dexcom GL
    sns.scatterplot(
        data=metrics_both_df,
        x=macro,
        y='iauc_dexcom',
        label='Dexcom GL',
        color='#55A868',
        alpha=0.5
    )
    
    plt.title(f'Relación entre {macro.capitalize()} e iAUC', fontsize=12)
    plt.xlabel(f'{macro.capitalize()} (g)', fontsize=10)
    plt.ylabel('iAUC (mg·min/dL)', fontsize=10)
    plt.legend(fontsize=8)
    plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


<Figure size 1200x1000 with 4 Axes>

Estos gráficos de dispersión ilustran la relación entre la cantidad de macronutrientes consumidos y la respuesta glucémica medida como iAUC, tanto con el sensor Libre (azul) como Dexcom (naranja). 

Visualmente, se observa una tendencia ligeramente ascendente entre los carbohidratos y el iAUC, lo que respalda la correlación positiva detectada anteriormente: a mayor ingesta de carbohidratos, mayor suele ser la respuesta glucémica. Esto se visualiza en la acumulación de puntos en valores inferiores, y acumulación ligera en valores superiores para ambas variables.

En contraste, para proteína, grasa y fibra, no se aprecia una relación clara ni consistente con el iAUC. Aunque hay una ligera acumulación de puntos en rangos bajos y medios, los valores de iAUC se mantienen dispersos a lo largo de todo el espectro. Esto refuerza la conclusión de que los carbohidratos son el principal determinante nutricional de la respuesta glucémica, mientras que los otros macronutrientes tienen un efecto más atenuado o indirecto. Además, la distribución vertical de puntos con valores extremos sugiere la existencia de outliers glucémicos incluso en comidas con cantidades moderadas de macros, lo cual podría deberse a factores individuales o a la composición específica del alimento.

In [578]:
# Agrupar por paciente y calcular estadísticas de iAUC
iauc_stats_by_patient = metrics_both_df.groupby('patient')['iauc_libre'].agg(['mean', 'median', 'std', 'count']).reset_index()
iauc_stats_by_patient = iauc_stats_by_patient.sort_values('mean', ascending=False)

# Visualizar el promedio de iAUC por paciente
plt.figure(figsize=(12, 6))
sns.barplot(data=iauc_stats_by_patient, x='patient', y='mean', palette='Blues_d')
plt.xticks(rotation=90)
plt.title('iAUC Promedio por Paciente (Sensor Libre)')
plt.xlabel('Paciente')
plt.ylabel('iAUC promedio (mg·min/dL)')
plt.tight_layout()
plt.show()

<Figure size 1200x600 with 1 Axes>

Se puede observar que los participantes con mayor respuesta glucémica promedio son el número 49 y el número 33. Los que menos respuesta glucémica tuvieron son el 6 y 8.

In [579]:
# Agrupar por paciente y calcular la media de iAUC para ambos sensores
iauc_avg = metrics_both_df.groupby('patient')[['iauc_libre', 'iauc_dexcom']].mean().reset_index()

# Reorganizar datos a formato largo para graficar
iauc_avg_melted = iauc_avg.melt(id_vars='patient', var_name='Sensor', value_name='iAUC promedio')

# Renombrar sensores para mejorar visualización
iauc_avg_melted['Sensor'] = iauc_avg_melted['Sensor'].map({
    'iauc_libre': 'Libre GL',
    'iauc_dexcom': 'Dexcom GL'
})

# Gráfico de barras
plt.figure(figsize=(12, 6))
sns.barplot(
    data=iauc_avg_melted,
    x='patient',
    y='iAUC promedio',
    hue='Sensor',
    palette=['#C44E52', '#55A868']
)
plt.title('iAUC Promedio por Paciente (Libre vs Dexcom)', fontsize=13)
plt.xlabel('Paciente', fontsize=11)
plt.ylabel('iAUC promedio (mg·min/dL)', fontsize=11)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(fontsize=9)
plt.legend(title='Sensor', fontsize=9, title_fontsize=10)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


<Figure size 1200x600 with 1 Axes>

El gráfico muestra el iAUC promedio por paciente para ambos sensores (Libre y Dexcom), permitiendo comparar la respuesta glucémica postprandial individual. Se observan diferencias notables entre sensores en varios casos, lo que podría deberse a discrepancias de calibración, ubicación del sensor o diferencias en sensibilidad. Algunos pacientes presentan valores más altos en Libre y otros en Dexcom, lo que resalta la importancia de analizar ambas fuentes para comprender mejor las respuestas individuales. En general, hay coherencia en las tendencias, pero la variabilidad interindividual es considerable.

In [580]:
metrics_both_df[metrics_both_df['patient']==49]['carbs'].describe()

count     27.000000
mean      57.268519
std       26.650419
min       13.000000
25%       39.000000
50%       63.000000
75%       67.500000
max      124.000000
Name: carbs, dtype: float64

In [581]:
metrics_both_df[metrics_both_df['patient'] == 15]['carbs'].describe()

count    27.000000
mean     53.629630
std      31.678855
min       0.000000
25%      24.000000
50%      66.000000
75%      77.500000
max      94.000000
Name: carbs, dtype: float64

In [582]:
plt.figure(figsize=(14, 6))

# Boxplot iAUC Libre
plt.subplot(1, 2, 1)
sns.boxplot(
    data=metrics_both_df,
    x='meal_type',
    y='iauc_libre',
    color='#C44E52'
)
plt.title('Distribución de iAUC (Libre GL) por Tipo de Comida', fontsize=13)
plt.ylabel('iAUC Libre (mg·min/dL)', fontsize=11)
plt.xlabel('Tipo de Comida', fontsize=11)
plt.xticks(rotation=45, fontsize=9)
plt.yticks(fontsize=9)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)

# Boxplot iAUC Dexcom
plt.subplot(1, 2, 2)
sns.boxplot(
    data=metrics_both_df,
    x='meal_type',
    y='iauc_dexcom',
    color='#55A868'
)
plt.title('Distribución de iAUC (Dexcom GL) por Tipo de Comida', fontsize=13)
plt.ylabel('iAUC Dexcom (mg·min/dL)', fontsize=11)
plt.xlabel('Tipo de Comida', fontsize=11)
plt.xticks(rotation=45, fontsize=9)
plt.yticks(fontsize=9)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


<Figure size 1400x600 with 2 Axes>

Existen diferencias claras en la respuesta glucémica según el momento del día. Tanto para el sensor Libre como para Dexcom, el desayuno ("breakfast") presenta los valores medianos y máximos de iAUC más elevados, lo que sugiere que las comidas matutinas generan mayores incrementos de glucosa postprandial. 

En contraste, los snacks tienden a mostrar valores más bajos y concentrados, reflejando respuestas glucémicas más moderadas. Las comidas principales como lunch y dinner presentan una distribución más dispersa, indicando mayor variabilidad entre individuos. Estos patrones podrían estar relacionados con factores hormonales, contenido nutricional o el tiempo en ayuno previo a cada comida.

In [583]:
clin_vars = ['a1c_pdl_(lab)', 'cholesterol', 'bmi', 'non_hdl']
gluc_vars = ['iauc_libre', 'peak_value_libre', 'delta_libre',
             'iauc_dexcom', 'peak_value_dexcom', 'delta_dexcom']

# Calcular matriz de correlación
correlation_df = metrics_both_df[clin_vars + gluc_vars].corr().loc[clin_vars, gluc_vars]

# Heatmap
plt.figure(figsize=(10, 4))
sns.heatmap(
    correlation_df,
    annot=True,
    cmap=sns.diverging_palette(250, 10, as_cmap=True),  # alternativa refinada a 'coolwarm'
    center=0,
    fmt=".2f",
    annot_kws={"fontsize": 8}
)
plt.title('Correlación entre Variables Clínicas y Métricas de Glucosa', fontsize=13)
plt.ylabel('Variables Clínicas', fontsize=11)
plt.xlabel('Métricas de Glucosa', fontsize=11)
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()


<Figure size 1000x400 with 2 Axes>

El mapa de calor revela una correlación clínica destacada entre la hemoglobina glucosilada (HbA1c) y las métricas de glucosa postprandial, especialmente con el valor pico tanto en el sensor Libre (r = 0.62) como en Dexcom (r = 0.57), lo que sugiere que individuos con mayor HbA1c tienden a alcanzar niveles más altos de glucosa tras las comidas. 

Otras variables clínicas como colesterol, BMI, y non-HDL muestran correlaciones muy bajas o negativas, lo cual indica que su relación con las respuestas glucémicas agudas es débil o inexistente en este conjunto de datos. La salud del revestimiento intestinal (“gut_lining_health”) tiene correlaciones ligeramente positivas pero poco significativas, lo que puede apuntar a un papel secundario o más complejo en la modulación glucémica.

### `metrics_libre_df`

In [584]:
print("Estadísticas - iAUC Libre:")
metrics_libre_df['iauc_libre'].describe()

Estadísticas - iAUC Libre:


count     1329.000000
mean      2893.718911
std       2868.901438
min          0.000000
25%        757.500000
50%       2041.000000
75%       4065.266667
max      15486.900000
Name: iauc_libre, dtype: float64

Muestra una distribución claramente asimétrica hacia la derecha, con una media de 1140.9 mg·min/dL y una desviación estándar alta, lo que indica gran variabilidad entre eventos. Aunque la mayoría de las respuestas glucémicas se encuentran entre 247.5 y 1633.0 (IQR), existen valores extremos elevados (hasta 5787.5) y varios eventos con valor 0, lo cual podría deberse a errores, registros tardíos o comidas sin impacto detectable.

In [585]:
plt.figure(figsize=(8, 4))

sns.histplot(
    metrics_libre_df['iauc_libre'],
    kde=True,
    bins=30,
    color='#C44E52'
)
plt.title('Distribución de iAUC (Libre GL)', fontsize=13)
plt.xlabel('iAUC Libre (mg·min/dL)', fontsize=11)
plt.ylabel('Frecuencia', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


<Figure size 800x400 with 1 Axes>

Confirma una fuerte asimetría positiva: hay una alta concentración de eventos con respuesta glucémica baja, incluyendo un pico llamativo en 0, lo que podría indicar registros erróneos o comidas mal marcadas. A medida que el valor de iAUC aumenta, la frecuencia disminuye gradualmente, extendiéndose hasta valores cercanos a 6000 mg·min/dL.

### Tratamiento de la variable `iauc_libre`

Mismo tratamiento que se hizo en las variables de Área Incremental bajo la curva para el dataset anterior.

In [586]:
metrics_libre_df['impute_libre'] = marcar_para_imputar(metrics_libre_df, 'iauc_libre')
metrics_libre_df[metrics_libre_df['impute_libre'] == True]

,patient,meal_type,calories,carbs,protein,fat,fiber,iauc_libre,peak_value_libre,delta_libre,time_to_peak_libre,variability_libre,early_iauc_libre,mean_calories_activity,mean_hr,mean_mets,age,gender,bmi,body_weight,height,self_identify,a1c_pdl_(lab),fasting_glu_pdl_(lab),insulin,triglycerides,cholesterol,hdl,non_hdl,ldl_(cal),vldl_(cal),cho/hdl_ratio,collection_time_pdl_(lab),#1_contour_fingerstick_glu,time_(t),#2_contour_fingerstick_glu,time_(t)_1,#3_contour_fingerstick_glu,time_(t)_2,lps_biosynthesis_pathways,biofilm_chemotaxis_and_virulence_pathways,tma_production_pathways,ammonia_production_pathways,metabolic_fitness,active_microbial_diversity,butyrate_production_pathways,flagellar_assembly_pathways,putrescine_production_pathways,uric_acid_production_pathways,bile_acid_metabolism_pathways,inflammatory_activity,protein_fermentation,oxalate_metabolism_pathways,salt_stress_pathways,microbiome_induced_stress,microbe_PC1,microbe_PC2,microbe_PC3,meal_time_hour_sin,meal_time_hour_cos,gut_health_index,gas_production_index,post_meal_state,impute_libre
20,1,dinner,193.00,32.0,5.0,4.00,3.00,0.000000,87.2,0.000000,0.0,3.052724,0.000000,1.865459,72.545455,17.793388,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,1,2,1,3,2,3,2,2,2,3,2,1,1,3,2,12.073691,-3.625122,10.078317,-8.660254e-01,0.500000,6,6,1290.829452,True
26,1,lunch,585.00,40.0,38.0,17.00,12.00,12.800000,83.0,1.133333,94.0,4.654002,0.000000,3.425929,83.776860,32.677686,27,M,22.265239,133.8,65.0,Hispanic/Latino,5.4,91,2.5,67.0,216,74,142,130,13,2.9,2025-06-01 11:06:00,89,2025-06-01 09:40:00,73,2025-06-01 12:11:00,81,2025-06-01 13:18:00,2,1,2,1,3,2,3,2,2,2,3,2,1,1,3,2,12.073691,-3.625122,10.078317,-2.588190e-01,-0.965926,6,6,2737.633905,True
56,2,breakfast,902.00,73.0,22.0,42.00,7.00,21.000000,77.0,1.533333,19.0,3.136700,21.000000,1.214408,73.644628,13.041322,49,F,30.946742,169.2,62.0,Hispanic/Latino,5.5,93,14.8,61.0,181,91,90,78,12,2.0,2025-06-01 07:38:00,91,2025-06-01 07:52:00,123,2025-06-01 09:21:00,80,2025-06-01 10:22:00,1,1,2,2,2,2,1,2,2,2,1,2,1,3,2,1,10.296673,1.411377,0.198310,-8.660254e-01,0.500000,4,3,960.423332,True
66,3,breakfast,712.00,66.0,22.0,42.00,0.00,9438.300000,261.0,138.133333,58.0,40.704149,1023.333333,2.317037,86.900826,26.190083,59,F,26.948690,157.0,64.0,Hispanic/Latino,6.5,118,17.4,154.0,190,74,116,90,31,2.6,2025-06-01 07:25:00,119,2025-06-01 07:38:00,166,2025-06-01 09:23:00,98,2025-06-01 10:23:00,1,1,2,1,2,2,2,2,2,2,2,2,1,1,3,2,12.298997,1.759080,2.869390,7.071068e-01,-0.707107,6,7,2275.939827,True
67,3,lunch,333.00,56.4,7.2,7.80,3.00,10567.500000,216.0,140.000000,90.0,45.559016,570.000000,2.039928,84.586777,23.057851,59,F,26.948690,157.0,64.0,Hispanic/Latino,6.5,118,17.4,154.0,190,74,116,90,31,2.6,2025-06-01 07:25:00,119,2025-06-01 07:38:00,166,2025-06-01 09:23:00,98,2025-06-01 10:23:00,1,1,2,1,2,2,2,2,2,2,2,2,1,1,3,2,12.298997,1.759080,2.869390,-2.588190e-01,-0.965926,6,7,1950.389318,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308,49,breakfast,712.00,66.0,22.0,42.00,0.00,10687.500000,296.0,134.000000,90.0,42.461071,750.000000,3.288797,103.884298,35.314050,58,F,36.090876,184.8,60.0,Hispanic/Latino,7.2,148,25.2,124.0,146,43,103,81,25,3.4,2025-06-01 08:19:00,145,2025-06-01 08:35:00,257,2025-06-01 09:33:00,182,2025-06-01 10:37:00,1,1,3,3,3,3,3,3,3,3,3,2,2,2,2,2,-7.121564,1.133549,0.655322,9.659258e-01,-0.258819,5,6,3668.575234,True
1309,49,lunch,416.25,70.5,9.0,9.75,3.75,9602.100000,219.0,119.800000,120.0,36.968724,861.900000,3.702880,112.264463,39.760331,58,F,36.090876,184.8,60.0,Hispanic/Latino,7.2,148,25.2,124.0,146,43,103,81,25,3.4,2025-06-01 08:19:00,145,2025-06-01 08:35:00,257,2025-06-01 09:33:00,182,2025-06-01 1

In [587]:
features = ['carbs', 'calories', 'protein', 'fat', 'fiber', 'bmi', 'age', 'insulin', 'a1c_pdl_(lab)', 'triglycerides', 'non_hdl', 'cho/hdl_ratio', 'microbe_PC1']

# Para iauc_libre
train_libre = metrics_libre_df[~metrics_libre_df['impute_libre']] # Seleccionar solo las filas donde no se necesita imputar
X_libre = train_libre[features] # Solo las columnas que se usarán como características
y_libre = train_libre['iauc_libre'] # La variable objetivo en esas filas

y_log = np.log1p(y_libre) # Aplicamos transformación logarítmica a la variable objetivo para estabilizar la varianza y mantener valores positivos en las predicciones

model_libre = XGBRegressor(n_estimators=500, max_depth=10, random_state=42)
model_libre.fit(X_libre, y_log)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [588]:
# Datos a imputar
to_impute_libre = metrics_libre_df[metrics_libre_df['impute_libre']]
X_imp_libre = to_impute_libre[features]

# Predicción
preds_log = model_libre.predict(X_imp_libre)
preds_final = np.expm1(preds_log)

# Asignación
metrics_libre_df.loc[to_impute_libre.index, 'iauc_libre'] = preds_final

In [589]:
metrics_libre_df = cap_outliers_iqr(metrics_libre_df, 'iauc_libre')

In [590]:
plt.figure(figsize=(8, 4))

sns.histplot(
    metrics_libre_df['iauc_libre'],
    kde=True,
    bins=30,
    color='#C44E52'
)
plt.title('Distribución de iAUC (Libre GL)', fontsize=13)
plt.xlabel('iAUC Libre (mg·min/dL)', fontsize=11)
plt.ylabel('Frecuencia', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


<Figure size 800x400 with 1 Axes>

De esta manera obtenemos también para este dataset una variable con menos outliers, una distribución sesgada positivamente con mayor sentido fisiológico.

### EDA del conjunto de datos

In [591]:
# Definir columnas
macros = ['carbs', 'protein', 'fat', 'fiber']
metricas_libre = ['iauc_libre', 'delta_libre', 'peak_value_libre', 'time_to_peak_libre', 'variability_libre']

# Calcular matriz de correlación
corr_libre = metrics_libre_df[macros + metricas_libre].corr().loc[macros, metricas_libre]

# Mapa de calor para Libre
plt.figure(figsize=(10, 4))
sns.heatmap(
    corr_libre,
    annot=True,
    cmap=sns.light_palette("#C44E52", as_cmap=True),
    vmin=-1,
    vmax=1,
    fmt=".2f",
    annot_kws={"fontsize": 8}
)
plt.title('Correlación entre Macronutrientes y Métricas (Libre GL)', fontsize=13)
plt.xlabel('Métricas de Glucosa (Libre)', fontsize=11)
plt.ylabel('Macronutrientes', fontsize=11)
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()


<Figure size 1000x400 with 2 Axes>

El mapa de calor muestra que los carbohidratos tienen la correlación positiva más consistente con todas las métricas de respuesta glucémica medidas por el sensor Libre, destacando su mayor impacto en la elevación y variabilidad de la glucosa postprandial (r ≈ 0.23–0.25). En contraste, la fibra presenta correlaciones negativas moderadas, lo que sugiere un efecto atenuante sobre la respuesta glucémica. Proteínas y grasas muestran asociaciones muy débiles o nulas, lo que indica que su influencia en estas métricas es limitada en este conjunto de datos.

In [592]:
# Agrupar por paciente y calcular estadísticas de iAUC
iauc_stats_by_patient = metrics_libre_df.groupby('patient')['iauc_libre'].agg(['mean', 'median', 'std', 'count']).reset_index()
iauc_stats_by_patient = iauc_stats_by_patient.sort_values('mean', ascending=False)

# Visualizar el promedio de iAUC por paciente
plt.figure(figsize=(12, 6))
sns.barplot(data=iauc_stats_by_patient, x='patient', y='mean', palette='Blues_d')
plt.xticks(rotation=90)
plt.title('iAUC Promedio por Paciente (Sensor Libre)')
plt.xlabel('Paciente')
plt.ylabel('iAUC promedio (mg·min/dL)')
plt.tight_layout()
plt.show()


<Figure size 1200x600 with 1 Axes>

Se observa una gran variabilidad entre individuos, con algunos pacientes presentando respuestas glucémicas considerablemente más elevadas que otros. Esto sugiere que factores individuales —como características metabólicas, composición corporal, microbiota o hábitos alimenticios— influyen significativamente en la respuesta postprandial. Además, algunos pacientes (como el 7 u 8) tienen valores muy bajos, lo cual podría deberse a un mejor control glucémico, comidas menos glucémicas o errores de registro. Esta heterogeneidad refuerza la necesidad de personalizar las estrategias de monitoreo y tratamiento.

In [593]:
plt.figure(figsize=(14, 6))

sns.boxplot(
    data=metrics_libre_df,
    x='meal_type',
    y='iauc_libre',
    color='#C44E52'
)
plt.title('Distribución de iAUC (Libre GL) por Tipo de Comida', fontsize=13)
plt.ylabel('iAUC Libre (mg·min/dL)', fontsize=11)
plt.xlabel('Tipo de Comida', fontsize=11)
plt.xticks(rotation=45, fontsize=9)
plt.yticks(fontsize=9)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()



<Figure size 1400x600 with 1 Axes>

In [594]:
clin_vars = ['a1c_pdl_(lab)', 'cholesterol', 'bmi', 'non_hdl']
gluc_vars = ['iauc_libre', 'peak_value_libre', 'delta_libre']

# Calcular matriz de correlación
correlation_df = metrics_libre_df[clin_vars + gluc_vars].corr().loc[clin_vars, gluc_vars]

# Heatmap
plt.figure(figsize=(10, 4))
sns.heatmap(
    correlation_df,
    annot=True,
    cmap=sns.diverging_palette(250, 10, as_cmap=True),
    center=0,
    fmt=".2f",
    annot_kws={"fontsize": 8}
)
plt.title('Correlación entre Variables Clínicas y Métricas (Libre GL)', fontsize=13)
plt.ylabel('Variables Clínicas', fontsize=11)
plt.xlabel('Métricas de Glucosa', fontsize=11)
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()


<Figure size 1000x400 with 2 Axes>

# **15. Selección de variables** <a class="anchor" id="15"></a>


[Tabla de Contenidos](#0.1)

Para seleccionar las variables con las que entrenaremos al modelo, es necesario tomar en cuenta que, entre más variables tiene el modelo, más ruido se le añade a sus predicciones, por lo tanto es mejor "acortar" el número de columnas.

Para esto, es posible analizar la correlación entre distintas variables, y así, poder eliminar variables altamente correlaciondas, pues es equivalente a la misma información para el modelo. 

Se hace un análisis de correlación por datasets de origen, dentro del dataset completo de metrics_both_df.

In [595]:
# Obtener las columnas de gut_scores_imputed (excepto 'subject' si no está en metrics_both_df)
columns_corr = [col for col in gut_scores_imputed.columns if col in metrics_both_df.columns and col != 'subject']

# Calcular la matriz de correlación solo para esas columnas en metrics_both_df
corr_matrix = metrics_both_df[columns_corr].corr()

# Graficar el heatmap
plt.figure(figsize=(14, 14))  # Ajustar según el número de variables
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap=sns.diverging_palette(250, 10, as_cmap=True),
    fmt=".2f",
    square=True,
    annot_kws={"fontsize": 7}
)
plt.title('Matriz de Correlación – Variables del Microbioma', fontsize=13)
plt.xticks(fontsize=8, rotation=90)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()



<Figure size 1400x1400 with 2 Axes>

Se pueden observar muchas variables altamente relacionadas, aquellas cuya correlación es mayor o igual a 0.75 o menor o igual a -0.75:
* `microbiome_induced_stress` y `protein_fermentation`.

Encontramos así que:
* `microbiome_induced_stress` puede representar muy bien los valores de `protein_fermentation`.

In [596]:
cols_to_drop = []
cols_to_drop_now = ['protein_fermentation']

In [597]:
len(metrics_both_df.columns)

74

In [598]:
metrics_both_df.drop(columns = cols_to_drop_now, inplace=True)
len(metrics_both_df.columns)

73

In [599]:
cols_to_drop.extend(cols_to_drop_now)
cols_to_drop

['protein_fermentation']

Ahora en variables del dataset bio:

In [600]:
# Obtener las columnas de bio (excepto 'subject', 'gender' y 'self_identify' si no están en metrics_both_df)
columns_corr = [
    col for col in bio.columns
    if col in metrics_both_df.columns and col not in ['subject', 'gender', 'self_identify']
]

# Calcular la matriz de correlación solo para esas columnas en metrics_both_df
corr_matrix = metrics_both_df[columns_corr].corr()

# Graficar el heatmap
plt.figure(figsize=(14, 14))  # Ajustar según el número de variables
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap=sns.diverging_palette(250, 10, as_cmap=True),
    fmt=".2f",
    square=True,
    annot_kws={"fontsize": 7}
)
plt.title('Matriz de Correlación – Variables del Microbioma', fontsize=13)
plt.xticks(fontsize=8, rotation=90)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()


<Figure size 1400x1400 with 2 Axes>

Con el umbral anterior encontramos:
* `body_weigth` con `bmi`
* `a1c_pdl_(lab)` con `#3_contour_fingerstick_glu`, `#1_contour_fingerstick_glu`, y `#2_contour_fingerstick_glu` 
* `fasting_glu_pdl_(lab)` con `#3_contour_fingerstick_glu`, `#1_contour_fingerstick_glu`, y `#2_contour_fingerstick_glu` 
* `cho/hdl_ratio` con `triglycerides`
* `non-hdl` con `cholesterol`, `cho/hdl_ratio`
* `vldl_(cal)` con `ldl_(cal)`
* `#3_contour_fingerstick_glu`, `#1_contour_fingerstick_glu`, y `#2_contour_fingerstick_glu` entre sí mismas

En este punto es importante destacar información sobre algunas de las variables:
* `collection_time_pdl_(lab)`, `#1_contour_fingerstick_glu`, `time_(t)`, `#2_contour_fingerstick_glu`, `time_(t)_1`, `#3_contour_fingerstick_glu`, `time_(t)_2`  todas estas son lecturas capilares redundantes o incluso ruidosas, prque son muy puntuales, pueden tener sesgo por el momento exacto en que se tomó la punción y no aportan más información que una curva de glucosa continua densa.

Así tenemos las columnas a eliminar:

In [601]:
print('Número de columnas original: ', len(metrics_both_df.columns))

cols_to_drop_now= [
    'body_weight',
    '#1_contour_fingerstick_glu', 'time_(t)',
    '#2_contour_fingerstick_glu', 'time_(t)_1',
    '#3_contour_fingerstick_glu', 'time_(t)_2',
    'collection_time_pdl_(lab)',
    'fasting_glu_pdl_(lab)',
    'cho/hdl_ratio',
    'cholesterol',
    'vldl_(cal)']

metrics_both_df.drop(columns=cols_to_drop_now, inplace=True)

cols_to_drop.extend(cols_to_drop_now)
print('Número de columnas tras eliminar: ', len(metrics_both_df.columns))

Número de columnas original:  73
Número de columnas tras eliminar:  61


Ahora analizamos columnas puramente de las métricas que obtuvimos:

In [602]:
# Obtener columnas excluyendo las que provienen de bio, gut_scores_imputed y otras irrelevantes
columns_corr = [
    col for col in metrics_both_df.columns
    if col not in bio.columns
    and col not in gut_scores_imputed.columns
    and col not in ['patient', 'meal_time', 'meal_type']
]

# Calcular la matriz de correlación solo para esas columnas
corr_matrix = metrics_both_df[columns_corr].corr()

# Graficar el heatmap
plt.figure(figsize=(14, 14))  # Ajustar según número de variables
sns.heatmap(
    corr_matrix,
    annot=True,
    annot_kws={"fontsize": 8},
    cmap=sns.diverging_palette(250, 10, as_cmap=True),
    fmt=".2f",
    square=True
)
plt.title('Matriz de Correlación – Variables no Microbioma', fontsize=13)
plt.xticks(fontsize=8, rotation=90)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()


<Figure size 1400x1400 with 2 Axes>

| Variables relacionadas                        | Correlación | Interpretación                                                                                  |
| --------------------------------------------- | ----------- | ----------------------------------------------------------------------------------------------- |
| `delta_libre` – `peak_value_libre`         | \~0.93      | A mayor pico de glucosa, mayor delta (lógico: el delta refleja la subida desde el valor basal). |
| `variability_libre` – `delta_libre`         | \~0.87      | Cuando hay mucha variabilidad, también hay grandes cambios de glucosa (alta delta).             |
| `variability_libre – peak_value_libre`   | \~0.84      | Glucosa más alta tras la comida suele estar acompañada de mayor oscilación.                     |
| `early_iauc_libre – delta_libre`         | \~0.80      | La área bajo la curva inicial está fuertemente influida por cuánto sube la glucosa rápidamente. |
| `variability_dexcom – delta_dexcom`       | \~0.91      | Mismo fenómeno que en Libre, pero captado por el sensor Dexcom.                                 |
| `variability_dexcom – peak_value_dexcom` | \~0.85      | Altos picos glucémicos → mayor variabilidad.                                                    |
| `early_iauc_dexcom – delta_dexcom`       | \~0.81      | Refuerza que el delta es un gran predictor de la carga glucémica temprana.                      |
| `mean_calories_activity – mean_mets`     | \~0.87      | Gasto calórico postprandial está muy relacionado con la intensidad del esfuerzo.                |
| `post_meal_state – mean_hr`              | \~0.83      | Confirmación de que esta variable compuesta representa bien la activación fisiológica.          |
| `post_meal_state – mean_mets`            | \~0.89      | Similar al anterior: alta actividad → mayor post\_meal\_state.                                  |


Las relaciones entre picos de glucosa, delta y variabilidad son esperadas y coherentes fisiológicamente.

Las variables compuestas (post_meal_state, etc.) se comportan como resúmenes informativos válidos.

Hay redundancia en algunas variables (por ejemplo: delta, variability y peak_value están altamente correlacionadas entre sí), lo que puede invitar a usar solo una de ellas en modelos, o aplicar reducción de dimensionalidad (PCA, por ejemplo).

El comportamiento entre sensores (Libre y Dexcom) es similar pero no idéntico, lo cual es normal por sus diferencias técnicas.

Para evitar colinealidad, incluyendo variables que aportan información redundante, se eliminarán variables muy correlacionadas, pero dejando las más correlacionadas con carbs.

In [603]:
print('Número de columnas antes de eliminar: ', len(metrics_both_df.columns))

cols_to_drop_now = [
    'early_iauc_libre',
    'peak_value_libre',
    'early_iauc_dexcom',
    'peak_value_dexcom'
]

metrics_both_df.drop(columns=cols_to_drop_now, inplace=True, errors='ignore')

cols_to_drop.extend(cols_to_drop_now)

print('Número de columnas tras eliminar: ', len(metrics_both_df.columns))

Número de columnas antes de eliminar:  61
Número de columnas tras eliminar:  57


## Selección de variables relevantes:

Otro mecanismo de selección de variables es el de el entendimiento de su significado y cómo pueden afectar al modelo en su explicabilidad, o añadir ruido por ser medidas o variables "irrelevantes".

In [604]:
metrics_both_df.columns

Index(['patient', 'meal_type', 'calories', 'carbs', 'protein', 'fat', 'fiber',
       'iauc_libre', 'delta_libre', 'time_to_peak_libre', 'variability_libre',
       'iauc_dexcom', 'delta_dexcom', 'time_to_peak_dexcom',
       'variability_dexcom', 'mean_calories_activity', 'mean_hr', 'mean_mets',
       'age', 'gender', 'bmi', 'height', 'self_identify', 'a1c_pdl_(lab)',
       'insulin', 'triglycerides', 'hdl', 'non_hdl', 'ldl_(cal)',
       'lps_biosynthesis_pathways',
       'biofilm_chemotaxis_and_virulence_pathways', 'tma_production_pathways',
       'ammonia_production_pathways', 'metabolic_fitness',
       'active_microbial_diversity', 'butyrate_production_pathways',
       'flagellar_assembly_pathways', 'putrescine_production_pathways',
       'uric_acid_production_pathways', 'bile_acid_metabolism_pathways',
       'inflammatory_activity', 'oxalate_metabolism_pathways',
       'salt_stress_pathways', 'microbiome_induced_stress', 'microbe_PC1',
       'microbe_PC2', 'microbe_PC3'

Analizando el significado de cada variable, como se hizo en los distintos EDA, encontramos que:

| Columna                                                                     | Motivo                                                                                     |
| --------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------ |
| `salt_stress_pathways`                                                      | Baja conexión con respuesta esperada                                                       |
| `oxalate_metabolism_pathways`, `uric_acid_production_pathways`              | Relacionadas a ácido úrico, no a carbohidratos directamente                                |
| `flagellar_assembly_pathways`                                               | Poco impacto, solo ruido microbiano                                                        |
| `self_identify`                                                             | Categoría con ruido social/contextual.                                                     |

In [605]:
print("Columnas antes de eliminar: " , len(metrics_both_df.columns))

cols_to_drop_now = ['salt_stress_pathways', 'oxalate_metabolism_pathways', 'uric_acid_production_pathways',
                   'flagellar_assembly_pathways', 'self_identify']

metrics_both_df.drop(columns = cols_to_drop_now, inplace = True)

cols_to_drop.extend(cols_to_drop_now)

print("Columnas después de eliminar: " , len(metrics_both_df.columns))

Columnas antes de eliminar:  57
Columnas después de eliminar:  52


También borramos las columnas creadas anteriormente para trabajar variables:

In [606]:
cols_to_drop_now = ['impute_libre', 'impute_dexcom']
metrics_both_df.drop(columns=cols_to_drop_now, inplace=True)
cols_to_drop.extend(cols_to_drop_now)
print("Columnas finales: " , len(metrics_both_df.columns))

Columnas finales:  50


Se eliminan las mismas columnas que se eliminaron para `metrics_libre_df`

In [607]:
len(cols_to_drop)

24

In [608]:
cols_not_in_libre = ['impute_dexcom', 'early_iauc_dexcom', 'peak_value_dexcom']
for col in cols_not_in_libre:
    cols_to_drop.remove(col)

In [609]:
print('Número de columnas antes de eliminar: ', len(metrics_libre_df.columns))
metrics_libre_df.drop(columns = cols_to_drop, inplace = True)
print('Número de columnas después de eliminar: ', len(metrics_libre_df.columns))

Número de columnas antes de eliminar:  64
Número de columnas después de eliminar:  43


## Comprobación de valores nulos

In [610]:
metrics_both_df.isna().any(axis=1).sum()

0

In [611]:
metrics_libre_df.isna().any(axis=1).sum()

0

# **16. Preprocesamiento de variables** <a class="anchor" id="16"></a>


[Tabla de Contenidos](#0.1)


## Encoding de variables categóricas

La variable `gender` es una variable categórica que define si el sujeto es de género femenino o masculino.

In [612]:
print(metrics_both_df['gender'].value_counts())
print('')
print(metrics_libre_df['gender'].value_counts())

gender
F    823
M    467
Name: count, dtype: int64

gender
F    845
M    484
Name: count, dtype: int64


In [613]:
# Aplicamos un label encoding simple y eliminamos la columna original
metrics_both_df['gender_encoded'] = metrics_both_df['gender'].map({'F': 0, 'M': 1})
metrics_both_df.drop(columns=['gender'], inplace=True)

In [614]:
# Aplicamos un label encoding simple y eliminamos la columna original
metrics_libre_df['gender_encoded'] = metrics_libre_df['gender'].map({'F': 0, 'M': 1})
metrics_libre_df.drop(columns=['gender'], inplace=True)

La otra variable categórica es la variable `meal_type` que indica qué tipo de comida es el evento actual:
* Desayuno
* Comida
* Cena
* Snack

In [615]:
# One-hot encoding de meal_type_clean con n-1 columnas
meal_dummies = pd.get_dummies(metrics_both_df['meal_type'], prefix='meal', drop_first=True)

# Aseguramos que los dummies sean tipo int
meal_dummies = meal_dummies.astype(int)

# Concatenamos con el DataFrame original
metrics_both_df = pd.concat([metrics_both_df, meal_dummies], axis=1)


In [616]:
metrics_both_df.drop(columns=['meal_type'], inplace=True)

Ahora para `metrics_libre_df`

In [617]:
# One-hot encoding de meal_type_clean con n-1 columnas
meal_dummies = pd.get_dummies(metrics_libre_df['meal_type'], prefix='meal', drop_first=True)

# Aseguramos que los dummies sean tipo int
meal_dummies = meal_dummies.astype(int)

# Concatenamos con el DataFrame original
metrics_libre_df = pd.concat([metrics_libre_df, meal_dummies], axis=1)

metrics_libre_df.drop(columns=['meal_type'], inplace=True)


In [618]:
metrics_libre_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1329 entries, 0 to 1328
Data columns (total 45 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   patient                                    1329 non-null   int64  
 1   calories                                   1329 non-null   float64
 2   carbs                                      1329 non-null   float64
 3   protein                                    1329 non-null   float64
 4   fat                                        1329 non-null   float64
 5   fiber                                      1329 non-null   float64
 6   iauc_libre                                 1329 non-null   float64
 7   delta_libre                                1329 non-null   float64
 8   time_to_peak_libre                         1329 non-null   float64
 9   variability_libre                          1329 non-null   float64
 10  mean_calories_activity  

Por ahora se entrenan los modelos sin las variables `meal_dinner`, `meal_lunch` y `meal_snack` porque pueden ser propensas a data leakeage.

In [619]:
metrics_both_df.drop(columns=['meal_dinner', 'meal_lunch', 'meal_snack'], inplace=True)
metrics_libre_df.drop(columns=['meal_dinner', 'meal_lunch', 'meal_snack'], inplace=True)

# **17. División del conjunto de datos en train y test** <a class="anchor" id="17"></a>


[Tabla de Contenidos](#0.1)

## `metrics_both_df`

Para este enfoque, se utiliza la división "leave one man out", es decir, el modelo se entrenará con los datos de algunos de los pacientes, y se probará para predecir los datos de otros pacientes que no había visto antes.

In [620]:
# Obtener los IDs únicos de pacientes
unique_patients = metrics_both_df['patient'].unique()

# Dividir los pacientes en conjunto de entrenamiento y prueba
train_patients, test_patients = train_test_split(unique_patients, test_size=0.15, random_state=42)

In [621]:
# Crear los subconjuntos de datos
train_df = metrics_both_df[metrics_both_df['patient'].isin(train_patients)].copy()
test_df = metrics_both_df[metrics_both_df['patient'].isin(test_patients)].copy()

columns_nutrients = ['carbs', 'protein', 'fat', 'fiber', 'calories']

X_train_both = train_df.drop(columns = columns_nutrients + ['patient'])
y_train_both = train_df['carbs']

X_test_both = test_df.drop(columns = columns_nutrients + ['patient'])
y_test_both = test_df['carbs']

In [622]:
print(X_train_both.shape)
print(y_train_both.shape)
print(X_test_both.shape)
print(y_test_both.shape)

(1087, 43)
(1087,)
(203, 43)
(203,)


In [623]:
X_train_both.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1087 entries, 0 to 1289
Data columns (total 43 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   iauc_libre                                 1087 non-null   float64
 1   delta_libre                                1087 non-null   float64
 2   time_to_peak_libre                         1087 non-null   float64
 3   variability_libre                          1087 non-null   float64
 4   iauc_dexcom                                1087 non-null   float64
 5   delta_dexcom                               1087 non-null   float64
 6   time_to_peak_dexcom                        1087 non-null   float64
 7   variability_dexcom                         1087 non-null   float64
 8   mean_calories_activity                     1087 non-null   float64
 9   mean_hr                                    1087 non-null   float64
 10  mean_mets                    

Los modelos predictores que se utilizarán son los siguientes:
* XGBoostRegressor
* RandomForestRegressor

La característica que tienen los tres modelos es que son modelos basados en árboles de decisión, lo que quiere decir que son robustos a la entrada de datos y no requieren llevar un escalado.

En este caso, nos centraremos únicamente en la predicción de la cantidad de carbohidratos consumida de acuerdo con la información del dataset, así que se borran las otras variables del caso.

## `metrics_libre_df`

In [624]:
# Obtener los IDs únicos de pacientes
unique_patients = metrics_libre_df['patient'].unique()

# Dividir los pacientes en conjunto de entrenamiento y prueba
train_patients, test_patients = train_test_split(unique_patients, test_size=0.15, random_state=201100)

In [625]:
# Crear los subconjuntos de datos
train_df = metrics_libre_df[metrics_libre_df['patient'].isin(train_patients)].copy()
test_df = metrics_libre_df[metrics_libre_df['patient'].isin(test_patients)].copy()

# Separar características de las variables objetivo
columns_nutrients = ['carbs', 'protein', 'fat', 'fiber', 'calories']

X_train_libre = train_df.drop(columns=columns_nutrients + ['patient'])
y_train_libre = train_df['carbs']

X_test_libre = test_df.drop(columns=columns_nutrients + ['patient'])
y_test_libre = test_df['carbs']

In [626]:
print(X_train_libre.shape)
print(y_train_libre.shape)
print(X_test_libre.shape)
print(y_test_libre.shape)

(1129, 36)
(1129,)
(200, 36)
(200,)


In [1103]:
X_test_libre.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 324 to 1186
Data columns (total 36 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   iauc_libre                                 200 non-null    float64
 1   delta_libre                                200 non-null    float64
 2   time_to_peak_libre                         200 non-null    float64
 3   variability_libre                          200 non-null    float64
 4   mean_calories_activity                     200 non-null    float64
 5   mean_hr                                    200 non-null    float64
 6   mean_mets                                  200 non-null    float64
 7   age                                        200 non-null    int64  
 8   bmi                                        200 non-null    float64
 9   height                                     200 non-null    float64
 10  a1c_pdl_(lab)               

Ahora se tienen los conjuntos de entrenamiento y prueba para dos distintos entrenamientos:
1. El primer conjunto de datos contiene los datos de ambos sensores glucémicos
2. El segundo solo contiene la información del sensor `libre_gl`.

# **18. Entrenamiento de modelos de predicción** <a class="anchor" id="18"></a>


[Tabla de Contenidos](#0.1)

## Entrenamiento con todos los datos de la respuesta glucémica postprandial derivados de la `iauc`.

Se entrena con todos los datos disponibles en nuestro dataset

Primero se prueba con los datasets que contienen la información de ambos sensores

### RandomForestRegressor

In [284]:
param_distributions = {
    'n_estimators': randint(200, 800),
    'max_depth': [None] + list(range(10, 50, 10)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'bootstrap': [True, False]
}

rf = RandomForestRegressor(random_state=42)

random_search_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=100,             
    cv=5,
    scoring='neg_mean_squared_error',           
    verbose=2,
    random_state=42
)

random_search_rf.fit(X_train_both, y_train_both)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END bootstrap=True, max_depth=30, max_features=0.7, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.1s
[CV] END bootstrap=True, max_depth=30, max_features=0.7, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.0s
[CV] END bootstrap=True, max_depth=30, max_features=0.7, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.1s
[CV] END bootstrap=True, max_depth=30, max_features=0.7, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.1s
[CV] END bootstrap=True, max_depth=30, max_features=0.7, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.1s
[CV] END bootstrap=True, max_depth=20, max_features=0.3, min_samples_leaf=8, min_samples_split=5, n_estimators=330; total time=   1.1s
[CV] END bootstrap=True, max_depth=20, max_features=0.3, min_samples_leaf=8, min_samples_split=5, n_estimators=330; total time=

RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
                   n_iter=100,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [None, 10, 20, 30, 40],
                                        'max_features': ['sqrt', 'log2', 0.3,
                                                         0.5, 0.7],
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000002135CBF1C50>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000002134FB60E50>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000002134E74AE50>},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=2)

In [285]:
y_pred = random_search_rf.predict(X_test_both)
rmse = root_mean_squared_error(y_test_both, y_pred)
r2 = r2_score(y_test_both, y_pred)

print(f"Best parameters: {random_search_rf.best_params_}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R²: {r2:.3f}")

Best parameters: {'bootstrap': True, 'max_depth': 20, 'max_features': 0.3, 'min_samples_leaf': 8, 'min_samples_split': 5, 'n_estimators': 330}
Test RMSE: 23.54
Test R²: 0.271


In [286]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test_both, y=y_pred, alpha=0.6)
plt.plot([y_test_both.min(), y_test_both.max()], [y_test_both.min(), y_test_both.max()], 'r--')
plt.xlabel("Valor real")
plt.ylabel("Predicción")
plt.title("Real vs. Predicho (Random Forest)")
plt.grid(True)
plt.show()


<Figure size 600x600 with 1 Axes>

In [287]:
# Obtener importancias
importances = random_search_rf.best_estimator_.feature_importances_
feature_names = X_test_both.columns
feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

# Graficar
plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance_df, x='importance', y='feature', palette='viridis')
plt.title('Importancia de características (Random Forest)')
plt.xlabel('Importancia')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

<Figure size 1000x800 with 1 Axes>

In [288]:
# Crear explainer para Random Forest
explainer = shap.TreeExplainer(random_search_rf.best_estimator_)
shap_values = explainer.shap_values(X_test_both)

# Gráfico summary
shap.summary_plot(shap_values, X_test_both, show=True)

<Figure size 800x950 with 2 Axes>

#### Bajando el número de features

In [289]:
# Top 15 por feature_importance_ del modelo
importances = random_search_rf.best_estimator_.feature_importances_
feature_names = X_test_both.columns
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
top_rf_features = importance_df.sort_values(by='importance', ascending=False).head(15)['feature'].tolist()


explainer = shap.TreeExplainer(random_search_rf.best_estimator_)
shap_values = explainer.shap_values(X_test_both)

# Media absoluta del valor SHAP por feature
shap_importance = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'feature': X_test_both.columns, 'mean_abs_shap': shap_importance})
top_shap_features = shap_df.sort_values(by='mean_abs_shap', ascending=False).head(15)['feature'].tolist()

In [290]:
# Unir ambas listas, sin duplicados
top_combined_features = list(set(top_rf_features + top_shap_features))

print(f"Total variables seleccionadas: {len(top_combined_features)}")
print("Variables seleccionadas:")
print(top_combined_features)

Total variables seleccionadas: 19
Variables seleccionadas:
['time_to_peak_libre', 'iauc_libre', 'microbe_PC2', 'delta_libre', 'age', 'post_meal_state', 'iauc_dexcom', 'mean_variability', 'delta_dexcom', 'mean_mets', 'meal_time_hour_sin', 'delta_ratio', 'meal_time_hour_cos', 'variability_libre', 'a1c_pdl_(lab)', 'gut_health_index', 'time_to_peak_dexcom', 'mean_calories_activity', 'variability_dexcom']


In [291]:
X_train_reduced = X_train_both[top_combined_features].copy()
X_test_reduced = X_test_both[top_combined_features].copy()

In [292]:
# Entrenar con los mismos mejores hiperparámetros encontrados
best_params = random_search_rf.best_params_

rf_reduced = RandomForestRegressor(**best_params, random_state=42)
rf_reduced.fit(X_train_reduced, y_train_both)

# Predicciones
y_pred = rf_reduced.predict(X_test_reduced)

# Métricas
rmse = root_mean_squared_error(y_test_both, y_pred)
r2 = r2_score(y_test_both, y_pred)

print(f"[REDUCIDO] Test RMSE: {rmse:.2f}")
print(f"[REDUCIDO] Test R²: {r2:.3f}")

[REDUCIDO] Test RMSE: 23.23
[REDUCIDO] Test R²: 0.290


### XGBRegressor

In [293]:
param_distributions = {
    'n_estimators': randint(200, 1000),             # Número de árboles
    'max_depth': randint(3, 15),                   # Profundidad máxima del árbol
    'learning_rate': uniform(0.01, 0.3),           # Tasa de aprendizaje
    'subsample': uniform(0.5, 0.5),                # Proporción de datos por árbol (0.5–1.0)
    'colsample_bytree': uniform(0.5, 0.5),         # Fracción de columnas por árbol
    'gamma': uniform(0, 5),                        # Regularización para crear nuevas ramas
}

xgb = XGBRegressor(random_state=42)

random_search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_distributions,
    n_iter=300,
    cv=5,
    scoring='r2',
    verbose=2,
    random_state=42
)

random_search_xgb.fit(X_train_both, y_train_both)

Fitting 5 folds for each of 300 candidates, totalling 1500 fits
[CV] END colsample_bytree=0.6872700594236812, gamma=4.75357153204958, learning_rate=0.22959818254342154, max_depth=7, n_estimators=814, subsample=0.7229163764267956; total time=   0.8s
[CV] END colsample_bytree=0.6872700594236812, gamma=4.75357153204958, learning_rate=0.22959818254342154, max_depth=7, n_estimators=814, subsample=0.7229163764267956; total time=   0.8s
[CV] END colsample_bytree=0.6872700594236812, gamma=4.75357153204958, learning_rate=0.22959818254342154, max_depth=7, n_estimators=814, subsample=0.7229163764267956; total time=   0.8s
[CV] END colsample_bytree=0.6872700594236812, gamma=4.75357153204958, learning_rate=0.22959818254342154, max_depth=7, n_estimators=814, subsample=0.7229163764267956; total time=   0.8s
[CV] END colsample_bytree=0.6872700594236812, gamma=4.75357153204958, learning_rate=0.22959818254342154, max_depth=7, n_estimators=814, subsample=0.7229163764267956; total time=   0.8s
[CV] END co

RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000021357619610>,
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x00000213542C3B10>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000021351B1DA10>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000021351BC64D0>},
                   random_state=42, scoring='r2', verbose=2)

In [294]:
y_pred = random_search_xgb.predict(X_test_both)
rmse = root_mean_squared_error(y_test_both, y_pred)
r2 = r2_score(y_test_both, y_pred)

print(f"Best parameters: {random_search_xgb.best_params_}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R²: {r2:.3f}")

Best parameters: {'colsample_bytree': 0.5784888413580969, 'gamma': 1.886429826210363, 'learning_rate': 0.010778507315093968, 'max_depth': 6, 'n_estimators': 456, 'subsample': 0.5422585038403409}
Test RMSE: 23.25
Test R²: 0.289


In [295]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test_both, y=y_pred, alpha=0.6)
plt.plot([y_test_both.min(), y_test_both.max()], [y_test_both.min(), y_test_both.max()], 'r--')
plt.xlabel("Valor real")
plt.ylabel("Predicción")
plt.title("Real vs. Predicho (Random Forest)")
plt.grid(True)
plt.show()

<Figure size 600x600 with 1 Axes>

In [296]:
# Crear explainer para Random Forest
explainer = shap.TreeExplainer(random_search_xgb.best_estimator_)
shap_values = explainer.shap_values(X_test_both)

# Gráfico summary
shap.summary_plot(shap_values, X_test_both, show=True)

<Figure size 800x950 with 2 Axes>

In [297]:
# Top 15 por feature_importance_ del modelo
importances = random_search_xgb.best_estimator_.feature_importances_
feature_names = X_test_both.columns
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
top_xgb_features = importance_df.sort_values(by='importance', ascending=False).head(15)['feature'].tolist()


explainer = shap.TreeExplainer(random_search_xgb.best_estimator_)
shap_values = explainer.shap_values(X_test_both)

# Media absoluta del valor SHAP por feature
shap_importance = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'feature': X_test_both.columns, 'mean_abs_shap': shap_importance})
top_shap_features = shap_df.sort_values(by='mean_abs_shap', ascending=False).head(15)['feature'].tolist()

In [298]:
# Unir ambas listas, sin duplicados
top_combined_features = list(set(top_xgb_features + top_shap_features))

print(f"Total variables seleccionadas: {len(top_combined_features)}")
print("Variables seleccionadas:")
print(top_combined_features)

Total variables seleccionadas: 25
Variables seleccionadas:
['height', 'time_to_peak_libre', 'tma_production_pathways', 'iauc_libre', 'microbe_PC2', 'delta_libre', 'ammonia_production_pathways', 'microbe_PC3', 'age', 'post_meal_state', 'iauc_dexcom', 'mean_variability', 'delta_dexcom', 'mean_mets', 'meal_time_hour_sin', 'mean_hr', 'delta_ratio', 'meal_time_hour_cos', 'gut_health_index', 'a1c_pdl_(lab)', 'hdl', 'putrescine_production_pathways', 'time_to_peak_dexcom', 'mean_calories_activity', 'microbe_PC1']


In [299]:
X_train_reduced = X_train_both[top_combined_features].copy()
X_test_reduced = X_test_both[top_combined_features].copy()

In [300]:
# Entrenar con los mismos mejores hiperparámetros encontrados
best_params = random_search_xgb.best_params_

xgb_reduced = XGBRegressor(**best_params, random_state=42)
xgb_reduced.fit(X_train_reduced, y_train_both)

# Predicciones
y_pred = xgb_reduced.predict(X_test_reduced)

# Métricas
rmse = root_mean_squared_error(y_test_both, y_pred)
r2 = r2_score(y_test_both, y_pred)

print(f"[REDUCIDO] Test RMSE: {rmse:.2f}")
print(f"[REDUCIDO] Test R²: {r2:.3f}")

[REDUCIDO] Test RMSE: 23.38
[REDUCIDO] Test R²: 0.281


#### Entrenamiento aplicando transformación logarítmica

In [301]:
param_distributions = {
    'n_estimators': randint(200, 1000),            # Número de árboles
    'max_depth': randint(3, 15),                   # Profundidad máxima del árbol
    'learning_rate': uniform(0.01, 0.3),           # Tasa de aprendizaje
    'subsample': uniform(0.5, 0.5),                # Proporción de datos por árbol (0.5–1.0)
    'colsample_bytree': uniform(0.5, 0.5),         # Fracción de columnas por árbol
    'gamma': uniform(0, 5),                        # Regularización para crear nuevas ramas
}

xgb_log1 = XGBRegressor(random_state=42)

random_search_xgb_log1 = RandomizedSearchCV(
    estimator=xgb_log1,
    param_distributions=param_distributions,
    n_iter=300,
    cv=5,
    scoring='r2',
    verbose=1,
    random_state=42
)

In [302]:
y_train_log = np.log1p(y_train_both)

random_search_xgb_log1.fit(X_train_both, y_train_log)

Fitting 5 folds for each of 300 candidates, totalling 1500 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002135F408C10>,
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000002135F40AD50>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000002135F3F2810>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002135F408F50>},
                   random_state=42, scoring='r2', verbose=1)

In [303]:
y_pred_log = random_search_xgb_log1.predict(X_test_both)
y_pred = np.expm1(y_pred_log)

rmse = root_mean_squared_error(y_test_both, y_pred)
r2 = r2_score(y_test_both, y_pred)

print(f"Best parameters: {random_search_xgb_log1.best_params_}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R²: {r2:.3f}")

Best parameters: {'colsample_bytree': 0.5602612893932322, 'gamma': 3.655118357291408, 'learning_rate': 0.06777970833568912, 'max_depth': 10, 'n_estimators': 697, 'subsample': 0.7108813298652563}
Test RMSE: 28.23
Test R²: -0.048


## Entrenamiento para `metrics_libre_df`

In [304]:
param_distributions = {
    'n_estimators': randint(200, 800),             # Número de árboles
    'max_depth': randint(3, 15),                   # Profundidad máxima del árbol
    'learning_rate': uniform(0.01, 0.3),           # Tasa de aprendizaje
    'subsample': uniform(0.5, 0.5),                # Proporción de datos por árbol (0.5–1.0)
    'colsample_bytree': uniform(0.5, 0.5),         # Fracción de columnas por árbol
    'gamma': uniform(0, 5),                        # Regularización para crear nuevas ramas
}

xgb_libre = XGBRegressor(random_state=42)

random_search_libre = RandomizedSearchCV(
    estimator=xgb_libre,
    param_distributions=param_distributions,
    n_iter=300,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search_libre.fit(X_train_libre, y_train_libre)

Fitting 5 folds for each of 300 candidates, totalling 1500 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002134E2B5710>,
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000002134E2B5990>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000021358326790>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000021351553690>},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=2)

In [305]:
y_pred = random_search_libre.predict(X_test_libre)
rmse = root_mean_squared_error(y_test_libre, y_pred)
r2 = r2_score(y_test_libre, y_pred)

print(f"Best parameters: {random_search_libre.best_params_}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R²: {r2:.3f}")

Best parameters: {'colsample_bytree': 0.8058265802441404, 'gamma': 0.03533152609858703, 'learning_rate': 0.01691872751242473, 'max_depth': 13, 'n_estimators': 258, 'subsample': 0.6999304858576277}
Test RMSE: 25.29
Test R²: 0.206


Queda claro que el mejor modelo hasta el momento ha sido el XGBRegressor que incluye todas las columnas originales planteadas para `metrics_both_df`, es decir con los valores de entrenamiento de:
- `X_train_both`
- `y_train_both`

Y siendo probado sobre:
- `X_test_both`
- `y_test_both`

Además, el modelo que fue probado con los datos de `metrics_libre_df`, tuvo el peor rendimiento. Solo por detrás de aquel en el que se aplicó transformación logarítmica. 

En la investigación se suele creer que un R² de 0.3 en problemas reales de nutrición y respuesta glucémica no es raro. Hay mucho ruido biológico no controlado: estrés, sueño, microbioma profundo, hidratación, medicamentos, etc. En este análisis ya se están utilizando sensores + datos clínicos y microbioma, se cree que se alcanza la frontera de lo que es predecible con datos tabulares y la limitación de eventos de comida actuales.

De cualquier forma, se exploran otros modelos de regresión.

## Modelo de Stacking

In [628]:
# Modelos base
base_models = [
    ('rf', RandomForestRegressor(random_state=42)),
    ('xgb', XGBRegressor(random_state=42, verbosity=0)),
    ('svr', make_pipeline(StandardScaler(), SVR()))
]

# Meta-modelo (modelo que aprende sobre las predicciones de los anteriores)
meta_model = Ridge()

# Modelo de stacking
stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

param_grid = {
    'rf__n_estimators': [400, 500, 600],              # más cerca del valor óptimo
    'xgb__learning_rate': [0.03, 0.05, 0.07],         # explora suavemente alrededor de 0.05
    'svr__svr__C': [0.05, 0.1, 0.2, 0.3],             # más fino en torno a 0.1
    'final_estimator__alpha': [0.05, 0.1, 0.2, 0.5]   # explorar ajustes similares de regularización
}

grid_search = GridSearchCV(
    estimator=stack_model,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    n_jobs=-4,
    verbose=2
)


In [629]:
# Entrenamiento
grid_search.fit(X_train_both, y_train_both)

Fitting 3 folds for each of 144 candidates, totalling 432 fits


GridSearchCV(cv=3,
             estimator=StackingRegressor(cv=5,
                                         estimators=[('rf',
                                                      RandomForestRegressor(random_state=42)),
                                                     ('xgb',
                                                      XGBRegressor(base_score=None,
                                                                   booster=None,
                                                                   callbacks=None,
                                                                   colsample_bylevel=None,
                                                                   colsample_bynode=None,
                                                                   colsample_bytree=None,
                                                                   device=None,
                                                                   early_stopping_rounds=None,
                                                                   enable_categorical=False,
                                                                   eval_metric=None,
                                                                   feature_types=None,
                                                                   gamma=None...
                                                                   num_parallel_tree=None,
                                                                   random_state=42, ...)),
                                                     ('svr',
                                                      Pipeline(steps=[('standardscaler',
                                                                       StandardScaler()),
                                                                      ('svr',
                                                                       SVR())]))],
                                         final_estimator=Ridge(), n_jobs=-1),
             n_jobs=-4,
             param_grid={'final_estimator__alpha': [0.05, 0.1, 0.2, 0.5],
                         'rf__n_estimators': [400, 500, 600],
                         'svr__svr__C': [0.05, 0.1, 0.2, 0.3],
                         'xgb__learning_rate': [0.03, 0.05, 0.07]},
             scoring='neg_mean_squared_error', verbose=2)

In [630]:
# Evaluación
y_pred = grid_search.predict(X_test_both)
rmse = root_mean_squared_error(y_test_both, y_pred)
r2 = r2_score(y_test_both, y_pred)

print("Mejores hiperparámetros:")
print(grid_search.best_params_)
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")


Mejores hiperparámetros:
{'final_estimator__alpha': 0.5, 'rf__n_estimators': 600, 'svr__svr__C': 0.1, 'xgb__learning_rate': 0.03}
RMSE: 23.31
R²: 0.285


In [634]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test_both, y=y_pred, alpha=0.6)
plt.plot([y_test_both.min(), y_test_both.max()], [y_test_both.min(), y_test_both.max()], 'r--')
plt.xlabel("Valor real")
plt.ylabel("Predicción")
plt.title("Real vs. Predicho (Stacking Regressor)")
plt.grid(True)
plt.show()

<Figure size 600x600 with 1 Axes>

# **19. Clustering** <a class="anchor" id="19"></a>


[Tabla de Contenidos](#0.1)

**Objetivo**: Agrupar comidas según su efecto glucémico, en función de macronutrientes (carbs, fat, protein, fiber), y combinando con su impacto medido (iAUC, delta, etc.).

In [635]:
metrics_both_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290 entries, 0 to 1289
Data columns (total 49 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   patient                                    1290 non-null   int64  
 1   calories                                   1290 non-null   float64
 2   carbs                                      1290 non-null   float64
 3   protein                                    1290 non-null   float64
 4   fat                                        1290 non-null   float64
 5   fiber                                      1290 non-null   float64
 6   iauc_libre                                 1290 non-null   float64
 7   delta_libre                                1290 non-null   float64
 8   time_to_peak_libre                         1290 non-null   float64
 9   variability_libre                          1290 non-null   float64
 10  iauc_dexcom             

Para la selección de variables, vamos a centrarnos en las variables directamente relacionadas con el alimento y su impacto:

Macronutrientes: `carbs`, `fat`, `protein`, `fiber`, `calories`
Indicadores glucémicos: `iauc_libre`, `delta_libre`, `variability_libre`

Se va a trabajar con Libre, pero no con ambos sensores en simultáneo al inicio para evitar duplicidad.

In [636]:
# Cargar las columnas seleccionadas
selected_cols = [
    'calories', 'carbs', 'protein', 'fat', 'fiber',
    'iauc_libre', 'delta_libre', 'variability_libre'
]

# Subconjunto de datos
df_subset = metrics_both_df[selected_cols].copy()

# Normalización
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_subset)

# Convertir a DataFrame
df_scaled = pd.DataFrame(scaled_data, columns=selected_cols)
df_scaled.head()

,calories,carbs,protein,fat,fiber,iauc_libre,delta_libre,variability_libre
0,2.275679,1.127079,0.831969,2.321018,1.493202,-0.531346,-0.590855,-0.676511
1,-1.526767,-1.149086,-1.443405,-1.315685,-0.924552,-0.485291,-0.430130,-0.693022
2,-1.422112,-0.945251,-1.443405,-1.181489,-0.924552,-0.474514,-0.355125,-0.630800
3,-0.243005,0.481600,-0.305718,-0.611157,-0.924552,-0.594994,-0.419415,-0.437472
4,1.124480,1.262970,-0.564283,1.502425,-0.320113,-0.755589,-0.649787,-0.673558


**Determinar número óptimo de clusters (k)**
Usamos el método del codo (Elbow Method) y Silhouette Score.

In [637]:
inertia = []
silhouette_scores = []
K = range(2, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(df_scaled, kmeans.labels_))

# Plot
plt.figure()
plt.plot(K, inertia, 'bx-')
plt.xlabel('Número de clusters')
plt.ylabel('Inertia')
plt.title('Método del codo')
plt.show()

plt.figure()
plt.plot(K, silhouette_scores, 'gx-')
plt.xlabel('Número de clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs k')
plt.show()


<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [638]:
# Ajustar KMeans con número óptimo de clusters
optimal_k = 3 
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
metrics_both_df['cluster'] = kmeans.fit_predict(df_scaled)

In [639]:
cluster_profiles = metrics_both_df.groupby('cluster')[selected_cols].mean()
cluster_profiles.to_csv('cluster_profiles.csv')
cluster_profiles

,calories,carbs,protein,fat,fiber,iauc_libre,delta_libre,variability_libre
cluster,,,,,,,,
0,310.843003,29.799590,19.669795,10.651237,4.058481,1483.475587,24.987144,10.699507
1,500.973819,59.539370,27.088976,16.629232,2.473622,5900.994289,103.670604,34.194361
2,796.398000,76.149467,39.109222,32.954333,6.473333,2420.963149,38.682074,13.820320


**Resumen comparativo**
| Cluster | Perfil                 | Carbs | Fiber |  Fat  | iAUC Libre | Delta Glucosa | Observación              |
| ------- | ---------------------- | ----- | ----- | ----- | ---------- | ------------- | ------------------------ |
| 0       | Ligero y bajo impacto  | Bajo  | Medio | Bajo  | Bajo       | Bajo          | Snack saludable          |
| 1       | Alto impacto glucémico | Alto  | Bajo  | Medio | Muy alto   | Muy alto      | Riesgoso glucémicamente  |
| 2       | Completo y moderado    | Alto  | Alto  | Alto  | Moderado   | Medio         | Comida rica y balanceada |


**Cluster 0** – "Comidas ligeras y de bajo impacto glucémico"
* `Calories`: ~311 kcal
* `Carbs`: ~30 g
* `Protein`: ~19.7 g
* `Fat`: ~10.7 g
* `Fiber`: ~4.1 g
* `iAUC Libre`: 1,483 → Bajo
* `Delta Libre`: ~25
* `Variabilidad Libre`: 10.7

Interpretación:
* Estas comidas son bajas en calorías y carbohidratos, con un buen nivel de proteína y fibra, y bajo impacto glucémico.
* Ideal para snacks o comidas balanceadas con bajo riesgo glucémico.

**Cluster 1** – "Comidas de impacto glucémico alto"
* `Calories`: ~501 kcal
* `Carbs`: ~59.5 g
* `Protein`: ~27.1 g
* `Fat`: ~16.6 g
* `Fiber`: ~2.5 g
* `iAUC Libre`: 5,901 → Muy alto
* `Delta Libre`: ~104
* `Variabilidad Libre`: 34.2

Interpretación:
* Estas comidas tienen un alto contenido de carbohidratos y un pico glucémico muy alto, con poca fibra.
* A pesar de tener una cantidad decente de proteínas y grasas, la baja fibra y el alto nivel de carbs hacen que tengan un gran impacto en glucosa.
* Este clúster representa comidas que podrían ser problemáticas para personas con control glucémico delicado (por ejemplo, con diabetes o prediabetes).

**Cluster 2** – "Comidas energéticas moderadas"
* `Calories`: ~796 kcal (las más calóricas)
* `Carbs`: ~76.1 g
* `Protein`: ~39.1 g
* `Fat`: ~33 g
* `Fiber`: ~6.5 g
* `iAUC Libre`: 2,421 → Medio
* `Delta Libre`: ~39
* `Variabilidad Libre`: 13.8

Interpretación:
* Estas comidas son muy completas y energéticas: altas en proteínas, grasas y fibra.
* A pesar del alto contenido de carbs, la respuesta glucémica está más controlada que en el clúster 1.
* Podrían representar comidas principales equilibradas, como una comida post-entreno o un plato denso pero bien balanceado.

In [360]:
# Reducimos a 2 dimensiones
pca = PCA(n_components=2)
pca_components = pca.fit_transform(df_scaled)

# Añadimos componentes PCA al DataFrame original
metrics_both_df['PCA1'] = pca_components[:, 0]
metrics_both_df['PCA2'] = pca_components[:, 1]

# Ajuste automático del jitter
jitter_strength_x = 0.02 * (metrics_both_df['PCA1'].max() - metrics_both_df['PCA1'].min())
jitter_strength_y = 0.02 * (metrics_both_df['PCA2'].max() - metrics_both_df['PCA2'].min())

plt.figure(figsize=(10, 6))

for cluster in sorted(metrics_both_df['cluster'].unique()):
    subset = metrics_both_df[metrics_both_df['cluster'] == cluster].copy()
    jittered_x = subset['PCA1'] + np.random.normal(0, jitter_strength_x, size=subset.shape[0])
    jittered_y = subset['PCA2'] + np.random.normal(0, jitter_strength_y, size=subset.shape[0])
    plt.scatter(jittered_x, jittered_y, label=f'Cluster {cluster}', alpha=0.6)

plt.title('Clusters con jitter ajustado a la escala de PCA')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.grid(True)
plt.show()



<Figure size 1000x600 with 1 Axes>

**Separación visible entre clústers**:

Los tres grupos se distinguen visualmente bien en el espacio 2D generado por PCA.
* No hay una superposición significativa entre clústers, lo que indica que el modelo captó estructuras diferenciadas.

Distribución coherente con el análisis anterior:
* El clúster 0 (azul) agrupa comidas más ligeras.
* El clúster 1 (naranja) contiene las comidas con mayor impacto glucémico.
* El clúster 2 (verde) parece tener comidas más calóricas y balanceadas.

**¿Qué representa este clustering?**

Este clustering agrupa eventos de comida según su perfil nutricional y su impacto en la glucosa postprandial, medido por:
* iAUC Libre (área bajo la curva post-comida)
* Delta Libre (variación máxima)
* Variabilidad (fluctuación glucémica)

En otras palabras:

Este modelo identifica tipos de comidas que generan respuestas glucémicas similares.

**Ventajas:**
* Da una visión pura del efecto del alimento y su impacto glucémico inmediato.
* Permite comparar diferentes comidas independientemente del contexto.

**Limitaciones:**
* Ignora que la misma comida puede tener un efecto distinto dependiendo de:
    * La hora del día
    * El nivel de actividad física antes/después
    * El microbioma del individuo
    * No capta variabilidad interindividual ni condiciones fisiológicas.

## Entrenamiento de un modelo tomando en cuenta el clustering

Se plantea el entrenamiento de un modelo a través del siguiente proceso:
1. Separación del dataset principal en train y test
2. Hacer clustering según lo hecho previamente para train
3. Predecir o asignar el cluster para test
4. Utilizar el cluster asignado como una feature de entrenamiento para verificar si los modelos pueden predecir mejor los carbohidratos

In [642]:
# Obtener los IDs únicos de pacientes
unique_patients = metrics_both_df['patient'].unique()

# Dividir los pacientes en conjunto de entrenamiento y prueba
train_patients, test_patients = train_test_split(unique_patients, test_size=0.15, random_state=42)

# Crear los subconjuntos de datos
train_df = metrics_both_df[metrics_both_df['patient'].isin(train_patients)].copy()
test_df = metrics_both_df[metrics_both_df['patient'].isin(test_patients)].copy()

## Clustering para train_df

In [643]:
# Cargar las columnas seleccionadas
selected_cols = [
    # Glucosa Libre (dispositivo Libre)
    'iauc_libre', 
    'delta_libre',
    'time_to_peak_libre',
    'variability_libre',
    
    # Actividad física inmediata
    'mean_hr',
    'mean_calories_activity',
    'mean_mets',

    # Hora de comida (codificada cíclicamente)
    'meal_time_hour_sin',
    'meal_time_hour_cos',

    # Fisiología básica
    'age',
    'bmi',
    'gender_encoded',

    # Microbioma (ya reducidos como PCA)
    'microbe_PC1',

    'metabolic_fitness',
    'gut_health_index'
]


# Subconjunto de datos
df_subset = train_df[selected_cols].copy()

# Normalización
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_subset)

# Convertir a DataFrame
df_scaled = pd.DataFrame(scaled_data, columns=selected_cols)
df_scaled.head()

,iauc_libre,delta_libre,time_to_peak_libre,variability_libre,mean_hr,mean_calories_activity,mean_mets,meal_time_hour_sin,meal_time_hour_cos,age,bmi,gender_encoded,microbe_PC1,metabolic_fitness,gut_health_index
0,-0.489660,-0.555407,0.990660,-0.639162,0.078810,0.640309,0.950031,-0.606629,-0.831062,-1.61844,-1.32247,1.272508,1.351842,1.246679,0.366003
1,-0.443939,-0.394224,-0.595993,-0.655596,-1.766833,-0.939993,-1.201850,-1.114933,1.577160,-1.61844,-1.32247,1.272508,1.351842,1.246679,0.366003
2,-0.433241,-0.319006,-0.509449,-0.593665,-0.075961,0.107004,0.223835,0.087728,2.458630,-1.61844,-1.32247,1.272508,1.351842,1.246679,0.366003
3,-0.552846,-0.383479,-0.595993,-0.401243,0.647593,1.203502,1.716925,1.290390,-0.185781,-1.61844,-1.32247,1.272508,1.351842,1.246679,0.366003
4,-0.712275,-0.614507,-0.942172,-0.636223,-0.507772,-0.593485,-0.730014,0.087728,-1.067251,-1.61844,-1.32247,1.272508,1.351842,1.246679,0.366003


In [644]:
inertia = []
silhouette_scores = []
K = range(2, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(df_scaled, kmeans.labels_))

# Plot
plt.figure()
plt.plot(K, inertia, 'bx-')
plt.xlabel('Número de clusters')
plt.ylabel('Inertia')
plt.title('Método del codo')
plt.show()

plt.figure()
plt.plot(K, silhouette_scores, 'gx-')
plt.xlabel('Número de clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs k')
plt.show()


<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [645]:
# Ajustar KMeans con número óptimo de clusters
optimal_k = 4 
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
train_df['cluster'] = kmeans.fit_predict(df_scaled)

In [646]:
cluster_profiles = train_df.groupby('cluster')[selected_cols].mean()
cluster_profiles

,iauc_libre,delta_libre,time_to_peak_libre,variability_libre,mean_hr,mean_calories_activity,mean_mets,meal_time_hour_sin,meal_time_hour_cos,age,bmi,gender_encoded,microbe_PC1,metabolic_fitness,gut_health_index
cluster,,,,,,,,,,,,,,,
0,1659.846056,27.956627,65.087349,10.613366,80.245736,1.761344,16.595189,-0.288709,-0.271618,39.289157,32.521829,0.421687,6.146236,1.840361,4.286145
1,1796.416094,28.256393,55.763699,11.834996,78.438992,1.619202,17.772276,-0.249570,-0.331672,56.770548,28.928102,0.181507,-2.762522,2.445205,6.773973
2,5824.588812,101.917718,79.527027,34.001271,85.378499,2.017449,19.525218,0.216193,-0.451871,52.851351,30.997682,0.418919,-0.926083,2.139640,5.481982
3,1887.500005,31.260858,52.132780,11.915768,93.117471,3.294444,26.335965,0.216028,-0.587594,43.286307,31.986031,0.535270,-0.498763,1.896266,5.211618


In [647]:
# Reducimos a 2 dimensiones con PCA
pca = PCA(n_components=2)
pca_components = pca.fit_transform(df_scaled)

# Añadimos componentes PCA al DataFrame original
train_df['PCA1'] = pca_components[:, 0]
train_df['PCA2'] = pca_components[:, 1]

# Ajuste automático del jitter en base a train_df
jitter_strength_x = 0.02 * (train_df['PCA1'].max() - train_df['PCA1'].min())
jitter_strength_y = 0.02 * (train_df['PCA2'].max() - train_df['PCA2'].min())

# Visualización
plt.figure(figsize=(10, 6))

for cluster in sorted(train_df['cluster'].unique()):
    subset = train_df[train_df['cluster'] == cluster].copy()
    jittered_x = subset['PCA1'] + np.random.normal(0, jitter_strength_x, size=subset.shape[0])
    jittered_y = subset['PCA2'] + np.random.normal(0, jitter_strength_y, size=subset.shape[0])
    plt.scatter(jittered_x, jittered_y, label=f'Cluster {cluster}', alpha=0.6)

plt.title('Clusters con jitter ajustado a la escala de PCA')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.grid(True)
plt.show()



<Figure size 1000x600 with 1 Axes>

In [648]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1087 entries, 0 to 1289
Data columns (total 52 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   patient                                    1087 non-null   int64  
 1   calories                                   1087 non-null   float64
 2   carbs                                      1087 non-null   float64
 3   protein                                    1087 non-null   float64
 4   fat                                        1087 non-null   float64
 5   fiber                                      1087 non-null   float64
 6   iauc_libre                                 1087 non-null   float64
 7   delta_libre                                1087 non-null   float64
 8   time_to_peak_libre                         1087 non-null   float64
 9   variability_libre                          1087 non-null   float64
 10  iauc_dexcom                  

## Predicción de cluster para test

In [649]:
df_test_subset = test_df[selected_cols].copy()

scaled_test = scaler.transform(df_test_subset)

test_df['cluster'] = kmeans.predict(scaled_test)


In [650]:
# Reducir a 2 dimensiones con PCA
pca = PCA(n_components=2)
pca_components = pca.fit_transform(scaled_test)

# Añadir componentes PCA al DataFrame original
test_df['PCA1'] = pca_components[:, 0]
test_df['PCA2'] = pca_components[:, 1]

# Ajuste automático del jitter en base a train_df
jitter_strength_x = 0.02 * (test_df['PCA1'].max() - test_df['PCA1'].min())
jitter_strength_y = 0.02 * (test_df['PCA2'].max() - test_df['PCA2'].min())

# Visualización
plt.figure(figsize=(10, 6))

for cluster in sorted(train_df['cluster'].unique()):
    subset = train_df[train_df['cluster'] == cluster].copy()
    jittered_x = subset['PCA1'] + np.random.normal(0, jitter_strength_x, size=subset.shape[0])
    jittered_y = subset['PCA2'] + np.random.normal(0, jitter_strength_y, size=subset.shape[0])
    plt.scatter(jittered_x, jittered_y, label=f'Cluster {cluster}', alpha=0.6)

plt.title('Clusters con jitter ajustado a la escala de PCA')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.grid(True)
plt.show()

<Figure size 1000x600 with 1 Axes>

In [651]:
train_df.columns

Index(['patient', 'calories', 'carbs', 'protein', 'fat', 'fiber', 'iauc_libre',
       'delta_libre', 'time_to_peak_libre', 'variability_libre', 'iauc_dexcom',
       'delta_dexcom', 'time_to_peak_dexcom', 'variability_dexcom',
       'mean_calories_activity', 'mean_hr', 'mean_mets', 'age', 'bmi',
       'height', 'a1c_pdl_(lab)', 'insulin', 'triglycerides', 'hdl', 'non_hdl',
       'ldl_(cal)', 'lps_biosynthesis_pathways',
       'biofilm_chemotaxis_and_virulence_pathways', 'tma_production_pathways',
       'ammonia_production_pathways', 'metabolic_fitness',
       'active_microbial_diversity', 'butyrate_production_pathways',
       'putrescine_production_pathways', 'bile_acid_metabolism_pathways',
       'inflammatory_activity', 'microbiome_induced_stress', 'microbe_PC1',
       'microbe_PC2', 'microbe_PC3', 'meal_time_hour_sin',
       'meal_time_hour_cos', 'gut_health_index', 'gas_production_index',
       'delta_ratio', 'variability_ratio', 'mean_variability',
       'post_meal_st

In [652]:
test_df.columns

Index(['patient', 'calories', 'carbs', 'protein', 'fat', 'fiber', 'iauc_libre',
       'delta_libre', 'time_to_peak_libre', 'variability_libre', 'iauc_dexcom',
       'delta_dexcom', 'time_to_peak_dexcom', 'variability_dexcom',
       'mean_calories_activity', 'mean_hr', 'mean_mets', 'age', 'bmi',
       'height', 'a1c_pdl_(lab)', 'insulin', 'triglycerides', 'hdl', 'non_hdl',
       'ldl_(cal)', 'lps_biosynthesis_pathways',
       'biofilm_chemotaxis_and_virulence_pathways', 'tma_production_pathways',
       'ammonia_production_pathways', 'metabolic_fitness',
       'active_microbial_diversity', 'butyrate_production_pathways',
       'putrescine_production_pathways', 'bile_acid_metabolism_pathways',
       'inflammatory_activity', 'microbiome_induced_stress', 'microbe_PC1',
       'microbe_PC2', 'microbe_PC3', 'meal_time_hour_sin',
       'meal_time_hour_cos', 'gut_health_index', 'gas_production_index',
       'delta_ratio', 'variability_ratio', 'mean_variability',
       'post_meal_st

In [653]:
columns_nutrients = ['carbs', 'protein', 'fat', 'fiber', 'calories']
columns_to_drop = ['patient', 'PCA1', 'PCA2']
target = 'carbs'

X_train_both = train_df.drop(columns = columns_nutrients + columns_to_drop)
y_train_both = train_df[target]

X_test_both = test_df.drop(columns = columns_nutrients + columns_to_drop)
y_test_both = test_df[target]

In [654]:
X_train_both

,iauc_libre,delta_libre,time_to_peak_libre,variability_libre,iauc_dexcom,delta_dexcom,time_to_peak_dexcom,variability_dexcom,mean_calories_activity,mean_hr,mean_mets,age,bmi,height,a1c_pdl_(lab),insulin,triglycerides,hdl,non_hdl,ldl_(cal),lps_biosynthesis_pathways,biofilm_chemotaxis_and_virulence_pathways,tma_production_pathways,ammonia_production_pathways,metabolic_fitness,active_microbial_diversity,butyrate_production_pathways,putrescine_production_pathways,bile_acid_metabolism_pathways,inflammatory_activity,microbiome_induced_stress,microbe_PC1,microbe_PC2,microbe_PC3,meal_time_hour_sin,meal_time_hour_cos,gut_health_index,gas_production_index,delta_ratio,variability_ratio,mean_variability,post_meal_state,gender_encoded,cluster
0,1526.600000,23.200000,97.0,8.387954,783.600000,24.6,101.0,6.988962,2.709378,84.504132,25.842975,27,22.265239,65.0,5.4,2.5,67.0,74,142,130,2,1,2,1,3,2,3,2,3,2,2,12.073691,-3.625122,10.078317,-5.000000e-01,-0.866025,6,6,0.943051,1.200000,7.688458,2183.838194,1,3
1,1626.600000,29.200000,42.0,8.192057,1422.900000,39.2,36.0,10.717140,1.243350,64.793388,11.859504,27,22.265239,65.0,5.4,2.5,67.0,74,142,130,2,1,2,1,3,2,3,2,3,2,2,12.073691,-3.625122,10.078317,-8.660254e-01,0.500000,6,6,0.744879,0.764317,9.454598,768.417458,1,0
2,1650.000000,32.000000,45.0,8.930279,1861.600000,35.6,39.0,9.151036,2.214637,82.851240,21.123967,27,22.265239,65.0,5.4,2.5,67.0,74,142,130,2,1,2,1,3,2,3,2,3,2,2,12.073691,-3.625122,10.078317,0.000000e+00,1.000000,6,6,0.898851,0.975770,9.040657,1750.146848,1,0
3,1388.400000,29.600000,42.0,11.223976,1273.400000,40.2,36.0,15.503717,3.231845,90.578512,30.826446,27,22.265239,65.0,5.4,2.5,67.0,74,142,130,2,1,2,1,3,2,3,2,3,2,2,12.073691,-3.625122,10.078317,8.660254e-01,-0.500000,6,6,0.736300,0.723907,13.363846,2792.213647,1,3
4,1039.700000,21.000000,30.0,8.422984,1902.700000,55.0,44.0,27.186248,1.564802,78.239669,14.925620,27,22.265239,65.0,5.4,2.5,67.0,74,142,130,2,1,2,1,3,2,3,2,3,2,2,12.073691,-3.625122,10.078317,1.224647e-16,-1.000000,6,6,0.381811,0.309814,17.804616,1167.775562,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1285,4023.771240,162.400000,77.0,50.646609,6549.116699,166.2,77.0,60.219531,3.217988,98.768595,34.553719,58,36.090876,60.0,7.2,25.2,124.0,43,103,81,1,1,3,3,3,3,3,3,3,2,2,-7.121564,1.133549,0.655322,9.659258e-01,-0.258819,5,6,0.977130,0.841019,55.433070,3412.822280,0,2
1286,5523.300000,66.066667,91.0,23.106481,6276.000000,92.2,106.0,33.690988,1.665565,83.603306,17.884298,58,36.090876,60.0,7.2,25.2,124.0,43,103,81,1,1,3,3,3,3,3,3,3,2,2,-7.121564,1.133549,0.655322,-2.588190e-01,-0.965926,5,6,0.716550,0.685815,28.398734,1495.186394,0,2
1287,8406.618774,103.533333,62.0,32.078574,2048.849609,130.0,72.0,46.664682,1.276112,72.528926,13.702479,58,36.090876,60.0,7.2,25.2,124.0,43,103,81,1,1,3,3,3,3,3,3,3,2,2,-7.121564,1.133549,0.655322,-8.660254e-01,0.500000,5,6,0.796404,0.687413,39.371628,993.826105,0,2
1288,5935.500000,90.666667,47.0,26.563904,6138.300000,102.4,47.0,30.363261,3.599744,93.925620,38.652893,58,36.090876,60.0,7.2,25.2,124.0,43,103,81,1,1,3,3,3,3,3,3,3,2,2,-7.121564,1.133549,0.655322,9.659258e-01,-0.258819,5,6,0.885408,0.874841,28.463582,3630.496892,0,2


## Entrenamiento de un modelo

In [655]:
param_distributions = {
    'n_estimators': randint(200, 1000),             # Número de árboles
    'max_depth': randint(3, 15),                   # Profundidad máxima del árbol
    'learning_rate': uniform(0.01, 0.3),           # Tasa de aprendizaje
    'subsample': uniform(0.5, 0.5),                # Proporción de datos por árbol (0.5–1.0)
    'colsample_bytree': uniform(0.5, 0.5),         # Fracción de columnas por árbol
    'gamma': uniform(0, 5),                        # Regularización para crear nuevas ramas
}

xgb = XGBRegressor(random_state=42)

random_search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_distributions,
    n_iter=300,
    cv=5,
    scoring='neg_mean_squared_error',
    verbose=2,
    n_jobs=-4, 
    random_state=42
)

random_search_xgb.fit(X_train_both, y_train_both)

Fitting 5 folds for each of 300 candidates, totalling 1500 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000023DE60A9A10>,
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000023DEBC08250>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000023DEBC0AC50>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000023DE06E99D0>},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=2)

In [656]:
y_pred = random_search_xgb.predict(X_test_both)
rmse = root_mean_squared_error(y_test_both, y_pred)
r2 = r2_score(y_test_both, y_pred)

print(f"Best parameters: {random_search_xgb.best_params_}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R²: {r2:.3f}")

plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test_both, y=y_pred, alpha=0.6)
plt.plot([y_test_both.min(), y_test_both.max()], [y_test_both.min(), y_test_both.max()], 'r--')
plt.xlabel("Valor real")
plt.ylabel("Predicción")
plt.title("Real vs. Predicho (Random Forest)")
plt.grid(True)
plt.show()

Best parameters: {'colsample_bytree': 0.7632495970768904, 'gamma': 0.5615217220560959, 'learning_rate': 0.010775204453494716, 'max_depth': 8, 'n_estimators': 317, 'subsample': 0.7575875851668659}
Test RMSE: 23.22
Test R²: 0.291


<Figure size 600x600 with 1 Axes>

In [657]:
# Crear explainer para Random Forest
explainer = shap.TreeExplainer(random_search_xgb.best_estimator_)
shap_values = explainer.shap_values(X_test_both)

# Gráfico summary
shap.summary_plot(shap_values, X_test_both, show=True)

<Figure size 800x950 with 2 Axes>

In [658]:
# Top 15 por feature_importance_ del modelo
importances = random_search_xgb.best_estimator_.feature_importances_
feature_names = X_test_both.columns
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
top_xgb_features = importance_df.sort_values(by='importance', ascending=False).head(15)['feature'].tolist()


explainer = shap.TreeExplainer(random_search_xgb.best_estimator_)
shap_values = explainer.shap_values(X_test_both)

# Media absoluta del valor SHAP por feature
shap_importance = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'feature': X_test_both.columns, 'mean_abs_shap': shap_importance})
top_shap_features = shap_df.sort_values(by='mean_abs_shap', ascending=False).head(15)['feature'].tolist()

# Unir ambas listas, sin duplicados
top_combined_features = list(set(top_xgb_features + top_shap_features))

print(f"Total variables seleccionadas: {len(top_combined_features)}")
print("Variables seleccionadas:")
print(top_combined_features)

Total variables seleccionadas: 25
Variables seleccionadas:
['iauc_libre', 'putrescine_production_pathways', 'delta_libre', 'a1c_pdl_(lab)', 'mean_variability', 'meal_time_hour_cos', 'mean_hr', 'mean_mets', 'age', 'microbe_PC3', 'gut_health_index', 'gas_production_index', 'microbe_PC2', 'delta_dexcom', 'tma_production_pathways', 'delta_ratio', 'meal_time_hour_sin', 'time_to_peak_dexcom', 'iauc_dexcom', 'cluster', 'time_to_peak_libre', 'microbe_PC1', 'mean_calories_activity', 'hdl', 'microbiome_induced_stress']


In [659]:
X_train_reduced = X_train_both[top_combined_features].copy()
X_test_reduced = X_test_both[top_combined_features].copy()

# Entrenar con los mismos mejores hiperparámetros encontrados
best_params = random_search_xgb.best_params_

xgb_reduced = XGBRegressor(**best_params, random_state=42)
xgb_reduced.fit(X_train_reduced, y_train_both)

# Predicciones
y_pred = xgb_reduced.predict(X_test_reduced)

# Métricas
rmse = root_mean_squared_error(y_test_both, y_pred)
r2 = r2_score(y_test_both, y_pred)

print(f"[REDUCIDO] Test RMSE: {rmse:.2f}")
print(f"[REDUCIDO] Test R²: {r2:.3f}")

[REDUCIDO] Test RMSE: 23.48
[REDUCIDO] Test R²: 0.274


In [660]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test_both, y=y_pred, alpha=0.6)
plt.plot([y_test_both.min(), y_test_both.max()], [y_test_both.min(), y_test_both.max()], 'r--')
plt.xlabel("Valor real")
plt.ylabel("Predicción")
plt.title("Real vs. Predicho (Random Forest)")
plt.grid(True)
plt.show()

<Figure size 600x600 with 1 Axes>

# **20. Conclusión** <a class="anchor" id="20"></a>


[Tabla de Contenidos](#0.1)

Los modelos de predicción de carbohidratos obtenidos, resultaron mejores para aquellos que contienen la información de los dos sensores de glucosa continua. Además, el agregar una variable que defina su comportamiento a partir de clustering puede mejorar un poco la predictibilidad al tomar en cuenta relaciones que el modelo puede dejar pasar por alto.

Por otra parte, la obtención de un modelo de predicción de carbohidratos que describe el 29.4% de la variabilidad de los datos, con una raíz del error cuadrático medio de 23.16 es comparable a la de otros modelos que se han estudiado, con el añadido de variables incorporadas que se habían pasado por alto como el microbioma intestinal, y datos clínicos.


**Limitaciones**

El conjunto de datos presentó algunas limitaciones clave, como son los registros reportados en un patrón no estable, y de hecho, mal registrados como tal. Además de la falta de mediciones.

Por otra parte, una limitación de este tipo de conjuntos de datos es la variabilidad con la que se encuentra, pues siendo solametne 45 pacientes de estudio y menos de 2000 eventos de comida, es complicado que los modelos comprendan los patrones exactos para hacer sus predicciones.